# CrackSAFE Phase03 — unetpp_effb3 × 3 seeds
Development-only, original V4 exact training worker preserved. Seeds 42, 123, 2026. Attach raw dataset, source-safe v2 private manifest, and any prior V4 model-specific checkpoint ZIPs to resume.


In [ ]:
# CrackSAFE-CP v3.3 — Phase03 one-engine / three-seed launcher
# Exact, immutable V4 runner: only scheduling is changed in this controller.
# Training worker bytes, recipe, manifest, checkpoint identity and full checks unchanged.
import os, sys, time, json, hashlib, zipfile, subprocess, traceback, importlib.util
from pathlib import Path
from datetime import datetime, timezone

ENGINE = 'unetpp_effb3'
EXACT_RUNNER_SHA = '805ff45af895b1a728407f53921b969c0be66be3d521af85274849b73533d8d2'
EXACT_RUNNER_FILENAME = 'cracksafe_p03_sourcesafe_large_dev_v4_manifestfix_v2.py'
EXACT_RUNNER_B85 = (
    'c-m~Z>DH=R)+YG>T*Ygs9MnU6ARrhjHYkb;c8E=mlpX}>q_cO_Mbs76l@xoQh_AA8<R6f^)|%a#%`^SSKmI{SigwxbnGr-Ue=_u+=+FQ2fBoM-=#SxB'
    'wr~90^PjO{=s&I!Sw{2|2Wg}kP^_57k8uy(AX1Vbdi;b@kOW%b{zR$o=le43pMU(p2R@`3QDU2R&OtXa!wo{u@RPrO_=WHjn5M1Siu<!XE3pBiKg@PM'
    '|K|@M{Ah}6t2t*o@c%0@;v|=@8gBg0Kj^?uqQG^3X`-DCJ#~%vC%FYbwyztZkw5(8XT040+=A$0MDdSp{?OZDUlQ33JD888;YeiZ(dH*KBF|28j{RRh'
    'V>?cMo&0sm_N^b?jtuR$UA3Puh!RD0jbHm9Cs2QsNITnEF86O9KQqO$^5rPkQ;B}D{Kr53@rM}&-j5`iX_7{UB>mXlFFrqt?+3|myW&6oLyD|WiDKig'
    'A2s(v1N-Yoi?hFebR{to+cSQY_(#tjmb>VjJLRt*JNWD0-15(VdPbt;Ta{niJpHd9Cl37oaQ2ks{MSzq|Me3oz8-k_JC_ol)5Lc3cdF)>rWtX*9zXKO'
    'FKPKRj|W4|W8$x$ryZKMo70}Tq`xiFe)(Set^a>aqJz{=jOg$6Fc+C`{?FPn|NP^R!F)Sfu327?Hj{qm&pLsUb&N1Iq;A5vsw28y*K}AXaf~2TRd3?B'
    'j;J`+#G9x=p!Eh>$5B#kni@*v{xSaiV}t+ANmErtCl%a;F#|E`s%|1CrZsQ`hB1vGFa>QIXcLBsTpFz4FpQ#F!%*vW12YX$NB{g|$<7zKT8r(P9{tS|'
    'sS^!^Y$^tcYq(xl>j<eDIXf-qh3gI6AP56P^d_kxx`C>&hQpfqO9x%k5Q3=x`Nw=g&qhOu>+>u#+WgyS1T#${w}CVoxJn|JqN{bRsUb}m*LAbr%&lv{'
    'CP|V;E{N1%GuIYr7=)S=6hzbi{A0QpNF#pC@}o@${4@29Bn&0PG*$G^zy6F3%M2pVh$I!xKmWa|1%Bom_e4r;|M6$#kIiT_%nxJrzmPimR~@c5Du491'
    'V{XI?eSux@(q=iLJFh=}{<QT!|MlnDbnQ^een0c{Zo-lp+15Gn^O*U?I}epOkup@$jpSD<_<0uD+Mj>r=l?#M$Cs+8wreMkKmP^)6aVYa?Eg&9Py8+5'
    'B5CKq&V%yLfBg^mFXXTKU+Dk*wfesZ_EI;oL)X^wG|G%$`~KPd3;8GVd*6R+rKmxg>z3CMU0RO^Iomw-3|I0D#sA;V<Og!qeM<^_H`hmqQsb{bZBMZb'
    'DO7a**GiZtw-IHA{-0$}i7!%~RR5M>89|<i(IfXt8oK|zEoc6#c1pjlCUuqErSVy*;|=f^^<PF9Xy<>0iXBO|UimTYC@*vk>_^S((ho%axItb9|DC%3'
    'T>pOk_N||D%cVYlmk~)da_)xY28wP(xdir?;=GY(e{Si2#fkm{=Cv%(*Ca}{Brn7H_Mh;aSL<II+uE;$_v2q0#+97?=MS10{~ZE;n0qh^f+XkqUzguS'
    '?!SH$e`}Gu_y4f|?*jT?|J_xjC5-+mx}*7G`76#O^o~j*Ce8cn@#r@rPq|BRezCud$zv!@eqBC>!2enM?WUZRC-pDVP04*3{x|h^?BrF)Hhe7)2ITj*'
    'XKPU)sY;@q|NK||7bVY=Uu7#Z{zF9?zsP^za%?~IqZmg1BDsbV{f>sb{rqBRK-h1w#;*X(BVAK|$1IHe7X0ttAmyzv|0(4iE)N3#zqR`<TX9|KZ*S$n'
    'n;VpuRY`sPHWR_B2wcaiRrG(~80X#Rmu~+Ky#E}i|F8RhG0R&=ek>1cD@^|*F!!S5DygrXB{kJ8Bl-XQTL0}IGtDLCtxO7{-$C;~z`wiwU%%qA@;gF)'
    '_k7#``CT%8Q~nJIa@zk=iT>xWh$;Sz?uwTD{L&H1qxNFtjVMo<%ns7nef;E3`5OWJ4IX~H;P1YzL^+1=R72PEwY-06f%hxX;$LXP&MS)YtK<LwD)fNF'
    'e>!=VL)Srh-{k{2vpi>la_bd(`EF7Agl6KN@q+0ZT0rDn67@%o?C+^;#Xc}>FhRIRgjcWQ_4|a}gvo?|cHgw8A4gTcXw__!S&KXG_2%u<b-E;usdH-B'
    'zH{FSfwrTvgD9DW>T-_gR8S`lgDCYBuLBDg!hry@>}{%r<<Nk5q14>A2I*kSn+>NPPpHB3y*!!ze9z^ro*oGP?fO{xDqAZQ$H45X2eaL<ovj$NETprZ'
    '2V2gnz8FQP!>Y>QBY84fBW=Hft5%`-=<AwmkY$`+QnZ$Kcbqot&q`0T)v3(^MPVOhhKMjR2F@^fo%2PCxT4LtiSY|~m4#KP@a@v<<ESk%4t2P8U>`>S'
    'Xz3$QUh2Ru>5z+gSbom+bi+X&UC1`WdXX6uokNWx<_m3#bqcgMp2jt9x}%@r1?HA(u~qBLN*FyaHQ6Fl$9wTu<=kL0>lqBSY2l-tbg@<)1AYnc*9*s{'
    'm)-A!TZPd1tS+BBFXhsE6zR7AY4f`1l}n8-c^uoK-noR}{%DnMgFd&o#cd7BBu1pW{g@;>$#}7!)zmJ0KV5)PSCHR4{~iwR57pblaa$r9i6*TEEHxgJ'
    'YfZ$qRU;GpQN$ZnV?H_VZZOat7|i3TZ;J9nxi+9}J*8c{h79_G()m_WOQu`ZO%@WTjsERM*wr-gL92FsIKVUWmsg1Jg<7Xk{+?eQF_uk4gI)*a6cJh|'
    ';lf6d?rcC_Jx*48i}cHt#vRHC%;WoFt{V%(fiAX0pn6dc$#LN%NDacIN$|bxzLoyqbsU7}lj`)B+k>G#YxWK_zyZTrI6m3ybouV_(CZ|mw>T@$!GdOu'
    'oL9XG?L;Y?m*iLW;s+@wh*ignjMBZi2{){HX=ER$%t|evBW_5?<me%4o6+%GYi6Z|B@*Es?{xZ(+8Z2#a^?Q|D#p5@CcA}K<uVWCJ+G`N+X`0ZRkleP'
    'Jwl(oA-4478j#}GU&nGeqI;x$xE;TK#Z4N;#@M@;b}`U+G>f(BH(pi_4&%Sp(t*1SY;!2qH`W*Gf*6%uN?JSJdG6=MN5gUMtCu6i-O61&Fhcg)fX;vu'
    'vU4kAr>&b>im146?ZBpVFP#GDgH+2zANxWmL0zY`=vB%U{LUa3!q@l;e45Va!G01O-u8QtIJ``D&EQL?wbKT}dct+ALB+}$7&ET;+677JdMm#zYu340'
    'DWwk25FHkP^e~?U#=fXSC)Zswk6w4)8yKto)QGK0>%^fId^0=qM+XWT-SGp&(mMlcTxkRgURl!I<(V!AXJ$+8<qOPz##Pbf;Gol*$xQTSYGLDZ?d^8h'
    '8JUFM%$B}mZNgl%g7$dpAv`}2z|1T?+6J3Zo9VdIOWT8|*685d+i|XXyT)@U@m#BxJZ@@libz`(KAIg5><A_BNVmG0b6L~&$?q>E(AQz8BJ|tQ&6sJy'
    'OT2gO#5z25R(MyI1u&7y>`AVS*6XVseoi+BZVAl3_8P8kWwe#-!ZUvL<~-60xC-38g}W$fKa<O2rK8XkN|l$-V@GUam3oHy*)7S=dAiY<O;5icnr3B('
    '?)SvDc!MO5Mf%4_ee`UeP71K6>TQ!^hFNd^J^&+YTpAr#-FwjGTX#l_HsW!>4!ad?v%XUY7_6L6ZMCd{^@JKT4suZ|y<+Vd);wI(%3(`hZm!SP@mzV8'
    'hax229svF+^ft~VtRuoA#?)H;1u1n(s)PNyEN<(Ng%sz_Mz8Q5zxzRLTkJ2W?!gZqc-=Qw39haDl&`PAVR>FIjj|z90e8>Zt+KZk1@o20Mt>F;>KD$k'
    '6uUIO?2hZWI4Z9}K14u?;@PUVer@o}3D|+5@(#zl{^A2@?sKNj2hP5AtE7Ck+T}UX*mkJxwb*Eno#yU4lvdcOtW2W*ksVDoD^D*feX?saW3Xdn*Oj$c'
    '$5V$ECp`NatjGn_TJ2@|keT<>k(uyKdIFu3H(9Jp0lP`ha%^oO<#s)DQuEjaaP;M{?6@mjiq&pFYR$8N$xoLGm`qv6<5P2H@5DZdb{TJ5RRMTR>Sd%$'
    'eQKb&3KM|yO8YUT9LpE|ggX%39+ua`?0_KwhR?(~Ll;ipQ>$F<4tL*CG+#=X4)pddLpx|*H^zAjyc1LH=)EOtpJb!L?PB)o%Fws?>3*JI-P^<OuR65d'
    ';5jfI_~YFWWwid$Cbg|25*UY3%rtCwD$ymZ2~D_R)M9VpgT#B6)$SG_e5h`LOd@y0%k?EgID%H$f=_l!*uw>-SQoe}x4`)u^o8lkzNDRHc{z-ZJ$336'
    'SaG!mIqzAZW&UMSSBqjcpgfS}*smUHK4Z$qVxfMt4&_O-Ez6sxzFlij*Q&>_XcOLCz*Yzux*mb5-5+rKbi`1r&2V_HKnM$JP(iTD>%B<O9#J^0?5$Kk'
    'jO%0L0UfB5KOZCyU4Q9{FA{hjyjknnfFWi3_*8&3wR6~KR|dzc@owKdN{>L;BTg6D+$`uquMO9WH`z2gG$Z{84Xk+L?6>5r<_1LbT_`wh$9zgATBnh8'
    '&#WKHj3Lf10ojeR-A-0Z3j3VNY_Q!#ZYY846FC8)nVS&-CJqa=_xx;D7plwW*>9JCq<U*p*jay;!{fN>i2MaDEhRcWO{us2_^yoOc8Jzs{yxM_rZuf}'
    'eVE-~oAC{v_RCCp;Mosj<JDF=G(*Tb*mwH-gO7>t{`FA|j@TA2-ace_YYB0<tUsRWV8zZoc1R_mTD%Cl1&BgB=q(nJY(dt$c0+a?CU`@T1}i+1Vr3EP'
    '5H8`+t+gPxo$qO%5{nmjFYf0$1X0`3t(6Jbnr!<jJzodw3fTyD*Sm0kIafU!zkDy76=bCkc=JbAA7pSfoB}=jRxK2{^hKX@C_xMSf>@qC>Ns6QUjvx-'
    'o1){d<trM1sg<7l(E~K;>g4=XmX{SR0o8%i*hgwZ4nMPda;fdB1TT~?-?J;c!?{aQEy;vEI~YAQ4vh8c&h00vXAXx2vtNrx!)DNXOl$W?--D(N<<z4a'
    'Vf_?>v1gY`h+4QPyQs506sNaK;w?53KzvsWs!gU<q%-qNabKQ2h0)~kEY-q!NOKaWwn;Nma=hD9Rz@osklyUF9gn+PqDvNTd5@1a;QSe|ZG?m!w!zaZ'
    '946X*`-10{s;rbq?pCYop?GEs&N^*8i8EHdjcsCIXz~qApQOdYpw~QX>BgRHcUB02d{?G4O4h4p%kv?tP!FcNYuyF9t#{3+&0b8n^B5-7IDc05U892y'
    'RdrUnDz>(xUt-}{F*n!o(t3GxQXNvUm!S*u1#rgs^DZwOVH@-+)m!JaIh5aEqkovT3}@>P^yqT5<huG2af^PDM7IPjv=Ol8SaM(aq{qJYwkBBzZL-M3'
    'M>a@|mu`-aBjTiV@`LMVm$)ClyVpj{vyV1wt0dW24>a@G?$s9!r1*fpZ{@Q+p2~DK+ZV1d)8!nsYI=l|JuI{}IyC%UFS572;yB^t%D{7py@%)>!4L&+'
    'F;l!=3Ldv@SvxnHm1Aw!mcGr3`l$~2YdXp{!Uwvu>1CF6YCR=u&y9ii+&8t2uWXAU#ft@|VfX;5^V4s)RdoQmPu$SeJkPjm4j9E&sgn#E72z-$G7E(O'
    '%)@qG8Q=XGY_9nWGlFMJTaFBGVYfb;iv7BNW3h0-;00IEtq~QPqyDA9q|aGrLp}DTZe?AS<#sfgdFNJB+*2D#nG?{yCT5!!)oqpu-HP{zZy2?z_Dg%9'
    'RFBZKZ6TfRzDMuY--OU<&fjMK&6`7R>8w41palfZ!A_7;!I?Epxnfm`8V)4y>15j3p4xZo?9UzncqwuC+dsJV`*~9t6kieX7%_FF-SlU-5rf%O_hK;E'
    'MFQl^wj1XXH$`V^2z-ID@ij9J>Ifdh@npNI^hS+&;krM?d!_okh%kD7kmc9{%u@RH7hBZ*jG=q5X{p$#3Jh0+_F}bpgKG|$sMP%G#6hXYre(BD7%Oyi'
    'e_xsYWvZY}Bp3inqFkXca{Hw^>!xJ=cD2}!tK!}>G$m<K=;}0R7y&X1?3bRECK|jzQ2YDF=opQj=S-W!T+>MRheBuEH$s$oN?mqVvtJoD$JilM$wnYt'
    'XAq*?U#XGFiu-eRe(6zFt);}Vv0ARsO1N$-iN%>>o1-tlu!Z7echekpcVSp-oy`pj4Iy%CG`4K#teq!UFZgUq@2!+_+vG|1Ne((Pu~aa9rBdt<sOE>Q'
    'M~$7RQ`no61+S3}8l_!Jx(CwvwHz-;Tef>*J9pk-Cf;NhR#W_ghVJSJm?q|e<`8=*N3e8h4j0916fq~rk%!0egIesLb-yRFavIins!hMcP>TvaBg-pw'
    '8qIE3)y+r7#bym-8Cl*r-Zqo;f-3eKeK31%c|JL_00BS<=Jwa5$oC!+L{@@#3l0{Y{oD(mixw%6T&wt1g&ypWJ8xp%8Mu#kL7Zy%u{4%}`e+6|me0|!'
    'rkyaSGi{ugk7e{?%k8auX*IXc0r|0DK<Ew2H!7;LALb-y+=k)}C63@ZeKEl0`FMJk_*V?CwiE0bA0+!VKQ+AwKuQbw4(e7fOw*P#TwVgSedZGX`uWCd'
    '0clSjt2uEA^{?=;VYI7e=j;^3;%wS~KJ3h1o@Q6GE$`pp$=(FijINnI6s*;H=lV6elOyh4p_9^TP%g~(56#KS>tx5)ozbZ<{|eJx(L3UD^p+RZ;5qQx'
    ';VEm>=DSdsB&JYbKkqM_-zz(h9u}~!ce##hSWsz`waU62ZqB@px0k5a1#ip^I2ST?B;UKsac>z~iyXhqUlF-0w}Dzyyg*%jw0m~R%UE&dWukI9X&AO;'
    'p;|xGuXjy#ff>^|IH-WUhcms>s!`fd5v}l<Jfi75dGM^eA!q&vIOt0SQ60Pf8cz#qwbkES??q*(fMsnpo=|dQCzTX%@#=@i99y%Lto3FGBnGWkZT4`c'
    'ryL*0^IH+B5X}K|nKOJ@dMQm)yA1`s@-9b{?t7EAUc8nBuiHVsBQum?-&sZLKEAuny~NxjJ*%F?&njMtyGqm3kp3c+3!GnM$z!L>oXXaftfxV%%`I0t'
    '*!QBRS{sMshuLIkuETb{`xNz7n4or$D=)2kc46W0EvKMLTB-5+s)=9SjaH+IF#~~aKHOK;-u?_M>|ODtxKsbTMSFOLqXSisJ{xh~(u$N>i<p*g)VJ58'
    '2v%8ulEGKx-b=E3svx?Z08zj9b|hsPDiy4_K@~wiBx7njJ<}uMQFv~%SaUvx_2`^76#<{Zy0i$Uy-KSdH{$BBcs<4QqkT%^-6tP++NGvJz`EQWpUv5N'
    '4)NT%idTwg(yt#6L#wIE^G9r6uB4(0mch{U3lVuy#;NT>WpJBgUDw>-G~u&Qp=Bw@+J`!-i@_w=EnZ@Jf@bfY3|Y&W7<i^u+}9?f7ThStmbh<wXdB$t'
    'd>#@9bE`{_$LSM_XSJpFS|m7kK9n+u9H@oHo^#omu;JrDbP<6Vqw0z=HIc`o^EfwaR#t-6$K@?-l5BG38M|5+7VJ)Zq10T^dZbh1$7}uFH4*$-Bq-x1'
    '*Iq&hM5@uKa?hcNLZ(yor^ky~nPcM;Uo1ZrNIfWn=BwU*?Y9zx7P>*{aJ=_=PaSDf2NjPGv+xB2s}Y(y8~L>chhmMoy+for%cOL18sD9P)>)hEZlFjv'
    'XJhU^uhpmUWmA1olWBPfPtB)2XKL%_MT(r#xIBsJ{$*^m=XDuywdY1`F8a^W>b3uVd`=wuQl%^|C$u~}6XGkVWDn>HnO`O^gyJn=8)-pYCrl|$>kNc^'
    'JAz`@Jo|QQZ=9FhuI_iZqUJM)K=0b~Re8Sh&v|!GRav^ND8T6@o5^kU7-BN+W?HzLcAi0F9^(*So1O%FblI0T=dV6*A+o%86v@~$;%nt2)=8i`8u=&4'
    'c2+5W+655<!0&6fRLO{1@hi<%q5vtFVGAu${i^d`Hk2HGhbb?dlMP|r=+Eyv!Hc9~UX#a#;Yw=7@_OJF!P;eERN-sjBo_r3L^+xt9Bu9jGTrUens-%d'
    '`k6-6eAscTUa#hK(0h2pE81!l7M>NDoq&XYpD8U0>J$8Xy$U*8lnQEXt9MH)c)1x)Jhq?RTCLr5vTe__+0|n@(H)@r7yaCI$cZVc@-PB!)ag+tiv6zL'
    'PEi;C`i(_x8(_C6<ZqM%e|r@JuT`qxduWU6TP6x>ajaP<5}Q^Q(tc?jjSY!fyWn;&A0nU>yJOTvSv6O*)P8m@JD%3M%yb<Z6H8ltiRPR{8$$STJ85-1'
    'wuP{rg#@&$$5vYq2kI3spkGl+z&3N5f6#7!x9FulYK?GV(g?Tmbp7diT9fsbSg*70WJ)D$6>co~T<BxWypp$IQ7gW0+30-TUawF2Ju>TK?3Pb!+j@_@'
    '+qT<mrK6)%G{>}{iAS#(B9qae7s2_+kJ2k}H()^$u08L#u>-5Zw0_2hQoXnMMq$h4W)`a7uIKW#*4nI%uOG~U@42NQ&Rc)IvbjGVb}A$cs&pmR#@`{`'
    '8EuEoMo@Nic&%;4n{9Hp>E_1w>E3>R>giXTd&B{Oq`T5EC@K1j;Zh16x8-bA7)^xk5hrHDazq}N_PjUYg7E|mOsVy~h_g=Zb}@&Y6?Q+KW+iF1-kXgN'
    '%UreU=S5I=1g`LnC5oPWs}OD5da>(PPN%XSS7G}zpUn?Y-)xxs6jmN@%B_E{r@-R-Z}Mpqy+jIGr|a|bInHaAT}pY3d;+br{!pwMruUDp)az?<_jwG;'
    '?XJfkUeXOId^bDf#DA9ja_&KJK{?gB(GBMO3YbN(jBxjPQhqSF*1q@4BKW*$q|-+m9P2fq#cnBs?z^}YK8HtkB#ZI6`HBPiKIR{?c9{+apSj7iv+&a='
    '_J)TY+uy^LDC#u~{ornYQDA2r$b9i{C-dF?6uCr{ob_)Q+zF<b9X37M8?HvN13bInBV#6by;(Gzpj|Y8;zPZ=&(@@^HT5TQNQ;o64#DtLA>`{Co3vl^'
    '!a0*$X5obK#X|P6TG3_kdTL7W+B|k70AGiCwLkUH9&>J}dTrj#FJ0GO9Wi^hYV{zk8;<JepKHqA%Pu-!)W`04abQ3?@Cx0fV3B4u0p!5R3yxT6I59q4'
    'YHn8`%Ph>@*Q+`QuYG0Q?=^|%g<MIQ?X=#brvlonbqL-=Kp$JFzD#eWsDl)t(~}M0x3PbD-}WQw4_dne@_mqx!z&Y2;;<5UK3zcB=PmCRBU(n9o`0%U'
    '9QVy0$JFxHb?dAk(AEb+nx)FrEb60eb9Ud3TN`_Y*KNuR_cOEzHmI|`X0Np@Z~tqbT;JPtXVXI0Aiw)AW0+d2#3GQG33*3MLL^jdVqsSkdE*MY+Z+nG'
    'r!IrhV|`FZNZmGAxgRNu{qCGC=*7oF2_fpYyW#NuMzV7pK+3l>sOQpSv-p~o-$Z&A)ZNo}c;u#$HtDl%K^5J#@nFpUjF9UU5Ps|j(W=<5aK!U&T~u!T'
    '#P2E?-cQxfwa9n{b!7SI@wO>f<i&C_Bf}JgGArnP&adoHxQ_5>tF-koRa2OT_F>!ifuWTysMll+eyywqeITe3!fOZ!Uq!qRT3PKC7Ojpk$*~3SxtzPb'
    'KHVge>%+tP`?wG7rwmb`+cmZaeQyVVTxA#`uwQUcBURvfwE1<yMe;0J!Cr%03T(5QBd0QAXa08v8dm>PWV3>3YTv}=dgwub8rgZ#+a~9X1mq$fe61Q+'
    '-M>lW>C?gO)fdYvrLA_Uao{!EFfzS6t>NUkmCx>ovLK>hHk2FpV35)lkHq49Q8xGOuU`J_<5KJ-m<ORy$EuR{uFdQ#FBRO0(965`ZB2ids-@+t(_rQ4'
    '0?@#Z5IzxUc1U$aM;=L0-XK;2k(=$bB-}Lf@}wa4<VLsV9)ue)I-5$PVUwMDNol*g&;?`#J6Teb3oQf?2c}}gl8YCY-GX+{hr&Hx4i`x<e5=Ri!f<Po'
    'g&>~qVkhXqtM-h%6=NVXmm~+y&Jr!ageaCQCPfD&{Ca|$bqW|ky!fnu%ZgS?><+=SW~&0g5BpLa@CqWLub3G-;$zi}?>qnDtzoNi!oGFC22X2sdJRm7'
    '&1P{Qw@Ulrtv{-6DZcfZ*Yb<I4!f1!Ih_rxu7yX{u&=yTp<1aZo|e@ehH#ltk+zJ<IomKC2#V+O;}kU~B*N50tg_`oKc9a4*TOzs-f$Yd(?~&V4PGA8'
    '>|lV_V612p28eSn2*KceSzK;Z?~Nd%_?m*x7_44Co2$>eH-1yP^8jibdsN4q%tK7R>NUR#riU~-18-#aj@;#nnAkL-@h?u4*SkfO6t3}Vv3xzIP(_)S'
    'IA7P3MLJxlwHZaUA7^2=AexmS)V^r%SvuL{yJx5<qf?j8Vg6*$(n#4Ka@zL{?MD2kvV@^vG4Cx79HzVbrT~+pQ^Rq&+I5CEx~nH;Pd)i&^wv?W27XLT'
    'CJthaer64)Q|J_&E=~a3Rc+M)k>&0An#fZ0sz<BEVR;!@VQW1@CE06c{voXl#(DL7-DlqJ`ldHqLKb$K#Ti|TMn9d!^ou4hdLWGk*si$SN6kTbxz=u5'
    'yCHWhw5fsiAe&Lr2b*7-{i+_a{qktsu}iT)7xxZRoGugxp1xgw>9mfe)@H+Aixu_G?TpUEr(_oJuCsZ$WsSL4;r5(0n@#W)my#%=O2JooOwbLY(F1Z5'
    '$4bN4L>4C>`_FMTjj!CkB3*WrI%ad9br%?>o@5_{i_8^|%YDy#RKwY>!&PZG#J9xq^0EEzsek@bokfWTXM2Uat3BBn)Q9iMpms)i$+#sab<mVh2}g|6'
    'LsT{$%+&TYt3*t=6S~9Q;C^q-$SXCQ%foOtJXy2S;4&EDskeVSSS$N(h{E(kFOI~%We%-zGaBiS!Rxcz7q>dP#0bESxOI>0*ueAAWgck7Ty9&|dWOGB'
    ')WMMT_Z`<JYfU&FIY}MdI^!kxIt%keG~4@6>)8h!X@8zwu?!X8KG`3^-Wp#R?_q?Ox}C6G_V@0)O<c;0)On|0o`IMPDIn{viX@=WfeaTqiR_^SI!nD~'
    'AMT~^XFO88JIxH18ja5OvwTb`3{VAk3eBz<*jI|{dfTKOPpevSa>QEpODLaS&aAepu5P3;dmk&c651+Vk)|c=Un{$Ci4V8gdrfYppjzBF3vo1hyT?=O'
    'O9wqHpVhm=^wA_1p)R8rZDjW~9b%o1qwR+Lr{~-!f_}N5O2eqP5zEqj-D68UFe&sX-rIDp)4i|SUN}BAJtQF3%&@h>4W<?up^JByWPQ5s_>1g4t>MQi'
    '6~0&^3%>g<;!fS-@VnvBCRZ3m=|`s94by;hzCUiKOD87$_weIEVv4tJm$&(>wCRFVP{#fPFebcSt|tz+U0#RG1aZF~k#60~wZf*I%`ygSbOBl+N{3d)'
    'dMm&O;Gzb(VjA;11UkEOEJOT>-FF*Bum}nPthV6dby88@(CWQ}x<F^Lu9?F@m@2y4d&2Q*XNe!&K&NO-kI!TYsO*wK;U0_8xjTV4L0-+U$M%gByEH<O'
    'tj@sWmecBpp>;jO^K${o)ZXsAZ<J;w6-JNgL*GsC!2@fvZN5{U&hBby-MmgzRcbP&#rE258gJroE7u1Fci+7X<gR%kuP)b~fUwWB?+raBzFIu)HD{z&'
    'R<=aoQ*41Yn6Xg3Dpsr2QQbjZMSy_g)-}-ul_9vC$<gF|%~BeyRF{k9Iz@){ZR;?B>ui}=S8J8_^IU0OrV3@54qWIb)8&~1O3Q3t=M9azZ5>D%ToFED'
    'jCgVYp8I|oU$$Oh3G7YXa6etJjWPI8;87^a>x}TW5p?g@9E#%MX=NEa!Cm2?@|*f_G9ML)Z<WA%59Ly_m*B!!p2>qnW(+dv1fU_V0$RAHz4O|s^*Op@'
    'vDoK?PlV=;4Pq%lucX(cFH%^gbCi|_-$!U5i^=IkmIlZDqau2P26rJH`n@e;!_EHGT=w@sqHI4hHr1AEV#_Iyo6~H-a~r5O$evNLtB-iQKes)uuNDec'
    '|FZmaS+f=&ZRtUN<mSSPvMYP7s3<@a%)C22=<)Kl`SkBlaoR3h()C>6Mc-tW3$H@Opf^|Ulx}mHRDG5@6YS8dkL>eRfZzk#Q>0l$w&U>ls8p+47Ta$p'
    '==`%Rdj&v8;T0DEVdFl877l5vx<zGgD6~NS5*@erEiXqF3M9m4cF|S@cfCIWecE5mgjYsaC2hOdQm2V@lHwQN+Gs7c_1H-DY6xHMh#1UgQ>q>%^T)Pq'
    '7^_+0hVxHZrN%fcYHr+`PxjC6Df;wE-DXMVJJkZmtS!6A`Gr-P;iCwq`e<?BcS>iwFRV3?iTlB2eU9k$NL>!#?s0oLBwo+Kaoo{JZzv3EW<P3FJ9h7Q'
    'b?tDpR~j)JEUY@zj|eb&@?vdN3r$5<gFT1UtNDB%m?NnPjOTS&V~~D^>N(7tQFJe!3*osny9}d;K04g%vgG*G<5eqjeAlcGyQ$+nh08Tw5;#O#$AGvW'
    '9%maYx*c?Q2!K6{8LJ5yNQVI^EX8FQSBh8eJb0nIbMZqJ$Swzg%ofA#KB^esUT?|X@m>j$=r(WJ=iA6QQ`%Ud=acx=BL};~dAMkw0JZm|=Dn0|wXKMg'
    'FZbJxe@(}Mdm*cCRT-=BG$2gTI=qd`aes~OvvA4L#F_|QgnmiFKw0}1Z#L)O3UnjX`fUM~AQ-;h)>FEE%{yF=AmLG0p55@B2?8*DWCV{`?B6o}ymn^k'
    'RZx4<aalnRrS^dL4@O;+>!*Z9ar=RPCnq>bjGjFr&d-qU&1=2hHaoZKB`TYao}=zcNsoCnJy%IH+h18MS==|qZk7_&?>EAo>UN@d2rRD`1BGq3_sjOp'
    'i4<BM$tHpzXZ%&4!Fi+o$|YW#c?e|BtGCDb&Q6=osNH73yIa+?eeFsgZSjuQ7GK<fZ`&nMe4fXO(H~&5L9utfF2t)l2JXi;q<Re@Pg-zWnlqG3^i<Z%'
    '-Xq;WuuvtaE_r=)kHN!SzxsMR+I9uCWe;~xII_;EJuSjae{%UYZcyL%aL_7siR;esM|hd0#(H<ow5DXMH^b@%Q$_nx=Ne5vflk21{$-eMWWSRk>3-od'
    '6KeC3(_wDFM1}?<dAPW-tW~Mfbxz6VyPHTn_UV?V+?Br@*ITW^Yiq3^1y-7`E2yOEt!h?4i~V$do;E&*&ju>B#%r(J1utdjdE{4nzTr)dQN8`G-sZrq'
    'kW?3Sj*EhGpV~f&<pawa$dbGm?@{LC``uNyas2DcQ`Sl~G^4+|x>lTnMty<h{Se#DK1s@@Y>Qg$?fhldpRaxPsSQRUXA*r+Kct;zA?gk@ySu!UuJsH&'
    'yZV?{*yV5{_WG;C)A3I6z^Hhh?;JRvJ*Vo#xg~I+OxTvt)Y~e+Ojz04gXXBYHg_xGwr`YgmmytvHT83?->xo4u|DH#!|@)2+g!a{!}ZYuS{2WNbG_o-'
    '=@eazxr@-ER#R_1Xu|L9R_Q%V)oG*>)2~yX$$Na(hLw2jufPE@4=Pp(tuZpZq03Vie%PkEGA><VjY?d&;SqyfZfh#qBH(HRG#eg%S5)AXZKQ<gZ!S0W'
    '=7dYBfv@u!(@Q+I?98c0r81(}Dq~FV?NE5gLR9+FSI(-7s++gc9%1_L2RGrBL6Q>V4>3**#F>XLnU>v`w!O^2tf!CIFWp1%H?tU8Mz6faifXG}{%$+P'
    'LnBijF+M8F6aLEre(B|gv*;UqVFjh8%?*c3g8>eo?ng4LU%Z_T?Hg(nO10)T-Q-2lBG66T7zAiJn>@ZVY{4W|u{C*>9ked)_&}Y_;!Qaup&TLDSF}}U'
    '%4^^}m6ex;S6|PbY0=8Nz`m%iAkKV>0`8*Ix6I;qUoFjBYK)$}@#|R7>2kb=Dt+EcAMd02!YCI7f*s>eW)~|e$-8+M+~(Me<knbufjo)na3#XV+xAD;'
    '{1)<+ae1C7JS^mDi}YTn7>$Cus~pX&&lPo2K?X8KByIjX>{ST%YFvU7qbhrE*Xb>)EBX!s?^hTFP)BYX^(zyDEx>|%8Q`l`6RIyd?_z;?gK@X4DigiW'
    '4n14pLT`02E0$O60l-EeqEe2)+3i*QWSQBX)s)VJ6pqNUtlGyCyghb(Wg7)ftXXCM&K&_<5MnA?_9y(fm?NnIf?XZSI5oF>H%xm6dfA(Mcgvx=aCHML'
    'uy$Zb12$u<J-nJqcw+;YqsU&ikX@w&ocD)qx#20(3guQN^2Y?Np!f7p_hA=XiWE-@R~Jc5ZI#5{d-U{1AiSIi<(`UH;U_?jz%7c(-S0Q6qjq<YW8))~'
    '9MHOP5#ERT@dfvW00UOmxs!`!?6wMQo4piMe$rMauS6A_N*obh<)OXX!|+LM(iGLSP1lM`^`v1c0MPhA@8R(1E0;^XTR9Bq*3}fpz;1kG+uIjY5$eZh'
    'R3fjoHw01-FC}=&juM2N>SMM!49GoqYZ>nj!1?QRHr-V0Mfu+FzEz}Xcebh4$R->woK})N_5gFfS_|LQQ<ha|7!IAd_1X45yjofm#j0d&J_L$>`k8G{'
    'x|iH_GpQyndhBAXPvb3e(#j>Uq0@Y7n(l>XOuR(2R$R@Q$29|r+MRpar(Bi-g|YIeZ5FRJ71kJT(CF5hcK1FcuR~GS<~NiWEa|svXJa0ayY>o|N#(P9'
    'R6ora20!IX@3I=NFn(R^4#mNFvTHHOgRaCMak&Xb?Ur)8W88Lfl^-|iecJ#BrxFXrT{fnseR^jM*44DNk`H!u5{h<rzEO-dW-k2C$PymUy-Zf}PEU;X'
    '8=<(F0@rALdczMP(X*|TZD0Droiq2~F^OwXA106KH&a)@1DO@n*`a#Bz4JDu^J?6!&PG|0;5#RJ-P^lEr@QBwjE3nQH{3VfXwQ>L5$eur_PqHG-kAj>'
    'EAb`z1LK8f_E<2D*m#jx<57sX+p2}&((O>OX6M@{XjYo1gdg`HXw(3<%~*Ay!Xx&p?NDMmX-D-)DNVPsbkNcMW3D7<kML@@v{UGMRrju}oD=yVt?6x&'
    ')}6(KZXQ!ZX=7tadjROtM=O$EFM*FwB~)g_O1*S)^EaLk-pPJ>z{T9S1MN=rdV%!U*W<lG0hu=e-b*2V5R<+u`LA*l#<4b)qzP3C-&InLd<@%;g589-'
    '@X5Xbo5eys?rHlAhIC(}@#*!B1-ns?6K{L^^m?nuqvV#_$Q)X9%myRWA(UtkrPn~zIQI>IpyOJ2mKLlC<4M+$ix;+p%iBBE08dx8cmvwYkyKb%o5=a@'
    'wcTQNfo{zrVzg%#jXK@$xl6p{#t4wy@S%Z<t5OHFlxRr6uHE?Q`PUd|^o9_+Tn|lpZ!{?CM3ukk>eB{Op!p`AP3u62++$iM+mw=yAGLD-DUVF@Iw}|a'
    'CIVFNs(Eg^%3|RZ!OqiaSYC@ns5NV_CF0mI%00J>Y9jQ!94V{_F3XXi(Yq{5nCAyA9A2zB5zv57D(U{*Pc#hdICmVlo^CNW`HuUIu1T+tsC0^Xe<Zw$'
    '@wN0-cS@}zfUtf~*P8>Bm;=}ZYsN-u9`sgxI|8rXtR;cEpN?P9XXRJnru%-BN4uTDBi+@g+T1C~Qps@3n6N!R+>KXF!a>INaq@;bL(=@V$ftC>HRVrn'
    'a_Z%<a6axs*QsY<uQ%vjC$0N6-7aCL8!95i57H?=D2Ii`V)bMnus<EYHsDTl%I)=$ra`MnX_RiZ-Cn>$Hcz+oof2=W%@wIL6;#v)jF^wFQBXsBkIlh@'
    '$}TunZSwv(S}*5#R|;FYFsnT25vIjmYxrt(R0N&eMvY>_>$j|^-r215+vqGi*BMiuJhOB)IaF-(zN&Zbg+n_z<hDN3-ck@=?HfMy7VFV{+cXf3ykqhK'
    'ucJ_)cONY;;D_@kwclgGQrpqjk@<o&2_J6(q5;y|>Snrz8|~GzFh@SUDRl4AC$V3$aFn*Edk%Hw<4E4xw~NDFE1P`@wpV&*e{#YofdZiuik0hoK4dG?'
    'Ucr9W+Lz^yXeZsw!L(I<d{{q>F8A)8niuy(b;cs0A1gD6mEn5Z<J%*%(O&n-PJK$a4B09?BoJLM)En1wSB%XryH0RxS6bZ}70nNm7IxD(=hHZFD0^PE'
    '5v_GvOvVG~kx$&(aZ(Cin?XM8^m^$AZmh{i=q0yPShS7ZagebJD;*#zB8lQO5QWO-{T;3PBBoplSN%HLm(FlRFC@R`lCaK2FUnkZ1^U=m(9N-E<IkY?'
    '!ME{rHo^!o1d>7l8qI6|DfOv=g(CUoRL=JUp!%eBcv$;>Ym^A-(j3G6-Ci>9YtV_GmWc-|OXv;VZV}@9(oh7!A%buL+jL5Tk~q|tmVM`B?Vjd3`Rs=y'
    'W>mXk)%8ovffw&zRfr<0d47?rEUzlh_U<AK5BnS)ru!GTON8DUW>)Fw-C&QPxnuA9kKAa2MQ>DJKBcNsk~%v$Dqri>U3#sC=6-J+)LJl>doNq?#^YGo'
    'KPIQZ1l$E&D^!~A)lxYlJu=ilsG!Zyb@211(M#b(&Go%`ZMHr`kL0|zgU26-Ec(NQe4+*On@I1lz2Y`AQ=EOuiUpdhn`-W?wz)p(79nx(?KmZyL-}Hz'
    '^Okm6$_8H>=-EMgG2Ap%F{SNH)(7Ck&-{ChKtFpFx!s|YausXLsRaNqN9kk{4;s?Dc)3aie^#Mx;#<#{Ijr`7{f&wMGiSi_OTK_yz6W9TqLz7e`vO7J'
    'Ny<JCG_R)@&^Q#kS#mthndb?m;(}v?_EP7E^|4)^qqzI318Q43Oh2=6cxM;Q6G%amc7V^6hCBNz#!-3TQ8bZy^bo+WVyAFyhaGH&%j;DJ-QY1;iBR@)'
    'Jo8^gT~gH|Q_*Aiag_VYmc**jf*(t6)-0ZrT3rsIlo;-;P8s1+;&$%b=s~3r%?d=pK|9b(x(khdlSLkHaqnN1;S#)X+$dY-v&B%UcvSV|4+`Eav}Pl5'
    'SG)}-Rrr!20j7YH?W^fE6I<FXcl|^WuN$Io?{rUuPD}Od8~F@yJJKxZ9K+8GH@}Mo8xJbXTzzlPr@M?v$3g%YGF_PrU%fYR0_GLxI3npR*tI5u^co5n'
    'bG4K?Id_Qm{Cg}LrS2F@`%bYjnH+s}>I$_I5*&wgP;ksqzvc~V)E+FTslou@Y4y}KPGdmjJ2(l$^Q^7_Sy~!~%=$WQNak_lNViDQ^olBH6YxT8(9yY#'
    'jg}75*~!zfr(#!c3RIr#sb7?7ObGhB0;Ayb#%P5+_gcCr5e24MF%#qD5Tyz?gI@G|^yPN)k@3e5y4e+2!3G*%l-rJrfV*!Q*o2QT{~^!I+4@~tl-{VF'
    '5Hg&icfY<HwhVdz51NoLc5Wo^d^jo=u(cy%2+RpiIZGxC*|<hfuSsKUm)Y(}t-5YGpA+9U=J#zWnC)S<KI(7xfMslgZm%URXowHuU1@IMt#7#+ItY3-'
    'sA=G@q-j;}NeD_dO6Nqf`F30G%+sf@t@v;iweVKfh<0wkY_w*2VfyGJsI(?`Uoi&;W`kOleAsT0?_*23i@L|+x>g~I<%+c|<L0!i@&dek=*&qMJJFrn'
    'p4Bbxa>h<S;H?{7?82QxbhvVjV`i1E7qYySF9SlmbwlSNP-}P|-%bxk=o{%qm^B}R??m*>4O`jC@W6LV;v>FHlU2Xe?QkPt_G;`?mJ1RRNc5wpk~lfM'
    'F9CK(6Arkmp1hfItlloXBtb{=^(<1fWgt4kxaQSalZ)l8&AKnyosUif3MFUWay~D9A-go3)N{Oa7Uwr-Rv$*seKfCJ(NX5!1mFRlijJC^S#eiR4$SC%'
    '_!N?9BCRTo?dObkhkN>{>0Bw)JZ%5?c7XUgWe!l^TQmfJvj_}ty(DU`9naPg`n)=OV~{oH&zU3~n07CP{B?Z%H031W_VQ+&VU3}@=(#O?roM@06=V>!'
    'T&4W;aXZzY?r`|F*VL>r$_L?fp}MNTM%jX{V?AUPhMnMZ+NW0^45eQ8Hx*_M8f8-YCGxqiD%Lg)g#hm-zWBhZ6msv(25`N5u5Lcl+KTMTr`^pVK3(U!'
    'zSPIhroHLI-4kgLh`C$Jm%_RFW~SkW8V9w*B&*-nwf>mt)VsY3@;2L7%Uw*DsMJ~6heq&WHlw9kb;KD)eEGzFO&7SW77In?y`Js%_xAtG#kn=F=k;Ou'
    'wKUX7(+t^<O-dV-gdz@7R47IJqI&KB?{A*7=iH2i&${pH8Z%2#iWMVK)4?#EmR~L@vw?l28|F&BeA=xse3w?A<TlXl@J8AMepKIR_oD4b)}`Bq#Ovgt'
    '(UiS~rd|V4j#Uw2C~vkOVS4ot8tkP&O61}=l%|aVGN>4ps5ZXrcwV4L_JF{!3nG8FE|Nug!~cBMS7;EAwr6UoCG6nK>R=y#nAyFrPr3JgbL;D?aL;Uv'
    '&c|;nq~2-`r^->j-x~@2?u6&1)M!fBP=Me58Vi{K^i}l%2&3%uSl;?mov35U-m-bi);!+U^AlT1fE!i$YsmOP)(pqAm9IeF-lgJyFVp&=iCvOYU2Fve'
    'd6BW>w6+3<K?m<J%~E?lt!?=Z!+V#U#LfQP2T)HA=L$opMh^1Kk*=TR6EK&o_oat>gB4;MvLD^X2kNkY8N-nXAig)eQSA;q*VRc@5kGQwie<NNa5;z9'
    '-W2aQQXXZ%Us3I`oY-d3qEi3sgv6zr2%XwNYt>zk4{NYU!NI5sef;F!AFrL0`A6Z%;04eU6L;ETV}4fia6>H11-GAk=3X{GQI>zrS8LEw3WW<l75=bV'
    'yZUu6mz8jOSW~LV=&#eZy<=Cy)~%r>)7V?|aS(io!}g|2f$b$%rFd#YRU`0#Up^b%pp%t&)T$rwobR?SK6-)8^*%~BTNjFn_3YC^y@q%-dzahiTOBpQ'
    'CLR;zQ&|fgn-$Uz@%m=J71>zb$5Km}_eYfUvZp6I_Lt_`|7K6;S?8Fjdb?o8bKM|qur}SF1T>Wcz7taKEnNQQ*0|1}=*LjI?yi$pR*eAoM(UMT{Y;&2'
    'ca$wL`7bl>iw|=Wm~@VE!mBlaF2fbsejFTnYM9bCb0cXD&JX8%^t<hLX4UQ8sa7?p&$&bOdinIml{Fmra9r#5e4xjzaxRge1(t0ZX;80T*Tp(0bsU9b'
    'VhXvS<A}r8t=((6>-*ig9TlkAo~T#9X)o^G#f-v@^M=Q`2We_-b3b}f(|Z+9{&;(cucuJUDde<$Jw8DFq_#fB+b7@kWpKcstQ1vsUy6}sI_E0Ox&q5f'
    '{Gx8wJu7&QI^AQwxwGm+zN_v1@S*q+V`u%}CYsUl%vS<&W&HBr={LXbwgQfs4GJcYmH3Qsnly=%+bp{_Dx=*s$oI&iXn}d%-8O&kc}0V*+AKZZ6IEUW'
    '#CWn$uao)jSZOLwf3xsftB%8I-Msj&THQu_3XwDSs5RY%msDTlWpEB$Wg11P0G)!Z|0pIq7g+Z$0(M!%B3Wq7#y-HBZ~wF0te~acW%mL9{8PBP-5f<^'
    '!<7J598c>sZlZ2r!XwU#QF6siXXw~P7Nd9n)ViNkQsSdS$J-2SWqmK&$78;i=E&wR<Mn%I`_R3*oa)pzVXW925>4z=y)OR!T-kLlxs8nRNvMwb;d3Q4'
    'l*7Etp+;|8kHy)#wn98kseQXPAP$3FiU5@{0{!NxJhmIoXf!LDXnliW57~Nb4|wwQcDtx-M$R3N%bZ#F>R3}Y@kL-!wGzN3^x_Qii9R^EZ%}<R1GK?u'
    '=at&w1_P$pH<#6!f4kpQS&PwJJF9!TJ7heVqVCgXMN@~^i`5&C+%luK($RV2Zy#)rtY3!t6<hR8{&~8pMOYan^FvRHr#;?D`m%j%CJYZvx?nb?=O?$W'
    '45D8}<zwrxE*L&w>`luL`$}6d9fA;W|BsU?>DN;j*l3>6;&zD<1@2&dLf+p!-sKNxj+=#Y!VQ#T3nC)9m8n6OZ7QM|v+rHjsj-c4)4ttM<&c^6Ywz+`'
    'H`URT9mK6m@!Hh}tohvxsuczcx^+jXV#D<M!=&!d5Fa}=gNW0M+oPdyvgG#anG3qGb<D3d-`VBbWSPe|d;|?lKVh_;JjpQVw-z~6#ut;WfyZr+$=juH'
    'oeURUNwC!`fxj-JaF5aF!O7OP<avRYg?c=Loh7>HOK)O*-d}KF)V6}d<IzsYu$UakYZDe-Cm$Nrn&sn5DBsFz3;&^qQM2GXTF<mFf7sn0pY?6T2zGUK'
    'Bvx0Uh}Oz;x-364zPV<E`H-229Vt9=>|Bkt-}(~0YC<Kd@865_;XL6_Lb@;VtuoV3XL>U!9lz$IqIbAIdU;Y^Rmj=gg*S6BCgyc<8}rzCGp%da>s4RA'
    '<pK5n^_O3qA)krb#P{xwPx(i3XzBFu_Ifd;{dI52q~))Y%~C!7{&zOg{~wmI$c0wAf7M|S@TVyTCw&RqQu=Z&OhLWamKjLfU<6H#6S>Ur!{qvyl;}3s'
    '8i4(WmE%W4U*|es2E+aEZ46~p+O--z%bH)n`RWYabx9T5z_Ip>7juE`1A8^uMQqvD(>P3iZ~>5Iw-^4cQYa^beRn!P&aTSf>YYz&^Y+Ic-StO~dGgdi'
    'F4hQX9@fw4kqB5uV`!Oa)H<7>+Y}=@=*aClsYBQPcjj-J*))RY3^_LhL~e~)#&mMdl_xOKHE)kmA1-@}-Z||hwX#lk-pe|$)U{jd1?P7?pKf$wC!zb3'
    'HJD49TO9&)Oeb}9Zgj*Wzj}_YJ*?hZ_UA+pKmL$ldHATjEPD1AKSJ;QvsG0wLNZjngF4rbJHTc3-p^3<7*^az53l>p`O&GuSJ6}(^;8<ZHfp*^z3pc+'
    'xNeg?`i{t>d=&-+QRUd<@E2sN+vDG7qg6Zv_>dXDx$k0m;V+s#wGr<Q1_z+B-f$IZS8mN`rMCaluWm;k=h8f0x4!fFrkc**jst@eZGSqaSYodRW54x='
    '9}`IE-voHJgFf=e^h&%o&bE=WZ4wh}chd0k2w!rk7%~s@4T&61nB-Ea^iSsMQ<0C}VzC-;`)6u3B87HkaqCxn=21C+sojK6Q_1+l+&tUC<q|g{o2H#r'
    '4}WY(5MDgD#^Id|!e+C-3tSp9I8`09K}(N_Es~VsAnRz_IkFG{<40n3vd>E59i5SwNsj@N?qSJ2zI{A_rwe(Qf)DW>w&iE#eQx)LHE-0O&<aj-120r|'
    '%yDqW?rEQBO~~#P_G>m%%Vo;Ffaw0_-fhJAw2P$Ons3y~>~5nXX-!yZb))w<jrE702*C%*<+WdTyb7>;O#0-5ugcxcMDGtL$5a$QML6j=$nR2Cf_K&V'
    '$eIY4(PWxMUJr@AbMx@};6}y10OabE*1pTq<JYZ?ity8UT3sCI>-08UzwXZ6#s$Nn0qGNsuXRP6-%pV{y-cEmRPA-<mWNe}Y8~2<-hFe1y`Ecv@pZo|'
    '4XZc)CT;(=;{2F?E<Azz7hw<?ur{4eZh+rS#jbs;xn1$`xvT3HS(F|1m=vq_e330`3+wolpml<L4WV=V0nJ1eSE+sKWb9=qq7QaNr9@A1CnCaUYzL-x'
    'T->f(X(^6o=V9c(k>lbOB<bBjlC7Ohm#$I6q<#OeYAz9)3!9hKX()>G1FbFfdm8`=DA*qlG#wp&JZp#0l@o6Lb4C`^FVU7E?YUb$t>AI|O}o3%w`dEU'
    '-s67pZ(7B8Gv`q1P^y)zKy~9UbF6_oQblmg`0#pBl}UiNI*oC;ZF6c;sQ4k8eZgLZH;_hntP`hw{XrGlxIf%yg>=BmcdzE*ugZu`3qg4U&C8io-6{X+'
    'Ar+^dhG1Dpz%29i>G0uSF8O3{hs)lcL~EbHUJg1|RM(M<d1*E<tV-=HIFA}I^%0pSg0xcxnjZYWl3Gkm+Et!h@aU4)rw8;1e-qudBUA-%{w$>yK6|@-'
    'cBczAtL?J+M**52<fEP{!%}n8SnL55O13lLy7!#<?mgEFYG04+96fBW81%8F^@_f&UoPhNyN_LSal5f>7wwBJj9m6jxyqx%s{1CkiqZ!~wmJ>gi`%?n'
    'oe-|EI<$jigPX{nINfpxBXh_+$QyAFKQ*HG!}}xR^bsts&-_WJw%@E+w{UpqJQy6WI~ob{o6w%M28{KG=0_=i+ol5}t#e-<o&?BdhqrQWcwLU3%|7lp'
    '6f0ylY}~XNum@In3?yrMdg>iTD%RosZHcsJbbG&S4#@tx7_Hj$#>=->7p?avDORG~`2LGtlT82irpukv7V7if<)|I0I4!=Sz`mo?37xzqbN=4}B<TVj'
    ')ZX>eY*I^~4+A?d0_UL-AMhfoP}z!L>qgct&-3tv_pqXm=$$N(KKyGCjLrLUqXe#cRAaYZ-SgI3hg(B+uTvIxQH*yc9DHlF?Ae<UlPi!BP}6zXS(=2Z'
    '%;otHuP5(KmpGpOdXK$eAY?30DGxo$^(1z={-j#T8oju}JHeks@BTPNOTVsAS#7r-y>DJlHklNU4V|eXya2O!-|pGcPS-13uSnwBqMe1#8f$-tkKBtl'
    'oz))eFK<%VdF|cS3^XIv8zHeU9ItTog|B-x;P`|sMX&S*{q}Dpy=Ozm)*OR<wuQsIg)AyR7u>auXMFK&+*?MM32sxyQyyc}*=q&-qu5rjwcH)Ye=Yqg'
    '7Nl<M>WVNF&FeuL3<o!+vb@O@e((J`4*5nPZ`BorbGo}aF}2=X8GS(P@Ei39<#aYVZaxl2iidMKMwYXWRIE3s+N+BW`R^c#7tqJ|n}t(05-lp$c@6#)'
    '-9d$G{dIa+RG#~;z6Oe^W$i1HVrY^&zR4)#cJ{%I?+wlVa(St0V~q`GLh?Ix6#GI9<>uE~6Mj6Z%-&}7o0cH0-<9(;p>K|q<G<db+Uh;PHqjg%?#yJC'
    'v6a?(mF|z%%bGm1kE+H8&IU`?GIo>ik&_5TCG@t*G$>o#(G9;Q8_neHi*FXSe2rr5LGn_AiHhnPfb6Z0*+9MA5jV{Atb2xK^k}wCM0zbbUEe5f^qR*8'
    '+(?%BqgA2Qjr6G0nrU}BOrsvdjF;3O(I;lPGLuXd6c_lA*BNpjUH$R0^R1+xUkNtw+N61pI-4CPt($fFWJ98S_$6!t-DLPana<h>UPJouhfR+8u>x%d'
    '=j6Tv2K{b5->n?aZGA_CU!|)X;wp!s4z+SvkN@-lVp}$wV9OIUy4v`mKDyO5mu6K@8^3dMY_-B{gzbp(XA6&Eay7#$1wO8te8#+X4VOV=b03T;sE=<i'
    'HpP~8e!9yArDdF2*Ft#r$X4a}COWGm<*kF{xRO8Z!DcZbfoxGRtd~gkYdUw_0JGx01OcZ`h^_DNH8x1$qu^(yD#iNO+aJ*peJo7xIa*$td&|0vJ2Kl4'
    'G=55-(+6SwT&xlwDu_eUUVh-X@_AweINTHsYZs*29IyuN^PZk=Vq`Z20nlevAm%sdf<C4v*gc6?y3uNjxg;27!=~J`<c_UVRb0f(%}TdmWAF!bzohvI'
    'dhD|QsIDKQ2t-w!d_Q^m3Y)cKFnA}`4jL+c)iCEmLfgWz%+`QraPnZA-JHj*EN!f1G-tct?bP6bAsZ82eFAhA#D%zqo=8g0C?58{dT0=28!V0KBGPP&'
    'XvtE)1XV6N7?m#gX={1oE^p?L@q(ANF}xXoBEU#HZDhATW9#jY$lzDq1>&LnV<3S6-?!mx)RARlfb7?e#jXauHU-c++<vW*7=~4-zn-TMdVn5U=jlEt'
    '&Yc@?Z-p}a!;Q|`4Yf80Zg{xLlFUUNOwY<`mggp$tbVTbO13%>9C^zH^o>2knD(}p!{JS-X4B_k8Smz=F71vu9|2o`M*Q20GUI#>sG9#lc;-D9%=WYW'
    'p+jCDtMfADuIU8-=7;>?b=$k!iCebT?NLDO>Pc5W8eXMI7n@)|+jj5Q?dxodE5D8Inv_93S1+XdnKpu1U1-!oZhq?G-cPBRTs&X~PilMr>w5`H!}n4~'
    'wLv_!XX>=S!tXm_zJlRvX*D2!;I`juDiqULdHlu}$uz5?+eBN-_mz{fjCF=Lxg~GYH$;3Px}@_dO}81Q4?*|2cx~&wQ0NH1<Noo(iyA^T_%9tmHkOGo'
    'usEKt^dg+KknX&*S|5*^zh3a`9_fph19%-Sk?N(s(e;+j%6zQM!;?hjJZF3LPS?Ho_epoe_0rq4+1QtNhWhh2Qo|X}OJu%R$aQv#B@;54`6B`VkQdH>'
    'pX=ALUEiF-WLISduPofuGH)}w*_X|{jomi-#%Y~v7jJdg+}F}XOEzVSR?OC6Y5`vBg_GpIdwL5kX=bodx92xq?Wp4O#MHMlK7fsdd-y#3msq8?vW+b('
    'D9OQj?~T@RZk!S9_YnOi9W)h3^TF@FnOT=LMs?0|4a37Zez&VMr$L=Xd~ucAMs=U{OZCBCR-=nRb(XUyc!MSue^gpu<V;Mjt1&~70_zU9%CY(qH=|mD'
    'trkMGB=3#P=c(7|frC73!GuKQ>;sh&J~HO-hroi+=uT`GEYfZ&5@;W*DfmZ_hCATX!Q+CbsueC4kz0HxA8Wf9UEHuC&^?G20`;^t`1jvxR@7{#E}=+q'
    'Nj)6SxP!hRa<||8d-vnQHSGR6Et}zW-#R=VbbFzj(YswOE9ixTs21}<XF%9&Z`YgN+Yb-OXmDY#cR#Z5G+BGQ*{wRO(6#*dDl%Z97b>#`VI5HY{XF_h'
    'lW#KJP9DzHlPmdjz>j;%O=0XQ2Pv#@`^`4j$J^`TUC3zkf7kdo?oH}8aw=cb=)KKdv#h_@`A=@2503eet4v$(=oPffl>V~LQN}tAcisN(H@%z(qoMQr'
    '>PU4sz8g8(8-Tv}R4qAO45sKG?|Y44%!ZS75kvm`HJWhf9TFR@*erhCe7XJ8OL}mvu0jQ?T!TD1h%vDqM2is=Z{aoXHm04O(+h#aUQk%&xB7`2al6eb'
    'ahlDy7lGdphSS=?xT(WQ*%&I*-D1>TZ$7^F_i~W~$iE=F2-5@P@%r&f6L(v(r1~Fv!S;Bg68GiW$xFA}qsPsB*$3@q1=^>(J=|RN`_)@;1yEH8j}iwx'
    'kuSV3y1lnUUpIUfYGGL3CW9u0?BC%cT+x<=xtDPb`DXldhM@EA>E}(v%dP1DB8Z*<w0$vqgTtMT54me;J}9;IJ9EfCzCXL;&o6oR6e?*$K;0Sk*m>he'
    'KseBO0kEgh#|b$5Tbskdz_p~rGWvcGr^{>@8eP5NqH+45AoPV-5n<VJHzT$qtOb8_znILvN+#9o5M^qY42!y|-~w*A>U7nq1y)X<{eIH8le@j$KV(L`'
    'K04&-^u5TNoJkDk2hoOzmWI@pQ@h0(9^S8w;9BG$ew4h~QQ6OdnK~heH^IItUKi9V8=M@@CiLCQ;8$<bk;>O7>t@5rV-IhR9jWc5e{k|JfHqsz-#r}h'
    '=VN0^(C(%#tDfhByNxjFw0yX&ej&LSJen<ZM)1WoA@E?*`@DzHfVv*Jm|Yf~ztZr?nJQpfTq%6%#NSPF6b=G+@tn@5mky(9=YIcsYM#GB^z=`SN_ELc'
    'yG&c~^sz+z`(`84jrZt9@A}7)unek+)mne0bJy1AJ;828fkwsk#Koi_0I^me>?+qrq%MiZXS>U?f|+A1!YghoWrigXtj0rPDwQ|ku}yh(Ick@?QsZ?O'
    'rlXD<4HRo62hzERt@|I}UY841D9*pf*!kMs&w+K0Q$!f@>HhCCV)G?czrfp_J~RRgjpN;PRzJOvxg}=Ahul7bue=!(3$1zwiCXt63}4Htx9IMN_<Yp5'
    'j^uUhmC|mL9#&FLECC(-4m|weds|#g*A?JxJ{h1@fzQppd|g|$)<VGDVb)B%wLd=)RPAk<RA`y5wphwQ@u2fTp(pF+vAqMtpmM2q<~OEm1!A)wE`6ox'
    'oW+hXZ8m?GN=MOS@h~uim>HjkO?S2n2Se1ZhsoTLnd|4Hln%f%GveEZg_b#@Nq$sqF)OejY2fGns*_2ddXsmIbq7rCaSd&D?-UWCtmZEk6j$mZ9!Q=7'
    'V5E5{)Ykx>?`I1<ur9LnQ{Q74=<h#<9=#p6amp6by4r#93RZ6!gxC;}TkjPkAjz4~^yX0~UOJ8MWrvKlgtmly*DG&+F9_DJn{^=78T$B>A9+EXFUdaM'
    'ty<whEOLME@>@gT!~+vc(>o`}?|s?W?;B3PE~8+|sW3v-KkY{tH-ojDBz_+dL1|TQa>rx~AkW9F0Zl6T@yZdRB3BcB_`uBlq&yfvt5d9&vfo?2CR~T?'
    'eT$w$%7*){Oe!J$tA<un49|Q8G_Nv0EeQBe8&liku-}mWY)=aPc}O9`V(i=;oX_BSaXDe(ruj*SE$O%X$U)F*zqmG@P~cV@ERd%$>d+CtYEiuj70lmk'
    'W5BV}XSpYldK{Qd;kS^8aoRkdb&W>*Dq!5PRho@6zOd5@{;?i;Tv;L2F$r-fcAM>pQ{3XPO@N=^*&q~-Vg0(O?1P;@{Rr13pCP@EFxLr5Y>=IJ_9>U+'
    '{nmUz+gI}dar1oRHj!OxAw`D}uxkk-ck=PKXRW3}T{unAtY4+r>iCIoryG_F+N8?<MKBiMI%aw)Eom}^fX>p-VXvu79^UvZ8kfZQ5e^(^?79#&7l5n!'
    '_JbX~L1k$+9J_T_8}wfte><nx&yr~r4@&nY95<(7rOlTZpPfbc@fb)+f-HD7Pq6R0c*qvU9i**f4tFrQPNAEKh<(Wi3E;h2;w{fSc#+Bz-_GwMG<x>L'
    'FCFiPhm?w)7+wrsK0%!1d3_R9JyaFJMvWF4lz?mK=~fsIWmoNNt9@};&NQgmyr+!?Y85rFcTdSGG7FkBy`93}2sieGd8eA{NgDm`G)ft%Ku*}M(!nyo'
    '9wJt<jba6!v)xR6eNduSK2QXw-lwdsVfY}K&k*3(?me`xqAT!aK+rF|PwkdlECRqT7Fz~u+$G4sZYbUt<<Z@WOq|50ZDhE=&*ovqzCaVHl<}X~0uylB'
    'MUngItxlLf&*&TB@UOLDYzp&}bRb5r0m|K$wC*i*_r&};A+wx}({isO+g>AC43|ZzjyC&4myLHaUI1vLX(j?x;|#_3h38ETX^fp9@b9kV!IYln&mT55'
    'vfidPIC|RJI+KT&L+$;8oqrrBmUZmAY$}3RYUEvR%Zs}YzRrx*XMyD4{$!;e!@o||xqG$VP$4}QEPtJJ!W|Zhbocb-m;Q5pN|mOLn)?mWC|?Ig0Pyy<'
    'KHDd;xC=9W|LwQB-~M`Z$_Vv#tct5$I%UOF`aMr8d9($EQ_~zTCHMJ8N@(j!t<BLihi<cS%iO5LB-;ser?UJ?2pG5=dtZQ3>MAJyMd7rvo;19$GIXks'
    'wdNK4%itm){C2qanI8%Rdf)NAxND!yU;*U7#Gs(gW62RM9@k7Z<&*8H#{#=|3|y|X{`P63GkbKL)``h68>Mggf^-HJkGG(Wbs4b!`6Q>h6E^C&HhfR8'
    '=LBJDgr|kI+hD!{t$q}cE6&_PhLRgl<*$Yw@KPODxefI;lFaiy7UuVeh{>oEjhjA+K;d0tKAJ*~=9}Gj&kr4h9}wcZiK~smXfhB{kNOpOiR_50IChWT'
    'sDf-3MIZ09_sw5u6Vw}QaO3>;oZO)1a(AN}d^S`-g*~~hIS_dFj2v3vlCQs2XN>EUH6z!zRqDWWt1BAtsug!d2gayh{GoF_-Nf}?xImY&0XM?-crg%E'
    '@3aZ-k6tudRnEc|MLKF!E>oyzCt39u7gy{Pr}^O90ebo!l#82EN1&|5HzvS#_TA=Kip{q##+ZiS#0nJQ=thq-0?5QOB{UJQtkC_Gs~$fna5Q9y9<<Lq'
    ')r<7b=)E}>PcYl8to#Ffhnqo>$+UqKbQ<$o%|n9<&FQV|xwk12{o@Yo(*gg1f<pi({ppW4hu?hN4e6{A+!~~M2^MjEF#&or=c{QN`JFVfire-lNN4Nh'
    'Bar6N?lBSAr+a<3(ldF41?e^)JSO|+aD+uo)LCKLcuci+6L?}e%~~>hz|7b={*Z4M8J@>GjvN>i8_8^RCLbi;xqpYVDR&-?$onH-M<%sifq()eEs1Hs'
    '7JF>Yjro^e?BYnD_)B`k+hyYl%ITJAy65gt^%rc(*Ea<IoQ6j(-Cwy$I0pLY6cJ~2wt2e+`A}6oQZ3j|4h=4471%C=^QZ<y_+N|-)XOsiw^Hc6Lm5uc'
    '9{rhhUk&Vsf0NJHy17uJ{3hSRfLd9{6n7)ooX@EDT4XH(q62pP`dNF-uh!u^wQ_D>_^}+$$k4FY=jttnPVtatfNrBh_zPFXite=G@N#UJ+12&>s?&h_'
    'nc`Vp^tL^|f$6$2<^s64?U|21LjdAdmBf1fDOZaxJ$Q`Io<2s;%6<@q*YD{1@Y$-nJpNg^5(0@ikw+Wruc3eHNpl9(%967-qRcN%=Rvlh>92|v8$g3K'
    '83=pLe&;lXht%_O_zs!)^){w;B5w~8y>B`1JNYWc%BYt#+ou$V266!5%h$CBExNYsC4-ysd@>u$Ts+bs%K2IArhnPa`VDTpU)}|`ZR%2dzsuEZYYa<b'
    'yvSub6{e;@XqS7hA6;z<y;PI)s$MTH7<JhVvPp)JX&{vu%G7fex)OKEyMlvi^{wWwmT^2fI(EVuwQ)BxxyY>lp6qO~1quhMOwbsdI)R1vW`#4a?R12w'
    'J`KmRAaJ=sq3tvDxo5Vmqn^2Af;G+^f;AL)8YU<^XZIP-ROnA&KFj;CHf-X+<!)ataef+e#k@$&@sMk7Fy*;1c=JKE&BodinK?;zLThoo#w|@46z#LB'
    ')boqcSxaql|A|zq-5&#S<rN95=DqO&=IL$kcA`n0*lL%L<W%t(prQB0qC*)Pf_GuUdAr9_8(-w2J+ohfGkE3Z+#9N`#ksZNu^?Kj<39APE<%+)XV{nE'
    'rNdYU{V*tCgT;{jVBhJ|q&Mr}3wu;+R|nc3mRRy@scq9r+Big`wa0X!Kcnqzm6y-8!M^XAZ}@O}v>At2<-(QQPWP_c3X7X}{Pg9Ti{ob8ww{++NprWa'
    'Ip4g;(3g3F-iQ&vE=<DdhdcjK@XeXD-R|pxVKlX0_)ZI6gw5$Nd)2@(wGnHtdHkh1#nnSw=MO7(_X5`KWW#H__1AjCg*~fu0)_ye$4M7_2e(X?hx6ya'
    'xJm<)s2s)dwMxFt;Qbyuz}``2jkUnA;!fe`Tso<hTl!7C@R_scdXl!yy-tWwX4nwjD?6FN_~|m5iQ<HQCm#cP?e|x^fgtyBKz@~^l#lUa!Ki<C-zN?o'
    'jDB#kd8QR(&QXv|-^P705fpLmMXm8~bot!|jM`|o3kNKlM|488v8e^4#aRG1N9nnrUYpz(f&#2U(eK{)xgK|B_q~BFI`Hef(fQ(t;?IKN#H<{?K3kWH'
    '-ksLtWYMfzwHUQOY<e*vTLW$ME}D|jd<PS%vp%?ry&bPQjh{DPwkMltrYhOsApg+8T|DZ=Wb}BmZ$WKudc>wJH9vTH7@Wh2zbnuLzFxK>*H~I#=5!KG'
    'M;$Z274Tc!NiC6U{`5fZo9`bv;5yrd#H{V(-P4{Xy?<(G5B4{s;uKDmdpBxK*urT8OD@~Z+Vv`voc#Jv2g1>OJwJ|r)e>5?o^Qr5TdJW{u3AWSE)F*B'
    'wBD?4V8VbDyOcJ*&bNC*pGlR~_<HLGTe2YqTey}l`szr{2BXs*x<+|>HP!0NvN_lp7EyiJeGck9f6>p`p1Fk`qr`{k4IcWI+9Z`u$8-*q@-;$uUP12t'
    '1Hxt_1-FQ#Gt9Sx^wN!N2o7ZSB}T$0JwWWNRFS1@cptG+tY>8jS6a)MF~-Nryl7#@nOVl#u)n|Pe}KCVkrVqxH~#gmRmVMN!&kMzy!C`SbMGPa5^%#k'
    'DSk;9I~KK$$C!&dryGvVcFLr72PmUWjs53y9x45`wo4@2VQTI5m}GkNeTvmNtxbY#7k|C&tM{_?73LUATcPdc*`JJ-2H7@hyts>Y-EpsRv#FT+DSa4j'
    'eif`+B!)j3vlDlDx%5W4b~~u`N%+Fe^Ep%b?(aTi{_tzpz(0qszqWZ{Si=&s(aPNOxvf{$)T%sjDWJsX$)VffifRT*EX>!>ICqJ*KdZ+KyQfZTuJFLn'
    '9L>fhNz|o#?d~Uigehwy^S9=yfhk|W$pcn<_ak`vZF*@}Ab@J_Qij&mIu1Lz8&1wY`%(Ni%8yfLyD~I3x$oWQGf23{uG%kO13|vQIv7r0TBZF-FXG%B'
    '*XN&?mzWQ0izTQY8e}6MjVsSrW2H8gsz|XNXoW<`yhklh*;Zr5?M`x?BmBd8z0Ga}g;AfjyPf%{@@uT!7xcleHJVxs$8>O>p2@A4q&?s68i!0Q^ZH+^'
    '+&mvA?W^^A55UI#bGUhsJ>M!drdHVn?PloADn{jJfwkubMS+98&&{vcvcFG6_~uV5TFk(!zs`ocaM&*F7usF(_N6u0%HKT4{XqtMub<nT0o?tpZ*E1!'
    'z&jzqGB@acqH=!wD0;gr!4LY)z+a2E#8~*ck^J7<S%d$r#XS6KZRC1vcoBGh7T1F{d+Xt$b>buY@6v--RVG{Gr5tiy7+kg|V>_d)cb@v=B9BICV-~1O'
    '`cE%cpVexP#ZvunC^hl4g<P2=@ez9?SaZ7865dQK8)4d_w{P?tvso^pb=MhIq^&5;_^-c4Qa%1`y4J=m{vPwueU;-`a7wCA*{rP%*sey;{xE@%`V;`g'
    'aB!{qC!VQm|9&U$ig5^y__-wMXQf`Bx4~gFyTpGG-V8W&3WlTCGJNGPQkcAk*ycbcjpcc^bF2a&hMswPokqQ%Q-rVr<@aWvt70@ZIs0vxBkx^sd1}_a'
    'qea_oqS=l3NIfEO#`0mV_4Vzn!v&RW;nBsan9olL9D7%xCDLHsLT&K4dA5msM=)ArW4KUfyU*(BY7_I1J@{Kt?4s4I&`JZoO5Nwh>}{Q%N^QrNj&jG~'
    'C&B!ATf+jHt#fVK`B}?XIL1N=x|^Fu4qxAPySK(q?>)UqDUlw8$M1Twot;ARDMxcRmIs~U!>-K$y4POs8Y~Z3GNSclx4(<0dE`SfWA3KW!#KXi`<65y'
    'E)}~+bXsHCets7HG|eM9*11#&T!84<age!4+WW-|+3LSozk_Z%;yP_u(|a^qqpU?!Bj8sXG67TVD2En>1MoP*9%HC=oy`0da-f#sluHH$L(uS>%!fR<'
    'i$P#`T_xdSY8`*eEiuAA<E&k?q~D|>ZzoH%oeAx$Uf<f+7QD*ZTMD@ypS$O|lrCsGjwIy99(P9YLdSdE;mC3fP2RBHnQ`n2zQ6wT%;PfGo)2R#TBp@2'
    'YTvA^F&&4$=JxQcdww>6D#H&Uyf6UV6$8Sfyr>cP$b5P6u>D`0HP&$GPcoaq%^CJ4&Hnyou!@*o=|Da(bH+O#zL7S}{!r#5{J`zzOiJqX-8*+4J1pOx'
    '|7z>6hOKlxg66k<MSt*%&kBet4Z(-{)p8Hl>cM8H<*0svp>IPx_{{MSsg*Cr<vi&plG%hnpPzv8LE3Lk=$C>844B7x6~mL>w_n|E>J}!N2k*7L0+`a!'
    'g68-(_m;cLdg?Qs=8deUzJXk4j@EjO5&n7AD=v0Q_s8z_GMXrqC_D2Wj)5oSX)$SxU8lZORvztH?;Vq#*@eG;*3ClO6*sy$9#kTYuXTRQGk9)9cBS^y'
    '`}MK?Jl$<YE)Di%AFplv^%E$6KRiLZzyUm{DCFpwbAx7lh0SZd45ej#vaOkYWHp7(jY8EN<y_mqlSN)`orpUaV-IY<2loR|kZ1Esa7PBfI)(Nl^9;J}'
    '@>5MXOv{6|I<I$KP*LDzD)6v@U8W-hDXpyZiuX#Wsj>hITFf_K(<+{FIgjf1`CDPD@}+9U_H9fTc`j^Dd~xREJLcy<T_=>wuDN{<!Cv(Z3iroskD5h9'
    'swMIsLMvDxdt4)RImW*9a>2x*>CUVNdowQv5gt!>?oSTx9az(9`AAKFK9G2e(|BF<()UI0n5t17K(FAzP?$z%;i2|@w4UJ;;520?nAIN8lx~Z3E_c+d'
    '>RTtEb*Xeb5cKt|AMc0o1Fao7KT}rSH2}Zi^gE6_zl$Sr1Gz!lX(CT}bcfu%AD=3{FZGE6V%{5j=E1o%l;96tkBnFFdh#tvp^Gl9>f%sz9N!uZJys0E'
    'hmf3(cdJK~uLm)%Os(rodZdRp3ei?`(gl>$*{bJ2dlyx;P8)bxrfg(AkLRnnS3-Sm+-1&Hw0X-8x2#^|-4ei|NYSy~nk((`*Jb&UhNS3ymdi|STIfJ!'
    'KMDm@#Czh3bY@3~%-*AVkLZS&Ya9{ljl25XYKIe?^rQ_?N&8ct$BIsAmFIwIW+g9nNnts?O`kU(8JUYtu#`;Oy-vm0WO^ugrMQg$-iv5oZ49maA=V$)'
    '-!{O!E7=ic;jpLD`yzVo_V3!?h@Jn`1T%~~n|HFh%dERkgx|`p+hw6Xg!_@(SDEHktf2FAtP=urzNhG_t~5AFzuyj%s0d~T_G|3-len{aK~F}T#6W3Z'
    '`&|8PTqmOOD@^+6Xu_fO2wLb)(sYeJPFKe1@vgmzob}myRdt%#2l;ao=T(5>>bUxD*6Z@I@W@wh`|cH$a_Rr>AXflSX>b}w`SL`H*I`00odpMa&-I44'
    'kdeBve;2pf8ZjE?&xq5F`2B!X(I?!cL$T!`9^*D^tsC<tYc|go@F@1rG-qKi_j}@HFMTr9n>DHtBc+Gm_nXzAqFbvErYT$RZ*256MyH1X<X&<6Sd;lJ'
    '(>Q`~OQ%oi^8pPF!Kpio!vMIg{0L5N<{vHhW*GW##w1`0-JqE|nX1hJ{J<Qg02$kj*#}Oy_UeXhS3zY1j5+`u2E;%&_ICesqniE^X*SV(0$$E*SV?yM'
    'x_HV*IyD>CAp@_vtM`&9@N@T-#bAFOd(dr&?^+kEypq2))&ZE~z<S+;wK6+j)B6Sa%3?xbOM5OrwR~993Fy04tvz3_o<SV#)O$etaCuOQ*}@(Z`%Cg-'
    'RZbr}jdr}Ie^QM1N!V=Jyl&@`Cx+X`{M82Pe!P1<FYSdJ^kvo|)l9fa;}TWT?k3GUzE6CaY#6WJsM_*Z!XT++#Kl?DBeB~Zc0vqKs}t%gW=$x?YS5{&'
    'aR_uApN_yqE!)Er1y41!vcs5!0QeM|zv>b;(IRuuT)kVfD!KBT#uhD?Ec+TZ9Z{RKED>DFxO2NSS#YHuMyYJ4VgG!nygDws_BLmHL+PZu8O~7OY?{XS'
    '^?c)8LhFGE1a4rF;!m2c)nI#m49>qGm_JM5)z_B;bKe^({WttPmB4bo@1`hS-WLaWAB(uqjnA*d_K6|f2Ufto$9zSAbXzNyC687A*=NuFsk@@fj)GM?'
    ')yL_n3`fAKmpqzQzPYKflnE2*4m}bUZIH(eG2E;_>E8%RqO{9D$Si7~Bui_>(?z2j^swd3-Nk9L+`iTxH`2~RdIfe&Bdc(vR=%|S`aLv%S8h5SFi8w!'
    'pZDl#7(DeO<Uwsnnxt@jQ)bT`n);Lw>_&|>7FS4o&+0l|UlMqggfB5ut#E7(blhF{#R=>BinnVu3O$~X9`oF9CkaLfxCmjJ={qXhzFy#s&tIy4oZ*97'
    'wT}amq^g!G^`Zd<C)zTfgV_$htmWn{c{ZYJ)F_S`E{|Meey-l;(uX6^%6kH{cJoi#u90zhCBsWpftlxaHw;whEDuDilKhRtx!KUi-lJuP{TW0*PJ@xE'
    'IGEL(zj0)zobOdy+gApCO@F>O+*+=zYJS{ux2=gYv)zLbkfji(MxDv(O|BlZ+7-#09hS*b#2mMslW@H$hmo(n2Rr?_MtM|ezU^%bemEd-Tjz9jKK=Bm'
    'R`Hse2fG9FRh53F^pTrTx90tHI1_Fat~FS=Vt?C0yFZj35IHr4;rQwA)KiqB+tEp5WohUqAdPOjduKGlx%K9t)dzbxJ1Ft7n<de6sDTsaSsSaF2I7@2'
    'Z%#ATb?S8TDwDf6OxcVjQ6+OmZgtnqtF8yrjgN3DEbDTdEwVSq#V?s>n0WpYMbGjP)N(_{{VtOQQgAc0T;ExAOufx#g-`lSkAN&<C0{#Wb4<)4b-QE}'
    'XBVQ34lWn$aGXALCF=T8b8=1|zGOHosc*(dPwcy!5ieg_b*NDp0rLx0KR2B2&w1Dl-raDK>#xxFA&S5K^H+1v_hZq#!ngN^akI7ozZIz#!mq4u>O{Vc'
    'uYcc*h~qCuu6Dgy^Yv_CNAVBj?51LyHyF=_iW?{Q);+odYHuSGy}^5Y7OEC1jT%WM90x~UBvtjsHGyo2^*gHc5V9BYAme>WuaL_1dJuopFR}mgnn8s('
    '*I<?EsPGeT3JRIp>Uq^#-Q6CNcxR=y0tWi$*KI^m=|NrX3Ym$5pYNO8yPbALH~fRBzcD|~oR~Zt<FEe_NByRO3d#tE8oyCF>IDFqpS6>C2-{JQc)y`v'
    'EeVys9IQtY&XGQ&7cZ{@G>a{H|H=6Mb#f=s(U=O2b=?l?cEl0M=dD@SHY>YU7k&N#_B`$0-bHq#extn~M{^a*-qK~kAcGdeK_1+qs+$m*D1f_`Goqon'
    '<4+njkIXHzq_}6Qw--`QUyK-E@0ad-)37^o!R+w?XdMMq^WyL9(Ia|2GV@xE>*YKKzCJnU>I`#lvA=4DNn3vfPLS>->7-UJ_c++y+>miwmp)y}IG88i'
    'oFH@`pb_hD;-c@^iLLG#Dxnn2W{b=8FkKMx89GnLuUW`{F!(d<3RZPu1MIK)Ql{q`o8O~OZE%9{ft^!JiD{m|vVge7{GKVdUFFjkzU%&}kp!nvJwr8V'
    'P@>`D(=Jr!=Fty7On3l|f$6afk#4xWjp12+`K^@QWu?aXLDsy^o0N^l3px&H^SqX<zi^q^y(PE#+rGE8_*`Jny-$_oMLaJ>y9*OLX*yr%<4;hG@8wnc'
    's7Y7xFVPuZf~XL)=QUpw)X(W4%Fl+kE8LwPfALaSRIr^#?UvK_5E+N}J!8E@t<+8pYyBHJ;>w?;E-;#n=8dNXj;NM`SE5V3hpbHce2oK~5+0wu&#KGf'
    '7L3n-I~^H=#c?!Hc0H(D=9AT^J(9uCw)%3LQ(0I($DS*Dt*SEy*VJi#`OED2JhmxZIWE+Xuh-nuyE^Yv-@BrWG{3LEYDTF(H<N4FIcbXes`PWN7$()r'
    '3b~M)8|H^g7cHA!V_p4_V?;jb$B(nI^C)A3_+An0p18fanRlaD!8yI>`4#OQx=%A*Tyx!lmQ;BPf}(MMJn|F47n3KTU8PG-Xy$B-;j?3(I?m%71n|01'
    'a<#*78-@C2<Ksz6_oGO{=*?S2st{X7IwjRs{5-q;)s!61Z?#BT+y}sM9dyKW(X($u`$cc>=$nWbh^pc}?yVMA1GdF}r2?kw?cRjVt@eecg-{VafmE&H'
    'FsV45KXD|E;6ANch*&`93ip_WET~p9G+t&;VPCbQv;E-R^=Hma9{zEdY>QOho9GC?J;Se|T8~DJi^KRvibV&+!5P1bl2NOAT-A{(8ip^0es4_nG}R)~'
    'zo@k{dIB17#nAoum34A#dU#X&m(-ZuDFLyI>-_rqDkTa#g9ZoM;d{ATvP0)r3*OV-tMl#6m=Bs?mMdVAY^(2HNY>>3%Zu49_sw_JQGH}+?82hNE7L-}'
    '*Y*015JJ0Uz4Xu9IBBL(*qt_@<a>|w_K{7H!(I9|VF+20uiblfxQ<%FN~Xl`URM`f6O4Ls^B|Oy^-r(Wmq!zReJ?P$TU@`bbn$yp3*yLC_U$uC_{?-o'
    '@#0GVXo#(!6dCkPXn{PE)JD2=Xk3x~N|-zG3cR?KkgfKYwhe4&tqa|T5C_}g6H=*z>Z$r@+q+5=pWjU^Y6due2?z;=6n_r<N70!!9}9(X__H)*l4(X|'
    'Yg7m=q*PjXp_D91S&|n1pZ)Q?HAc=k_xat|RpU@B{kJlPucC#syslp7yGRaZizPvj36(+={rC$d@1uFg{iJgLL|36m>3W;r6k+<9LUmG2JKq((ZX;8k'
    'rDyBOjNSiXOP_BgGH(A(zcqkRe{eobv}AX@O~$=q&Cf1OT|?L7Cqnh#!>FL7Y)gU-cizA01@L(q@xD!(6;jz6(j)jN9fG;m+w@K>sd5Gahox)Dy_PgI'
    '+VE&HroZ=rH?6Le_27F5H^~Wju55VG==Jr3ovgD`ed-;lfp9G@XdI>=OS8cDVn2OOI53ia9z6De@A0C%E$MU{Q!py~%s}T1wr$ow{mp!FQqL{8*2b@5'
    'u$v4zqF@p4p~(|i<N_D0nMwGjVFN2_o-8le#d`v$PNO`}FX=CNt|N@%a~R4;5spy_3v`v?$G<Dj`)w;c`PJi2bn*>{Z?3~^Zl{JoyYHT9+K96l^`6>z'
    'J3g=0wO)BZHoI!Q2C01!25El;#lafkV(qCxMPF!y&nvyDHD%(WSnpKN`#URqZI5}>bzA4?Q3eR)$?s8PJzr|a@z@Y+N2V({W58&W7aH$r?o|tq{RNXm'
    'Yu;rFY&Q4IwQ;Rm^_IQe)>!^qk8^0FE$99T{SeLXNolKG=F;D5rXS6Lq6S|~P29Y<=*$xZe_fj}3GMUf)?}HYU>`YKDt0=!?a)1~RvGMc*TT=iSuK%-'
    'M;T56VnuE7&t~iB`&OLL3zI~WtyEpN*l-p?36MX#ONCmrgrg0a5<L!+byOy!xf6`&bT(R<A~<Hwm!>DD(q0$q!08%5z>NkQK$E6JW|+~`mEAU%&lVbm'
    'rWH7FShOqP#6jZTemIJ-O_Ei&hc3R}h|-pOQ)NZ`z<V+#_u+gzilW!JP@hdc?9Xc~dLUcF%>w2x{eAyh+=$z2w{UA*bg$=w?6tOf<X<e>Mu!+{95}q('
    'drec_FL&z)H{}Df;sdTb?jzf8uak63CvIP&#El52ed^6nB%GAVV%M6ItQ+Yt+|J@Za`o!;p2w!$71CYa!s`9%;^k1jwCsTUi4D}3r0mBPqBdYQ4T>-F'
    '>s0DIi~QTVEazjt9U)yDV&v(|d;&spwO_q&FKx*ytyl<ys0wwza;&hm^|Q#h5K9iTg$lD}gZ!(iCGy3=Yvi8b7SePrVhgq(@>)f%%QsQK4$7~o9&xve'
    's*7NL*m=eT>>*Z0r`hvuMZ-I`tqr)gQrn;4`to_`>`6C1-WH>9-`O>HOqlN`UPiRvTmV)&!?SoN9o~z#SBoCc!{Bn9dLOD7UC4R;n-12>YP%{;<mWsy'
    'L2TFQIsN9l-u$!S<^!R3IYzTeNo^0|Ip$X?|JYfK)ULPR)`w<_>~9A`*{^iexeThL8FAm`+Y`ODTR&09Yi`bWsM8-~cg{^CKx-PtnZ8~Q*h905*8-=1'
    'mNwW2jT*R>E3Z7>YH#3m9E`-cFVy+jD{pHtHTg8fgV8?H1F#$bMs)GB%xr(55%YKC9b1!;`>15Y#cI@;8vg!%)b_>E)mo4>&=-Gg_We<ZGieFK5AGfG'
    'E}M71Oqg?Q?^>!{v@hhWQgh83jC4kHvoV|f<vvhUlBQc2gRFWtE=EHy+>srrg8uZ~J@>qzU#T+ysb|-+P}=@8p~ou$>O$*sG9C+q1@U*iWN-7P3(t;T'
    '`;}~NM!vNAIWL3S<2$3rHZ_IB?ekdESaZBB;D;toN;GMIHYnC?_}}khZYN(QGFQ2f2oW90Oy}iSP_(ZSe9Iy04V9wntsajY@zyy|1tbBlCnC72f;t8I'
    'VR>Q~5f&=Azl4*X2@q~)L<%gh&IUF9#GbFcT}PC{2_C0y7NZ{b774Q{4l4JD5I%7RHfL3^l)%S#Uem@BR=WnjcG^Y?R3U}~`?=xkI#IuX=I?usWj>rZ'
    'Ya)RLnow*ugv(LZ%lpnvamo&Z6gk_*QC#W_*8ng439J%&%0s{HWzkV<AI>^uu+YPSJCwFgH@p)oWcCLx+v77MeiB*sms=#q71C`sc5h~=^6;?c-G|Sr'
    'H|jd!vs0y|Dss5c*&^ry)C%D{wQTlrNOC<v{iFGMu=t^@$$!Eg?2HiNLZHCwwCRnvr0QVVZq<%P>u{kp^l#IgZU&>uG(V2D9znr7mb$SE8K2DP5&Ch7'
    'woCR|!s-?;fjV(^ABo#!QLn{Ac9pih<>q5G6}jzPq%K~&F=ey%dI$v5sysKe+jv6F`)${xkAsP!Y+lL8)^A4V4!}RUkd0Gkjr;Jf(R!iF!Nay1QGdQL'
    'Z9kahGzKzpx_!=LxV5c9qH^yB#&kM<Lua*jY;R}o37d5Z(8W)^YS&D+bmR1FiF7{A1^#GRCCOyO_ISIl={RN1si1N?yO&RdvO|vOgs4|HVKNcWjfP!@'
    '_X^-%L9_=6SUKwy=xA`Q8CUAtTAMuI@JqOB?JM|4CaZk&N|RepBU+W?qvtj*m#{x$+lT(O+D>Q4&LeB}&l$6u3=~=AC=4f^M?drD1LihIepm_}ZbHnp'
    '=Q$-VHZFoT?wtuJ+u$m)8-pVEp+yUW(EQ!a;b<`#x!AGI#Op1%{zJl%`-t(yE$2uuUp@*`GzGhK$V>G(2RiYr^CvauAu0}X2$(TDCx@yAo1ZJ~%PzTW'
    'pqy9-r2YRFJk{FK%GgJVxn0_!R?P?~hg$~dH5cXgj^;OXH(I)3@3o4COj%~Ev5V~_YC4T=ueo+V{<S%}d6CrHM-6Iq-m_`I^0rOJio9;a`_BT%ok{?1'
    's8WAN{f*P<w#Z$+uVU!-qeR#49*YN5p8bv4$zvl{=Kkb@{*7ax79X<(+kb_d{t(+KdAstF^3`AeU2BUX>28Kt<t|q{%WW4<KqG?<*XJg}5!KCXLRoFJ'
    'w<5S;+5?Hf9)7vduof1RO1pd`knszB?%ZC3e{?fG4dD#f-BkFr$tG?68ut1AyqCT(scHdNZtemvAR6cH^Z`w+?zQ`SYsU_0IO!0zU(?con>Nf&L+3@@'
    'tmk917Eje@GB45bg}U#wjy(_;&BK?w?7HK{16A~Tw`i=Hqc~>#rt6ur>2+FJ2+CWJf=b@Hc}>ON?~>o*e(svXAEgfQYpyEb|FtSzsy5wyZh!EeZ+s6o'
    'BI^aqQOoR7U3B(1yt!3WYU)L4H*d#1QFZ-O;I@xxb&!uQ-%f7U>-=WkX9EuCI@Rs__Tn0e`ErM?n6GC=Ub?BS!k^=J*V^jz3G9%CZDOmS(~GbSmO$x_'
    '{ec-<oG)58`vo|KDSz>0@Z9u#ZtV|8`5Cp_Z4vr^7prcPQ*`bz=XapoiH&?lXm=0jtL%i~lRpcZ#_8&2M>#t{(<_Co`DWRUD{*+5#T!28WcjPpBAYb+'
    'M~ac&3F6&RU`p_?>`h1_A)9<}dJ!*+2w60ce#_xkJ5iq~@5#E9O<h@rJvCwmLHmcLjs3t2vRiPuPM!@Z7eMLR6pIwhUh%`vp^9?hmI~JURlhz`$P_zD'
    'NNBF=ZDBR&OW0+$^#M}j6eoX~G_F|$lm%D)`8-*5u!8X2v~oTQ=Hz;AmWzW^iwDhQoO%q-trySJNWKOiPJ24leK>gTvYqo8yq`O2<-GN(E|w^6))<vs'
    'si4e(w)7Vap22!Y5a4q4x3NvR*Qkk@QGtKfc?*@x%vsA*B}GFy$-7fyqfqf)yFKbyaT|JmJ*rVX4fWe|o?Uc~otUmy53g)CNzYNW&)-9BqvcbxymSu6'
    'ZM0uE#!{Fb$YNpMihII+*fw>R);(Za#57f#G<K0JKK$u@4n?(R`9uHip96L1TK!)3sk+z@75whe5k3r?&&@LK3C#B@c`)I&cKxR^HY@zlo(nMPz9QrO'
    's1DQxR0R=>04j+LfBB^0f#ET_Ngcl{w%5D7SzZ<Lema5AUefCo$|+{5)x9vBOv=eDZhhT{7aS^r77d9Dqb_>x63Gtr`=A<+s}|SW-z>pcEudMju<@vv'
    'ZSf&@tFMkJSaE8ZADITG&&}0j;Ay<yiaz2YJXV^5<<PIc%J8u>u+x?0E@|AcPc>r~e&CsMXn1`Z9daR5P2Eu*9HY#OgB^XPaqK*Dbau^*=9xx-+zYqX'
    'pkAYwk&ynfpi&M*$knn(IUW5`ki;K%38(TnbDh*#a@N_l-T9N3{PvE+Fy_T-ErC6yMOJ#T#^Bwu+<<-wh|)41{3zKt;>*G}TzC1SOBR@e#%l!p0V2B`'
    '{#x-#3qIG@YbE`*mvfyvyXxx`ljcE~hJzKaE!JgkAa!cVv1Q|#Ghqcv`a77Y{OZ(3;@_s3VqL>nZ6~#2q!;f{R2snuGgQGa<GHh_P3E{>Ylx2ZGnMK9'
    '7(;C16pcc_SM3(1Dg8b2l$Vd~?zq@m6jfZ0=3=5dx~zJi=E9Z_^H$#fT(;}2pS$N*Wzu-B?CN(SB*&Tu&8n7t)wfl%K##`6IyIca*6cYIEN(`{&C!;c'
    'p17$V0N25HO><+#RvQD^>B#=OUHOSUKjF9XaERS5=aKr8j)!D?!{lAtS)5Q1CN3R=P~x}tU0gyp?<?Bx(X%3pYwmL5ak;u!HLa(WR7g+O3T6{JN2KRf'
    'ZZ++pk_72g<a=v!zE!{<9$=#{$5rD~Z(3NoGWh@WMxEua?z#1=+r#BT3nv5UVq|sD*9bg%As*c7FDlen&1Vc&oxFAo^0W70fjZ1#D1tFLy*)dN-h~fD'
    'b_}<de_~-iJ=ED5^-(q^I{W~#)!OsO0d;SyjH-THP4Cp8J&%bYpZrAkTc>>1e5?Cx+LFQNE!{jm>>yn&u+9UiK7+;Kss2$-vc<bE@tU^>p$)qFLuSp0'
    'pMF0xYt^mXCZVrMLmo`5$EbS)V%MP=)^SfLYLV8vQS4in7xbX*D#uq_2ct2v?(hA6Rn&-Wb-wvpe6hdp{Po1TQh&)K7l%s+)*<>2vz!+xLq@umm%yR@'
    '<m;F-y_c0*zbmxcj!%v|yCBC#bD%*Q%_cFaEtwAiCM#n(YAZ!$+vrc*U8lg4_u0h@#SZ%|6zW>aYJj%psWy3-BTAV3WrlyLB0~1qCf4ru2{4H2HF$V^'
    'y>G_@1$qU2zopclDQ2lKC|?J^P`@J%N%@2FF2yYD;c0&)g8dF6nId^m8#m(B>Y1R~)~AcK*}GcdOiUGfg86eY{iBT3oQwF=V6yqRtWTiwG-ul#>+;qn'
    '%}T_yw;%_NH^qK<@1A=;oKKqMSsnD}$XfbaY-)c@b#MwRqaG#e92bk?^YyOjl_$OZYqA$4P(<|C<!j$@7pN_=lYh8s*V~X^)0<PHIkM)jGa#Ce9o5e9'
    'Z%Xgcj+8}QzBqD=S%q63G8Has{Y8YDQ5&H56D-{SefBPO;Vswg_unsI*&(_ELGsR%qVX87Ar2oc+yngRuE(Zyo2Qger#JaIJj<=;rwXnP;e2f%#2#vb'
    'i(Z0qlg}A;hfUl4nFKh;_k+LS|8Z-bF4CPP=UV`3WA<@t-$aqP^V><b09A*>(i-@*^i%b=UbVPaGHwS~3&1{fa)K#g#ZCrcdYQWSSEC>^^>R;(KWOz`'
    '^SGw<&&P@rTc=%o1kcuf-)XK-{wyWFZ?{Km$f|e`y2sWVTAdAmN4eVf*eCFZ75S|Pyq4h40x~Z9&BdRfWlbgEeJ0>ds<+XGjrO#)S<L%GbIS064c+!!'
    'x~j&-nmHY0V^TEnDPP(49rONq-Q#sPMmX`q=geQlkbCKO^f&|B<l99yh}>_=TzA^E``Io4d}l*Ze(=R0Mqk4FMZ-93vEpU2y`I*;Ogh5$`xrJvrl(N3'
    '-do#5JIh8r?(v?MV?f8EZ}S+d8Fc*t-@eO<mqb5cljsD<gXinMbq6?L<fHWu5?zs{t4~CEFw@zq5iPhFYpQP;UGiM2S2l+Vpqjf&K>6{Vb>=m^;w{56'
    'd*i1-9$W?}a5Tm@v_$pwcESVMSo7K~0L4)60Po45Xz%(C^_uo8v_ZPPNu4THeC4W-m+}N#9dismO>F)UhR)HvRW|sAUksZ1m|tG|>5iacc%*c%m%+8g'
    '3T(RW=<|2|QB!CiYsX$_5AikaGU;SzWOj%9d%g$nQKo*j-<l8{Xz%dU3p-s}E1{nxBSbnpvmM2)sQuzaj%HBubaH&St7_2W2VCr1Z=K>ZDb1V{d&gAf'
    'y6Z9Av0GCY@awtFB@Fi;dV|B`sj1cFp!N;JirabZlle+G9TlJ(mA>C<wx4C+u8bF~PJW-T9W6QaBI#P>-3MKZqF4LP;HZZJ%iJrS`Y^+>(AjXE*{cJ?'
    '^Kt0+0g1v7<^ipd+<vtaz=~H)(PsG8&dr}i*LH%7KS0mEWO@ZTh8-bu#%$dyQ*;QurC7d`jS)WfpTUW2GAr6X3Oj(8xzl~zzdkmh;GNdOW^-;ywsKqE'
    ')MslB>My}P*tgN!%Av>@v4y8`)elgd=n@U82(o{6{m_iqKVtaTG<H<XAkghpe?L#Px_iSq@=lA+8h9VGCMoSYWVhb#^(SxffXZDW-PLIQP+0GXItTs9'
    'LUP|fBiwGl_OhuDArlV@xvyN$k=hk`Xta6+7wg>UUOtev>fYbX>#oH8d%v!yv|7ym51ic)<;Nw_RWbmo3;ye9+=alQ0Dy>*2$7w~yO5t*1+1?O!VsCK'
    'HbRwVRCMH4arz9NPwIMhS=M5f!|i|=eLA6;z<kXXX(Kz+)t@t_Kw^(a6*Zbg$0F}smtU2FYj`{&fME1@331v`wXnMho)>}K$mtLb34FY+=fZ8fA_h{h'
    '*pKadwA7BuA)Ib__>Av|YvdlSFIex@^DPe@&;BOt_B_Ir*X^;)ePz(8)n%fnu(!q;c&c$DhRabrV-tUK^{i&z7K#PVRJAXS*@n_#D+x5X`fYGDy@3SG'
    'e#RB}Ek(Z#bgiI@<TLaLFP?5Rf8lGRzrB-&@UE_8X;Vjus5#k?D@9x1{SSSppPm_Bb?Ff6{Yj4}H_Rzism^DqX&r|Z^tgHWKbDdrO{1&b2|DbzsBPF('
    'Ht}y*43#T?1jEQZ<F6<4db9eRkG?2<{Y>9Q&^ID_%jUj4C+pq8k$bJl<}PxZg;hD@zbvu0_Z?=Rr@Iu(TTPAWW&ee`2@m1-g%xB5-RQlUJqj{xkPa_Y'
    'aJYo4tyQQ)Q*O_r5c<&lG0HTMJf9zT=W%qPr(Q=_LyR&Ez<PoE#Wk!CMS?#r%GNqjSb5j#{M__!0NSOvyBiJ#552*nT!uP}T!w?3y@%<Iic@Nzx4wLn'
    'eF$b_u?h0n4oB@;+UDz;e$CTYSA3HT)n|%6zK%9k+Pxeb;!Jb-P#O&};nBI4`o%L=nKGFk{4vI;+rNq0+mVH0)$jG=udG7g3Dw|T_(bL*hFf3M8~?7='
    'X2y+Sb<^0|%gCEgn(U~9<JZY{Ny`uF8feHXgZul&rrr!X%EN@V>Syu+*1|xDU9?S2`Kj!R{8Fv2$8g2&n9YL`wD=QTZmTT$RHaETmW%-8xaqDt9VHJz'
    'TZCiR)|)+2*zBcCHT%l7KcN#a*M~S%kO}-#pXaE+iyl3(1Gn~z>wHwX+rH!P(*1)rUApm@H_y?IT;L6@1+7M}WcOG;raQ@{J$rO9ju``7nc4LZF|=$l'
    'FM?kXc9=ixmOL5~I-u#%B^(l5s10xY?|RG12FTDvA+YD^)f1HovX{iPuYBOgrO>JDb`7h`LEP%z<m(D1=Fb*Jy6I%GQZ62?jF*j*il}N-qScaGj!?{='
    'OqVmP%sfxKIw}8s%~KL~sdjjg4={Dj&glF3+QgOb$nC;q1}IInHxD<<J7SN#q3%Lu->LqvAf}q-+F%ybZTA)VGux|()rPJ;{<ddl3~bOzIjA;X!|?7U'
    'brllyJ~YPu{7F#R0)Vl)q?2wxr(tA&_8;gAe(z!*z@l2@%Ka(!n7a?DhMb0wG^zGOjT)Xu`}$gC?>x*>((r}NlkfI1_160PQ*`@wZ*hXg9B@#V%T!ek'
    'bnAPUOmsZP-s<vAHx0h0KPNSE`geY=JLBzN)Lmux=cL2k-?|^9lPpWE!sKaga-KHF+?yauE9p%oo|bE_PBf|ccfnlB)GBIXjaOTf$=IH>IjvSG&aLtH'
    '?mN75JW5GEJ-mpcc}>799Bu7QR~@=uJokWBdMKUp8SQ)B_H1_;v^SoW4#+w){|P{U`IM;_fgd&>yHbm5Jr-V~ziiC%C;7Kr#ZRtW2Rr7$Im>$!x;lT-'
    '5w;aJgGNHjeZ5R)GT#Bn!=Dnh3-0m>|JRqACI^j(C5i9nUNf9+YBq$0LYX$Ok$f2M$Mr+as08ET`&=Jj>rHs#ICIgjn(q+K()uPUf%>&hKLTofPo>_A'
    's|p>)Mx0$H=SQsuobp4bJq23(%6xEV;Wy8#Kj{Nx?|~R4z*73UWcH)yOQBxjW4S#?_;~i%)(EWM3xV_XQwS;eEU`?hf=gWdk-%E)B7i`A1s6etm{?#+'
    'zU^3K6wZP`a`c61vWIQ8QU6@tVl}~1%1smzc@zZ<cgJK-9{}MlFk>SU*Zl*{cZ|9_)L#|(*0WFb^8a7X?@jjH8dT+uRH^K*PCy&W`uJ>$0C)eK;P?8+'
    'elQ#!c2?uqqaGH}#B%BCzEJ8Cwih3a0Rg|Yp+}(azMgnDxI8wabD2+d`lumrx$aA011(y!>$c6g#}Brz)OO_Lx-o<K(=yE_0-A=fH@)C7#V=RC@twfS'
    '$DTan6S3Jk7bnl#*sr1k^~md^bzKTrcErCW)&gE*nnyC~SeH)wu}5E^_ugPn`%EJ?Dy#<h`pmUQw=Ygkh1`Bp<MuEdr}h<hT`?1Ca-`fihCmyXIdAN}'
    'ngv|m@^ZD7x66>NW#u4+X<Bl=?LMM;+TqN89vH{F0+)*t+qgsM^E;h!S!|EpcYFO_-zlP9N0V0Taj6?qXPu9q1^j?F*Y_5ewBf;jej|@D^r3wvDj9L~'
    'NM<ImTMT+sUT*}rZ*sIAUo3SC3%A#U+cv4sr!DbNTuVRo;3QN?C7Kb!=opXZJh0Te=EpmYg~f$LajJlPe{ctDhs|1A@$mKsf$~*XH2|VeYH3USVF=*n'
    '!3p&^oexE)MX&!Ll373dXp~PsZo2E++oLL_>gv>sJ5r8vmeZQmOIUyjT{*jzQtAKpZH<0wzs}TcTTj#N<<B`Gp7HB^7muR1cMFEK_oD?|SqC=>&d;WQ'
    '+goO=Ys0&}7{#MfISGwjEIny`RQJ8-2f8pvIHAv(v>Ezn2tJm>QBB<G*;j&(v86q#MTEX#ZcMyoa*39e`f}Q%B424b11xZpBJAqHr-C!MalEsULyhJc'
    '@ZbVyNs97mr(r7`>5KYa0yJX;sfycy3AnKk;gNt{t{o!q*QN5P3e^=d2B4#QFYQQ^%b|1__Cpw09%e0k3l8T9dUP7hp*MYNANBuXe2$fk9>3iWTS9qX'
    'Nk=T^;?r&Xn>L{zUTf2#67-I_ZFECxUtbJY`~E7ALquJdz&&)&Q8W2-(?w@Di$ARe9Y5bN8@7>{uvzoRPh}&&LKPZWKhIq~ss_@r(+YYWaqB%NiCE@w'
    '-K6R7eTew7D(Y;LDD5iH&gsQvRhRn@I3=}CU3hu32swc!0=QrWl{*7UdT+<d;QhSMKj6q8|L7uZIGza}C4VWJ9r*;!9`n&E9b$*`aona4$@spKPwmSa'
    'hiKHNV$yS_eBTFWI@A2t>eoRL7kj0};ce>`=Nj8@ccz=zr1pEX^c*&d#Z7jP@4Fp@6`aZt*U3CxzUw|FxJG3ebj@-mskKZ~3anad0)uLHyM4#R>llvq'
    'TANq1$x;BuP45IO*F;ONE0dE^8J}j28cD4?>|$~KSt!s|nt_OiPzQj{`Lwfs344$eyTbEtsMnD&guuHn>hLUfNUG_}inQ$bdR1IBM|xKTg3?e-;6K5Z'
    '*zs~{4O${*K8M3-xT8LL)t<;)be8Fa7iPCxwVmthUJ&$zs!_f^>oHkJu4FB;^%X9t8D68S=6QHpQ$2U%uB?tnJD9Ecl#~)u^KzBe3lCibPPH8;D|OLK'
    'Y?Bx&XZzHAw4znP49I8<i1_JjSM@{dDMvOWthtLz&Qo~-&98?X7wBo+pd51es+m?QM1Os4p<d6v446MvnqPMUP>DW5@wU2szrxcCccQlsVt4t-+f8=n'
    'xFT}i_%q|%yoa{c2lhSp+AjlTgY}qIbFI}@OLT0ug64P})qDJUhB@Mpfy60tM+<9vn_H*ZF+nDqeSO$tn8MxkkghvzN5<cw_VjPK(H@wa{6f-q@j(W{'
    '&Pd)M@V?F(=Rp%zcLWJ0#Y1g@*X*kG6%#TPCSYdm?Q`E)O|$M+v{TPLZ$3mTOWF%PRRId9Y$I~Q!56GB_ebCJntZM>`mb|d19aD?-gK+y=wEEo?$7!e'
    'QV97NxAR`Mt=AucCs}i=>4*JW2=4ND;2g$E{S5A+-(v40<-6&-Vj6Dl)2MO!synrv>Oc`V;KW_bhgUIP<MiQ5JWl1gpY>7i$p0c^H&VdErv{>=@3}9d'
    'HVk$Xvr(z{QGv}g=<1UM2@JIFF(zf+jKt~<9~?m$JAPZ8iYd2l^*1@TW<<Y9skI*~coA|7mepDRl<V-{`=(ZJJ$RaQ$RW)1S5v6fSJzn8kp~$sznoP~'
    'RvVW4PPKbgbzJ!f3#+cPf`+-j)d75T-%Xj8I>0BIwXat24EcZJ46GJo{@Gu9A8d)=>o17FFHL0?>wg<X0M3&2)aJ4Oz33HFuUX?KGfXESxG6kCNw&pd'
    'n!@uYie9RdP5_K|6G5nD?QZ7=X5T!XZMZZ9O}DsWm)Cu2$Ka%v$FIe>RNJZ}dEGHH-tJdA*T$1U5_YA}v1gv+>i(b>EDCss90H^78_YOo&^m8A=7}7f'
    '9LxG<jiUiCA-;upYS2R?4Pm%fkgqBx^2VsvkM;%7WsR-5>#Wv^FIbFGR0WI3!#Qi8PV?d!RvH|Co>#V5m>p4np=Ks@_W`{4ebh^AZ`yXxo%K1Dtfap<'
    's8?_E#<l*pKu_OMSWmjmCv?boHUBQ-<sfu#e4Xn~C&k~F1eM-ievMxd@ICTVztMSch&wwC9IFoLNM>9SfyC6e(GJ16n*TpJWX4bE9?t#>RAwRb;kIwc'
    '$)``%#Vd-ZIrb9P#5UG;_M4qp0XK5%;Qkl=pKVOH=H*+D2_Y>op*07DXJ?9b`gKI|Uk7Pc#3X9en$lqmNv`Y?F5*bs`T?FY0m>?F0L~x3=9blJ;A2%6'
    '7D}tSJk4{@r`G1;K8vltNwZj}O`+T1ll*+&rk>%r%Dk0*>g8+t#&yM$ZlG}g!N_0Hd7#*FFQb+BsK~<J=TL_4%ir;6K(*HK+^l>g(N1(5)^wJg9He|d'
    'S}`wwc`$R#_)hOH=k|ovA1<r+;tQfN?j8NA_^={af6<e*h@m;#KYU1*8KO>Y8L`@vUtDMlxt(L;ba}u3$h&Q}31@Z&9PFj;z4~6)bz<$ToyWG*+;)R@'
    'Aq01>d<=-rxV_@etiSWtuWEa<%kU^mw&@%?XjPFYDD>(%i-{rDw{XS{RVKHQS2LYFboKBVol<hz_~1>kkul(|D`)TMby#v0<K{vMp})1`tEoKaMGxyX'
    'wyXWonU-=vw8~?xnRIUJY;;A!>SCWxBz@Z52`?XLp@(#o-nGA!?}N$VF5dRWq0K*PwMwuRHt!aCiSp%rMx`<cYgP)6frQfy{BR`uNN@6&NXkHty8GAa'
    '(r>Z#7qJ=$$xoP`X*~Ah$>{Bi)&8q&o2MCO<BHW;fd!XM5k$nDTp!-6MGdSrIEf8!$1fLM7uqnzOwo;`_!b<o&bFf}3x2WItiWxewPWb2tFIX}8+?VF'
    ';yi`y4xF_ppSkqd{h*vK()Ru9j!BnO9ZcxHCXFEeG6>-K)4HD5@vHEh6bmQ3%KP&p#<G^hdM&gQ%(M(a>=W_%7Nlez>Cw+x20$(Di!)#T|0t_6>jL{z'
    'Z(%%h^SP}0z^LP=9pjMCQoD4^2<fVc68Cg}JF6nuZIHjPEV8L7Mct6cm)$Lrvoc14<J~U#?umG=`yb49`>Z!Gqkc3wOs@>~B~Cx}-cCaBO-M*52+ZgH'
    'g<sow-8;)n61vG1t#N~)S3l>YyxvygZjU6k5q^10f^aBE-3Aa6@pi>x4bpw>cV%xl*6Y8@Q|6mP9dnzfhHcFcITw3CyXaym>dpqM?WMz8xsBEob1HUa'
    'S$NGi{f$|zr`)s-qDUV;u#Y9Ufp0C63Po+BuuamzRr~pX5CdMRmr*%+*K`_gC8DK)_pW}3BjYyQgU0978tm=a6&J2F%=Tx877}Mt%?h%<R_|DiQL?NW'
    '+E3Ei=GJ!Pg}xKLiKfGXID*XRd)xhauwCUF?K3*P)cubiP~TCDughX`pkJfr&EnsORpAEB<D(JirZd1X?M~kYoytnlFXxMx;kq?N0`KTd1mCxI7c`A0'
    '@~`oT3RwG-iJM-MSuY~I$8PwURyXHFVaQg{;mlu6ER7$@giX&X(+GenR{vcRSIc!_grd}uVxnkX{e-9U1A&aTd6*wzO7|-96xLfa3f+z$@Q~BrBP~CV'
    '2-!JA$o-w$RQ1;4kQx^5vpTh1^S!e>z~01hnfKuNGH1nO=-qut1iVt8qC#Euc&M)Qkj@k|qh}?Ypu5Fsrj6Sv*}<9F<b2~3`ALc=k%_*Nf6bfEL*+>3'
    '-rI%$k_(s+0`eDU!`Y<Uw<MmMLe-upXUJc)(GGMk;)f?;^Orx;HN<(;m1zVU%9vkgzhU7_<77BBTDot_1T2?B$T`a5)5B(WXd51uQBUsFl{426_wcPi'
    '273QmTHEe;Yt>liG-2O%dQ3XVtuy7uTUw}szBH2Tx4*i^)@GqQ%;Rx-%ooh2YyH1}I-Y}#*N{!3S0#NXF9`n}R|8x0k0-9(`pVhoG&r`JD6p9q9l$V}'
    'JR=Reb7iz`cO5;MNYGD(6+Ua7dwgF25PymFjeg~PoKInIz*sv%^4_EDdW>4kHSEx)<Me97!5JktbJT9<TjrC{K-06Cr*@bho<Hvz5;wUmg6qfJUZ5qh'
    'U@k$y80*Pa$x+&~UA^0{WZOIKlX+Vzj0LM|mf;IS@c6{Wf7bHi8D6;4MsUpJZsa<B;~-vaX!@uLDV5j4XQQB{Y=H2q7$F~Ig+kYm4n)xI?~QJ?x;1<p'
    'p8zcAu1!{IxZhRan@l{ejKEs@WDY-O1j)}X=Nb{c=RwlY2lz3n%%VDSQ!5_RKiL<d#~{id_}%`c8BS5n<H;R*8=_R0;<oK3o14$?rGho-{qcpnkF;}u'
    'AB4y5>rO^u&ia=-GV*X{bqE>7T6%n7$53hjF4FCE?w-+_O@U`Q!{jbXIcPPgbkn*z7dj~q2is;Kti_~%{>@Atw4)lkm&q~VYdvY9fZF~8JdNr%q$ax^'
    'H!}^VAtCAXP)XcJTM3$v&i2_X1M-30h{l{5%|)+s88{7r6EB=$R2V<lCEU0<)vZ}&;4!n!V^y*f_+$n;))!9RYutvH+21g}%jC4ZpD#hHP(5w8pVMDy'
    'P_gC-E}cvFZA{+TY_vJ-U}QM$5{nvqjoK2CGMzWs7%hJ`*}9!(1<UG+c#i9J-Xez3{n<H9Tj=V=<PgpAr|pS_P-Tt*&*y<Pdxu*2TIE*KeUo!_h+J@!'
    'svcBtv}KUWcTyFz%AohYF~nsY4S}*OY*OuD4870r4x(*3gw=bZv^KC#*)C53uz?@9DRaMWzc{<MiyxaGytEs>F{|pZmmd3>u6I!PHh3T#pOlAPEAkXa'
    '06OpXbchq#Ih;;!`*R<Bn=@F$L-MP@z5CXy$?o4UXiW@7EQj4JO<KgGw`U$4;Q!6NzdNFN*1^1gda@?=u9FP{-`8%P?0ypabuhrT;WzJ1cjbb={q@7='
    'w@u>|>T-1(@(w@XeVv5Y>f2152KxL$yUCE-fC<3WlFjVZV%tt1m{WJo_nz-=(t9sfo6Tq9-pAF6=F4B!T+gU;gTGPxtO@s9Z5l5-l>$R+UTxz@MjzIi'
    '(@Duve~_eKo1-3E)An)wa9J^@=CJ1tnQk^Gzop%FM5i(7PGMV{!M3bDTW1wnbc?M5WUp3j>%EUd;{e^3K!glwQ_SkVv5fk^zFO5QTr1_fcxQV#3pX)u'
    'JD!>Qmi`GUoZ5w%!h)OevU(^<Ia!<AOY-Z*4zf}WeY4$})8x1o%A{xYF%D3=a$p<F)p}4g*LhX@bk7;w_1xw{Yk2XaG6%b(v+bQ)rrhX$7#)>pDEz~='
    'q03de=$(Q-UOVn}nCb2Eoa|1l?{6o#Gk~jfRB!#Na@ZfPg0vy78@=7u47yZfL!BA1HZP>-uugFEHt-IM@2=AV7Z~1jZ(Lnfh*Q873m{vyx9?vG)<)-W'
    'ONT(ht{2mjyl=e3+MuG*lS-){g?V{H2(EXnPzRe+%ZAp<t!(W&mw9DStqJcjD5Y@safxNE)1#H7H>{r^?kn%>w0%$e`%~2*D-xyF?LFUFHw|YgRVoMu'
    '{V@Al`x7*!GR*pQ-F)2BGmEt9kNOePQ%pHBCrC*DkoNK(S)bIGCu);Q#PT+~pX4-iXH(z#Sg5=ws&%hAYY&l}BrJDLOok(gnNBqqWm}cYkkARU=#2jE'
    'e@%Yc>757EpYWL;J9o#Jp-y`ast>>Gvj`biE<jYWtzslUBpWJ<_hGghxAtfDON@lSxOUKj!4|nHUKfxXmD}{Bj=cwaZLH^dLRizJJ|Gpmi7kN+_tD@h'
    'jL)-tw+-WyKl^GR*JaZ6_O$aVYJtN*Fz+*qHf<z<MnWlP_p7h25!Qx%&a8R8;ej|G?VU$QfH`G`We{4#an+c)a_stMo}mdQ)M}o(J}-6zR;`W`4{|4*'
    'JK4ex0Q|N0a+W+J3P#AMHs~@=^%S?fzo+qok8uGWR@*~G=IZqVo6pzuYNrsDpU(GirnY+JPpJ}*VB`b48{v$U4EBMVtBZ4@?X<pA$?^5dllz+WSJ=D1'
    '{_%O65}=X@0e`Q)PB$eSMYA!;&iZqteVbIfGjBh<?8lQRTX@Pw9oZ1fICQd-*{SR&biDq(a+i7eDX+lBY`XC?Y1ya<nzT7JUe6nHgJz*IB@02siy>q!'
    '0w@ex<4+sDVH)KoX3KH9FqhmW@VHY>?A0MZzpX65^E>%8vgK5KoxU%0mQR9JOFXZp7-JIojoO*=iPQay{OP)*F;82U#`#rwOJwTu{iQfx@ZCL!FC_Hv'
    '7$v6rS&aU`e(!y%09X&T-|A^~IqR((=pNu(D8Bp27{O@exDrkm`}<g%huc<4#X$;W_l-oh*SAh`UIJgu?B|?<-LYMLvUJ}Ou_FdS?>zUY(sZH<?}%#|'
    '8~ia=b{IXnR%1*@>s}YY9>;5M*GyosrBU&@YPJ0ZaLHOKP?gVrgD$rsA^q4o0Ocj@fyeT*tu)f2A9rN)Z_qpid%MM(&IcWeT5m*7!9*31Rb$$~PaTgZ'
    '7PPHhYMg&VL?$JH_v*Wue&d#X9Oc8&-ePVa7Od{-8Tnlt<U~Fm<11-|<+!nnwc9icKmy(zUXry?9p$eGGiz6Esjbptu6;#ok)>DBgIo1g4zhP2ySxnq'
    'Ze7{NW^_BPb{+s7{MoC1&<vfyp(nD^$8GGnL~P)LpSlWr^YFpQg>%;{4)9^u5Pn2Acmq554B(jzRms{-&ExN^Y%MC2IgUv4<V4hmzCPrnsu`(OFp;Z^'
    'TYXnEsMfw`&~^~tsOjjIZF)3)`ZbLXZ{*o&b?v=A6L!p$(&k>NFA4sTqWgs??dh%&d6)jMH0SE+r|bLk)0hxo)!rlMeGsq4lX+)-c-1O>@2QxzShhsp'
    '5WgC)JfGjoZvtfwYk92yJ+jcjC>uMxHQioJ*g0D(wao}X+fa3)`arLI?ePuCY!2^>U;mpymv^;!-1i_1w3Zzp8!nQ2w>IBZLfZN+ErZ-rua$;MAJOAx'
    '%XFK((v(7vlmYbTHC<-I2J<k}(&{KT@t5o^Pn1BEqKSe{KT+upw~)AckoHw=c0N4=D%Jyg>hVXx%7;VlbzLkC@c6|@Si&Z=0y-r>>P?Jw5ZUSa+TiZx'
    'pwq669_vw8^4-yzuh*MpkB(fi;Qk2gdGM(Dbu>fGXf(V7TjY+zL#OY7Q1ge1Ov!vlRG^)9%Hi~CJS!;Is1lt4(SL^SFBa!Wml|aWN({tb^4Dg!b#+x)'
    'U|3V7zTNSB2;f1TtyaeTF9~gIjIRPPu5hrXespUyb#k@y_T#av+=1X=3RWkYKjpj6KR>u>O^>}9*k;e8Fu$Y`h#Q17q(bO)c*d-`sjco+k)Pd44ljWA'
    'CoMtf@Nvx`D4e%DcfNdF|GNmYE+8m$T1M#<yLr;wKkmuxdj|_VI~x~^a;by#w4@)(SSb5e7+4F+8yst`H|440-lt0^ec@q=M|PB6l8U&4YnZ|tPwvrU'
    'YES5w^7YYqaoK#)o--6T5F~IzfsjsYrL%RiF<^q0?)JubUcZb-M!d-uBmL9W7Q-y2u-Oh)k7Xn?xicB!F1ls+T}c^8Hy0VQ_t(vRTr=<+?GkpN+82$j'
    'ORpQy+N>@hnQ4}by<OJ&uw!aW_bXP;f2TfMJG8J*rPdDPm0Y~!CgR#<xAtc>03v2({Tkg9LJyLj)3nNF>)g73uW_53b-sV(p!iDj^X+Ae9k4h$j8e7z'
    '>&v4Et4Pp!*WkU!Kv71~byY^_760;M5DRE}Xg4kpJUBM{O{yW_3)K9H{T@uP&6)by{E&c^UpR4tLw&XEAqK$b=rJ2zh^5i(PN)8`m0W~(z3baeyX;|p'
    '^+mUV8Ru`{S6y+Q>+6!~`TF=E-!-gli~G-%{fXoru(^a7Uvm6B>fcY>X4Ohr$vZ|NUyuUyAP}o@Uodgf7DkiJ^03LIR;HhO#Je-zo*Rz7eJrl!?BYkP'
    'e(W_i`@hcLg%?(TQC<$3a}V}|A#^L!l-@Qkqh2$2Y}K3m>hE(3TGt*mC-;9cp?p2|GB0AKg6&bo=;@>KnxE!ub&u62pY(%?h2)W~j<QQ9=ga3#O(#L~'
    '8rT-VM?@6#UMzOtEwOULZ1#pn?%O}I!uBO~{41|R{vf~xwMDn~89}>hbQj~UzL&MP=P#Z<Ik4V!4hM|r`Yzr*fBskxsE+vL!uDKR<DxC4o26g0la=3I'
    'pRErOpGoZLLK7@gkZ3qb_2pCIX9I6Nb&?h-(`-}@nH}-9Vuw@S)VUbO)$eL@8^@T_U~=qGAMAj;OL8O6N^5sEycRErPlOJ_P4gnHfAJG8CqeP+&|-HL'
    '1fHWR{7XU_8nd1MoOvUn{@#FEoW#U%!rhL#=rVk#*<El9sd?9zf$Wp<mylN_1!>~gxH-vc8RX?O_{Q?yIM(}(nU{L_L>OI{-rmug{Gx*=<vk;1(?Ksu'
    'cbmfAU*;Pr0!Uqwve8>O=)O;D0+XujS-i4Fc;}%bx3)m5=c<Q`;;ZbY@6F!ru>-svmAG%~)j@lT?DW@WfNhw)LQZ;_c|_JzYrEv2NE@LzZ7&WI*{fnE'
    'ccrC){vMt(uC0Q*;Xs&wm<=(jv}(=ucd%(*XRLsO!$G%}5jfmA<b}uXwzcPjUb0tlYkY_JzB4-C>uUizn7!c%HG_t1JYX`N;bQV?o=)!~_vc2=1h4u)'
    '5T@O2yWXN$61$^-H!vW9iKJ>TA^I=rP~U96vxH|ZkiJpP1&RSCu`NHl#$h@ym;Sq0q5G6Om4j2)y3K|^&^sDNKF!KMXc2VU`eSxAK<rM=0216aou2TX'
    '+%tCD19qB33b(8NkGA(*Qyf{=h5z?e6dNa@qLvT>M1%0&BRq2&Aqzql84&i`PYS5+>YiD9uk-Ean;#TPWkyDZeIt^Yxy?kZE~J@_=#N@RvvES&7k77C'
    'laoV~%tY{;ybd;y_)5{-*gl65D|_1$jr%ggHwVhSvg#IPceDynt4s>ts^opyDAUc0k{%&}X--p;imzS}2l&3cz8vU<)3TCvTxsp}irruPW^rSG+4Hb0'
    ')2V55RY%vW&cJs!x2tiQfvuf6nJ4$z<hZ{}l=h%th2eb3N~g-LbNpV)wb_wwAi}8N)Jt>3T<r0~K*Af+VqU9e>d`VXOrruy*!6}iyZfNVBN{W`L-zx_'
    'VC(rCG@TCkptZLznW7&rXV}qp?ERm_PRD?=664sxwl4Q(M<{28&A2^<EyRFAL%6Ty_NbOE#sg};4r*iLg6GUulL%<Q;d+uKQyqI<f!Ku)ndN68CNYa`'
    'U0tP|(4mToi|(vH49@&>f{7J^^-v<d-z&(lZenW@8xLTgLSK!9(UNK6247g)?w+|>#pgDt>Abr=0}*q}?1t@iudUI3dK;f*1TmwOE$q`cM>Bzd-&M!;'
    'z`TG2gT|szgpIlxN^<Nvbf>Eaq)a(?849jN|E_sbhN%oTX>1nzE;NS*0a#V@O;Xw>&&S;CR4dJ|cXxQbM?`D{b6Ucjbm_W8B$0yT;988`6X@ueD_NH*'
    'aU!F1tzMM4^<p|OGX(Ch9SYYxQa%Zp&VFq!qP3Gl1TsgXu9K;4b#_TEtXgs|ZmaS^7LV!F*qs$Mnyo|Etaz+B8UHO+?5syvBnVZ;1*3fv*EnyToUtj?'
    'iSgE%5_gyafCIVu$Y4Fww@UCXVJ9}Z%Y<spBeT#bcg7F-rL^Mfxo#<w%a`PKYF3LhtUFzCkkoEn(F&+z3|I5a)F;d~i-8k0_Hq+W;eiCx>SBx4@GeJ&'
    '((Js#u;{Ad!!c1@@Ev51x8VqRqvziL2G)h$on=-ab4l$h{F-g&&KqbLL$TAM<|ik9MyZ`L+0{k0MaqInWC`2Q@DMoUP<*=}qy-*0MeRv`a@NuAay#!W'
    'rA#W)8Q6<liC*tlwK`$?b|Y?X*r_GmTuf$1VmtM1sl<uS9y3(++>MV%vrb8N`buLU<^z=!a?_O6P;0?7e(H4Kmakf#U@=;s^JgZ%J@iapP!cx)0N}NL'
    'b&$NYs^<l$ALzTBt527MKr^m)j!FGg%gzFXx}aLe9g$ftwn&Fath&{4cGJXsbBr&xxA4wHvCQl!?v5u>fX`;s&lmAS%?%aS*MShsPrA8<YoE;8Sm}>v'
    '+^A`cnu+arRz{k$>%<DR0tOS;$#pW}-&)NmJ{fXZwnvEV>8c?JE)|+-t<%69>G@-v6_#y8fcmmt&H14DEQj-J-yxmpv<fmF;Gxi2KZ;$v9?M#p=As|M'
    '{Wc?KNnUQH=F~Mb9$v?b;~e2<SS1U|T6Ii&{{q@ZY&aWqA&&+Hfx6n;CADat0~(=D`gtSQKh)$&+xu_e$opKTsu4XpGW07=Xh}+wR;)YK)5~m?&du}L'
    ')w~km63Kn%7*96Krw|ovO3m%^HtPB$ksv4YRVR9g2`624QWCp3PYg_$BpV@Xsw=BHXBGUo9J!Q+)zCIp4C%@~S0Cl;c5WORb+&slrS(R*n>$epwtTFw'
    'lq-mdD1*Yrvemtr-5c6W`pZ3tfmUercy9XWPT<rJ&ExbW){loyU8Oo21S$;VGb7x=9-ug1Os!I8H8x`xomDm!$BWtgCX#_^r!hn@&06)&$LJItRIhhv'
    'c289ZYaa`AqiRE5;q?_hz%#e|FqK>@Z7i|RQQBazvWJIUPNzfVVZoZ5Cz_L<^lJEIHs7R|<8^J*j`ENk&n)FZ9Ms3Gk0FR_%PphblMAKhyjxKESsCrK'
    'p_86zWZ<gZqe{&{q2b@J7Mlw!<;C&ttke~Gc~2lsv~1_Q<;;29ilDc0PS5T_8LgqMn$1<BxpkKXWs#}3nSJfJb&;mHr>}HO+cdMK>XHs_&07TJ=F4sW'
    '+{a6uX2fPQCAo4nii_ojGM7p$APY&x+$0-!<H{mcVkK>c)%1cA^${Oi!~`{03@8X`(ft4`HRipT?&p$csbGdeklYQ$s#SlH%HsLqJq=gZ>>f9R1&~T?'
    'xx-c(YrI&HwGCTdua@ro3{(D1dfhdKbE&C?&SFdCVh3to#}4H<qU#wZ;~NEVs5&~rZW$4>nJ8uEn4mZ&8nHxHy*uN{y2_;Z#ApP{J7>9GVz;B!%jbKM'
    ')y@h`8q3`99E|m+p+j2D&d!2=y@Z%XJ*4N7#!6jZgO(bfFAd}r#b_#NHxrz=o#^9avu(CI=)K3U=IDik`SJ)pm8@+cS;c2GZO#4J+~qPVG&np*#`d-R'
    'dc(WjYMJ5#?M(S}zm3+icq0z#t%T4!)z^>QovJ@m?jHK~Dx}Ri5vQG0h^FtjS4}qPMmN}GSZq&s_4dZaR#{)Sa#}$3OyyD=ACgQA?}Wx&i3)X{tOmt;'
    'sb&J5ZaVxqMZ!m<s4VVn)M~CR9m!w#eVU55Xrm|FNxyTTCyM1mjJP?>MExy?I&v*1k0?e?F)9Yt$gClJtJjRcRNRP}t9Y3@>*azoMjN+n7b&7^Cc0Q}'
    'n!myh<rk1z;6`lChA?V`?i?uIuGFCsTU`N+tlmS%5jJhv{yo}2yG}P<>K!l%<oHG4#I8892r^8pNluE#Zt6@gnQfY^QTvjM3<xDp?36P?<u=G|L78@='
    'ZP)pGYJ9pZQgfe)b}1P3?ft~Md%BkNP+zQ!>n1Oq_w=5dhQy&MGzLyBZw(j`DPmA>Ji9Mbkc{LOViTF_-oMQ<&iF9hxL3_-=x#IpeZJ*0_$$fAd9lp;'
    'NRmIS<Fg)q?9jRXXa~*Zm0cLN(duAJH#h6k!l$sB>2&K^Cx2dz(lc~^O7Dc+5E3}1O&1oSxmLvrNo=D9lm1g?<6B=Hn4H6LV|*{eyBU8;l{)RYFK-uT'
    'R%@!9T%3(VPPZ~_Izir=Z)e7_l$;AfI=);pbmKUd$Ko1xaw&CmtjtE`L^3|e$3k{e>N2D4!S#0n=_8YiCac?Cwsq}+2(RV^l_Riq`{WcNh2W|~$7VUw'
    'U7YaaL4Zp8K+9!#w?}i=dZT{Tic!(ojHy^JR_igO%~zoAu%2yIai%add=bL$U-b<P!AEdotp^W<P?3P+V_xXsabzyA&5E8gqQy0fHn^qeKMbO>n{148'
    '^V?w@Ol&N!D-N%R(<rl!E+-|lM@o3paYKR2F;qv1E}7o)tWVdjDsCZG4ZXzmkQ=C78)~8@o2inUvbYo7Qit6`Fqn&D>?y_uplqgHSk6nMwvQaOdSb=c'
    '_i~X&ryI&BLL+x-o|1=JvbLMoGWK;B;!!nqJh{6Be?&_hdg+vN+v~ozp4~&HZ8yi|d#<Dps7^r~Cs~v}x<?exO+=?q@*VFgo|3V2v)fW_0Mf;?nXei1'
    '#3<Lm?+4(#YgnHg1Nb^Ug*m#U`)-9H#cf@JIcsB5O>v{qA^|yfVjv#cYM+Rb6ZtBkF?X0R24qE)ZUXDF`5V>krl<I%dq7*WhC7)xgZolOcid@h6E7__'
    '(ha1m&Cs|JT;BP0Dt*NQTx`O~=*UoAJ++xa>Cj+-#G*HzWCQZVkIjbBUXJU8V!XJMZ*04$*K}ucNp{Lwvb*XBOPaQ<)yLz8uv2w)qYw*U3r-Lxw~80K'
    '=k4}xm6oeY0XZ?Vz<%0=h6}o4lT5DNHe=XjZ>hK5EO}Ptv`(Rz)XsJ6XuTk_!#$%;Bp-AfO1TvX9djm-sLt#ARMORQ33t)mMz9T@qq^f970Y95`Rq4_'
    'YiNCh6jVG7bNcSWHn-9d%O19MUrH)tcx|=ikG(`NUtA3ndJ${Hb<D>y`mW@^1blJ%9ub@a9~q#13Js`wdBVbh>yeL*S`6CC)ti~yoLv(Rq#`PGblhIm'
    '$Q-tW-l(anXhpDbzB@+l@%Fyj)F<%Ouy3n_Z`>xdO@HK0Bd(?hWN_S?lNYVJJzWc%YoRg7$U2nF^Ec_V6ntc3!S|8Ov0g15StVUhlLs3BqFp36STVxN'
    '-Nwdjw`RuAb812s#nRTy5oa3{7i4L*SP?sPc$}327sS2R??#=99!wWL?c0-mX|$;~yP;(;lASBnh;_G!YtQg+kkB|g#tVgwG3|tgQL+#@Rd8`rtGo5w'
    'j>ouiyJQuT&Y<0G;dKIu#m{_mqMuQ!uxR$-P7cS<f%;;4Z;UyV>#Vvwnb@l1(K=7gBI|R5L8d}wYp_%a(hM^h$wG;CY3=elagr$$-$TRGN}0_yei;hn'
    'lCD^&wgQQvQEI~lIj}*3<9@X?++vkdwK+iwt;8U6N!pn#o!Ik5xD%}6%jSX1MzmViiVT{Bk(4P?NsbXA&C$HXwz|<=WIrv{M`@cqRThI$-C!`Miq5Y!'
    'Id|^G*JIJY5~g*MVix61^S-9VOLgjtXNiU~s6|Eh>{rEPvL#cetFKo+U0B-MqVU=1&%_z;2CnIQz|bO7Q_yCo7!+xz&vA+-jbIce&^R<L6_q7k?Q{{M'
    'mNjg}UqINZCbj4N!3wc=eyl0)Yy?eiYh1O`FFGX7DJP=1m8?r+Vd_}gH?tQ5w%psH$LJp2ZGk{XlSad?w~bAr;p|7N%6g7hIvPhfiR7-htOo$%SXZ%R'
    'nKBq@JvTSD91~;9)~>NC<$SOuCngcD4fT%gb}osK=T5gotF5%!aOTlVBWb&^+&hZFtG}GT%OqyPl2Z1a#ZzbEnsI2cYeizCIMYH(0pCHNMz+ujQPScB'
    'S@_eeRM481QC1cu__(g;ox_PhGCK~%ikICLwPQ*4-iCAIhQ5jNcs*NMR2fRjTFX7+bJLr7fo2OG-?&#%8=cw_WVmd)<@)C$Cy(`;?|Al=9X{^Nurxb6'
    'htB<7eo;HG7v(jiB%>CW9gMH*LKUr~^P^pZ%__-BYjvBiCbYbe=RUEg2WOd%>blpeWA~UscFrXxvFmOa-)qOtNRN!*+O6gvb&WjI^3NOVS;NPkmu{3|'
    '@#1bv4yWiSGStTrcznyX>)VMa71&m><EH{fC%B4^sA5IhCoi3m);V|3*hH8HEZMv{cBHUD&C>STxkr2j%639NrjOD)?E+XR_2VG6W#wk_%2fp{<%o4-'
    '&mrQ~xH8EBhc<}ZsN3HrCj0Gj-;83#;%Hafcajk<*IraE%C?se7*al?_3DUvTIkk**b9hQ&M(_{5<bkjM<D`*q9wT*$qCiFSnQm!!=|tiGWS-msa{GI'
    'Ob(W<F<I`O1*8$_(d*9CxJ^JXm^PgZO*e9s4<Pca!%|9T2G6UYBs@9MOs>MFeSD=BQ&#FCaPAcg+58ad6*iIPs-C=+ddNLmYxU<l2h=H6KC7eoQ?zi|'
    'ILXmvWeyZ)Fh`c7r7)VGGAXUtUSV`H5FqQcu+0hs{c=gPU|E@Ulu75Z%kA!i8dqF0Jrr5bXGw+24Yc~&2<7-3UoHetw<Ya@lX*}p&tP<2C)Sg>!^lo9'
    '0Evh&>(4GxA}F)%oIl#KSB9=G)}>S?A`GK#-=R1-UECE%r2G-tOy}b@1YR~}zLbSo2)`8V+fhFG647b1dM_TGd2E#KO?qM?xS64sjc?Y7H}+9ylDFF>'
    '0FUBzwOf@|lXj}9Mrw;twRIk?*Ks9V**0oR#x6>HCx1kUQc1$9?p22i*W3&~F0S&Ssb0@GhF%%Ax{GYm#6aHYS$(iu2I{FnD#bDJ(M6f(BbE{a^^FjL'
    'X`f@<J9T7}+8?l!)V82yY@5=%qYY#R2dZLU%oseaA}!nTkCa|v733%`+3iinsa<E@yKlC7)XvB2%0g>H>VknbQm7t@<+p+qSF29fmy`>Yyq(fUnIpud'
    '6SXG4I1LCSm`0kF;uXL|;!y3eIr*@Qpp0#6n%mgL)>*O^T(wT|F*P73(>iwLbR(arT3lH+<}Jh8wf4eAswm~PL+*x}|K8cQa`t{3${sFxvKf+?S`w~6'
    '6|Ozl?&^kYIr~sBKOa_5zBn0dr>R42a3r!9jZc?LeNi|ddcqPfhsdSiTj1uIK4a%~y$uj!c0J6*DzUwzjMK%5-VUxuYQH|oT3mDziDl}?^{f|{i^*Yc'
    '5>$mIy=twBnfw_tAdSYyJBuZ**?VY1N63oW97w*^U1?S_yM=u&$a0Wu?1a25#+;R~Mkn)QEqbh(p-G*yC#TKYviOMBMfQz6Bu25#H8z8FKc>KqNy3ic'
    'k@-r#Iu|am>iMuiaF^G+ZNeFtgO&pyxDh&t#-_zwd7oa6aHTL=RCaO-X<o67UEw<E8Rp9b@5i}slE2h_oV{3<xP5=aFGSq5$DKn=C33o5u_jS^#Z(5Z'
    'G}>uQyY*^ysG{a#*8mQplK$hdU2>gCj26{nae1n>6Y%ZSoUSt|!*2^DNw0bWJM<ff?gkc>fupREy02NznoHYv&O`lOMC|2{6;vz4WreZg;~FB<1-5PA'
    'D_wIbHOM!%gkYossXigfZj-<rHb3i%Hlpx$$Cv7n7`r&>v01^w?p<q9)}<W*&&^P9=k|*ik~kqfr_n11RU+O$R|~^9CM$F@)Qt2R(YiWl1iBR>APBT@'
    '6*c8lMV+n@*+s778R0{-joVQ~=f-WeDVOAIS=;jBIyu6qae7eL<z4x3t2_5uvz%NmRxRn0KLwLpeg?Ngl%{f2YUMIVqM0Ca`7yV|$jKTZPXbn(#!`CQ'
    'I23__a9O>~7;+F>bn}o(2S;6<;}llBlO4;BPx?K@8RlwJbQ0a=IZU@(L_eG9^fE}W@02=$I39%gd~)9h$@8WT(TS-o3<2#bLMjFm`H4YhtF99qt<`-)'
    'jm@d<rWBWQetQuJVU#_;@5Y<WV8HJx<T+oc_Yz#7Gt|<u(O_-A&uus3rHd|~-DE;)NYd)C$eBBqE-}>VmTF(vrHxYdnz|o}(n1ZUEVy|j1Zi|knowrM'
    '`52~X+}Rm9Z0@*VYG&|0hD@*ZTQ)7k>oCH!m|@ag#N?oQi8{e;&bV?5a+r^-gZiw)4TdfapWxvJS&MCC)akC32Aq(#$i7o596Ou-C07iUobz-Y?WhSc'
    '#f-{HsuQ4*i5_qxK7opDa8+Lw#A*l~Tv7`NR*)K13GC+kK6iuWQTq;%c@!Xeskqsv_lH)%)*yYAh!rO!0au8{X;d54qy92F)`CalrUy3TA->aYx`6Ia'
    'i>?z`6?22u0mIlaoQxN@>7#4pN3|sx@A!k)3~Y7Ik?k<u$*Zk)uz!s8Z^b3ssRXP>9GkZ0g+c!4a{{e_4@)hsfUko8)NIa6a~6vooMvjuT;Wvzw#q5='
    'HU-f*a&GBxTC8QSHDQ&XBCF<rxr@+r+E2DBsldd49+X#yiqqAkPGx|sM)c9Yjcyz9fnCnU`$UL62~KrJ3@nMw4P*6HuizgSd>c&)$@oFD&Lpe!lZm>~'
    '>9pn4w5mo^DS7NiTVkwpZAQ)8l(R>-{<3q@$h*{*vcs}`IHe_cJGwh^;E*ZQ2e*DhoXg{DmVgr{KO60GTqfk3h<2j5-o=A$tFSw?H|{)b`T{}8+=cvX'
    'Wzj6@T4>Uv{S9AvOJYGhncn(VhtObCt;JM7iys+;Y8OY-e12~AyZm+qtzh3#pXVe5bJ9_JL2#kH)!U_e$$PaH@C8Z*Qq;#Yc3QEEn^^@poUV-+v=Zji'
    'Rg^QQ$%3p0PR$-`RtS)xj-z?2pNXVbXo59dVbIrD>{>h*PNi+@vfM`7TzxT&CR^px298pK0o9h#oII;=x#a*t6Oqe4c+B>9lqw6$R5iYXp$meszOhtq'
    '2D+_tK?DWas;8RL${2yDm0{Jpw3~L|rrt$1Lb{KHNHLJjwORLaJPmQdx-L&AdORb^)tLZ=`NEXMGQM7R{P{`nm?~!mLNr^5+Wa(wqA?kdZaOmKrZPxJ'
    'i)=#4>i!bHGh%>>lBMC0&Fj&eN>I^db<S4;rR0svMMA|yA?8OS{a(W;m8|vF*Jbu~|Ng3+YEFNb-5;*mS|?EL$J@>R*y=&rG}pSW(~bI8i4uA2E+y-u'
    '?e5H7*Zipk-H3}hxiQpI%KMj>SaBZ|bA3K8ZQ5L><SqlL9kWaqd{KG3naRO+b<}QQF6Fj{$)<J;%%Jm{;?v`*N=2)byYJQ$fq7`qO2lBOQK>Agz-fR~'
    'S4AJb8RA-<XQYFh50QNei_T9z&UF^c%k;E!FD<+?oo(b{h}Zzk1m#b&<266+NE-&{R3=#Bvbz$08;i>glO4A-F&N80)7jARtB6KgZJX?_(Uv=%%T}Y+'
    '@*~SY(Y^$d#SN=iuv_mZLvxkMRr~p5+AkOyo}Jx7U9xi%txY;uIfT~fR`<l60{4CjaYUDO6Z>I%@8pT9a=A}hzD;%#WY*YaI)zGWbn0-mOm&$VL)mL5'
    'Q`)XqcqZ-)D{dUY=I*USN}0&M5KQMv)Mzn149@;_fYK$rU&@H{$a$;OOM&^dI}RzUV;)5o+jzHKFSIrp`9k}ptfF+HNc9wqFXQx(oc8d}J|Zq^GNC8b'
    '-qNBXWW#d(l)F4yDYKL~t!BmczS~z;2g$cqsjY(rSxzTrz!A1umhZCdY<=b2eVL^0n@@?JGTcro>6nx+Qx}S-*jtqnPTTEK-&ye`(TW{>^Oe#OM*Hb#'
    'eos!i&10ihC&$_D;<S{f2by7{&MZ+E%11}5>+t}ci??Zs!Q9-TAV7yiATcW8)4-NG7UyVNs;`Fi-~iy>A|gZr;&4)$X4~a_^uTr?+lN%s#+8J$h&k#v'
    'kYZ!wOVfdBi%rJyRUY2*+MZj_43Ve(sRla<Eaaxy*jZ{F!-ElbpHEj;BPmfk^>T|{+00ht)8crIHA4MV?3UJyN=2IHuT8fiJ4MsgmW~#rQW<I1xJ?a8'
    'p=R6WAeBovmuy#2gHR>73bcGRyI#AgXpG`QRhw5Pb@bx)f~#_~)YIov%I6TvI-W~p@jF7Kq9ZuS4X}>m%%tSr@b~ZqY+<Z`Cg(lJ=(;0zT}WevZ1>tO'
    'w_0g;aM(4aRh^hXM~Z<7sP6MEQFvNrn?}V7sP_<&9l4p^c(h9=y#ET%*~WA&viHT7JvU!Y(ET3e#M77i?6}G%51BoYrXx8>AmHhRS~aSNa&In+OJg0x'
    'wxKj9n4M)kDk90;WtfWZIz{WSOQfB}?KG~>9KWy1H0sSmCM6TAT?^N9v}g}7ZMcN>LGN(a4JhfvjC9=x>D`*+R3^*YVYy#9D}v&1bQHf18zNnqOR?+~'
    'i}eqIPOa~BcQ6hz`@VN=oum96(FWVm6uv-hOBTj>w3uE<L2aQnw`!pp?GP=~rFKxGRZC`erb)-=yQ<UY&f;+3)*?NY#yalCKn1dMxF`VH3@V#z@;s&D'
    '+Jb5*^3hiS)%FHV$*#ys+nj9Dq`5@X>r1tDQMlM}f4ir!$uK=jiLFr+Z(vAb2M2FuD#s1Sh5jJa&4=dLV%LesL@0Ia+?SBtmiqbVJgQIYTNWdnd#9i4'
    'LVa^|DGiq!LhwUJNkr|Cls0DhX2c&gW4@y}*bd<!RNweUv>TuAGwOz1>ywSYrT6*`QamSWR;}DM$MJ}ry{$vlUEtD@F0E{{RHWd_F+Fb~N`J|mX}+?k'
    'B(`~f?O1M1Xfg>WLfu&7h&TL|U6ZRDNT;2s92z}QEii+?YSShoQ6<{nsnx(Rv0(!trc^%N-K}kyQ)vW62g%LJqT|JgNmMSCI6Fi_{kz>P4|=IlUcn)I'
    'Qcq0ABNK`SC}Tv=V5nIS1iPI%bfrqYb-mE`5%Rn-El}zDWyhZ#LR=i3rXMfuw$WDKO4yS{Uzl#x{+5nv6U-D~OU<W)Ldo*O&1`0Y9DJctWo;fJz1hVr'
    '%oFJ<N~+~9(~HRBVK~^$vQ1Lo*zT;<zT}jII+37-SQUc`UWK;zCf@g7g6U&Yk0yt7VZ)4Rjp0#Q%8sn^1Zl}?qLfLjqCNjzPI2wTyfbhj{G`p_Gm>Vf'
    'FjQSMb|E#wbb`G_BzhePOf=0<-GN&VVIh37t&B8ludfbpeSLA`djFd5FBh=@>XY;!O$LkSQ}<RCJF$Ir(+Z?_oR(Op1dQ*+^*WNs50wgsPYCg7)bk4^'
    'Z=yWyMnZMpEQq&R2cVZkRuCgMs*jntyN-2~JDyl1Yn;)FBa>xlHBN=5)q}as*JkHEyNfASU=)m{)AOSayQgR$4fyDe?n?dhfoU7+G23bQk_Ot}w4LnY'
    '8d@)+^Qs<CafO+9whUyK>)D8(ah#4>ZnyJ<vhSUzk%8D-T&AO%8RaHaRHb{fDV4~g)O9wFZD0o4O_0E1kNDl_q(8sS&~0g&-m8IOF}tWKMK)>cOfiWQ'
    'vyd1Mwfx=rJfPqsDVc<%Ldg)c1f60oP?_tH$q>n?X0ouTEK$aZVfEe|)Q;?BLUApoxjY?rCq2ECYcVHUhfpZlYLL}v#-X^V>w{X7=J)Y?BH+{%ZFHIU'
    'uj9I0(<CNwLYP>JZB5TuFv_o^y|F(ZJXDdYiw^U9IDYl%&B`KL&6vH+h7krQf4P+{s;IPT-ibi29w}W~<=X`F&CfjAKb*5fCtu5E8z{QlcluU*U*e?m'
    'xf%<W$a3F4Vd)GjG<*StJ$ASJE~IJMRcYPY!gn#@r_WY<6svHc;>Z?;`-nVg?JT!Z%(}Y?%=O|o5~3!2bhij1nX}K&j56y}SL*wg&gR%%RAGXgCu8cY'
    'p^uK%I9cUO-3+4j8oY1I-OAZm8RkpJOI^D!2(-4o%W)v-6BWr1&pt#NVf(ssu;xwQz!>(ZO^;1koIcn!X8qB;bVO&vwS@|uCJs#%g0SLUWW^3T)$vef'
    '@8+aY%32uXPGNzo<3V~7?6>G-I@6Fg+dDxNOVy;sNAwz+yTor1bnWUCR`Q7f{dOI+R-=_vA*Uy-&23|y)w0V$)8)jspLEJ6O*~Qgqm;U@R>)y@t3V6f'
    '01<DD@8VF$?Gg7rWbtMH+(ZV*{R%fWFenbQu7H$_fvFW5`wLuq5w}wa7xUFDYb4si6WKX83X!EbAN7*`er{rpa-D4ryB}#d>hsx&pd1r~rWV7JGFw<L'
    '*@f&IBPXjMtaqykz1Fcn0wgSs5uDl1w}tR)azCA+M2EW6E2SGhqvNPGYww9#=1%Z?xYc6V8R4gEg>Lj%_VxP#HIr1T{oVeY+T{)_xtN>P#Kx{cAiiKs'
    '#iAHnt2gIpu81ATK>*T1D@QqB2_&@-X{sL4%-nh)$JjMjZB*3_b1|K|UFqh>h;hgehjB77F2N(Lu`vTvnhW}c4RX9U*4DPa9tco&zHOiC_`SPnovAWB'
    'KbY0pq=Pl`31+rf&6UNrgJx;L_PLl-g5fyH6wU)~ovW3^Or}@j&$B!qSZQk_&fe4$#6=3_Wj$xNDfO!3rxHbUCdt^aK&+~@nqfQCy~10mDL058*lujE'
    '5s8ZHP`(M>S#0x@gcv+tFB%5()gZoUW0`JI&o>n}QAiwj`Dvc@FU_ruZ#p%fdR`x#KE)Pw>X@!EY(TjU?Xy9y8#sg&hEyHN@z!}{cUJjUx5E~eu}YjP'
    '9k(Rk>}-cKz3tC|j80emcq9)^>QT<w&#x?=P1wl!vP)JJK3k$U(?;%au;~1`w{CCs;{e3;y&uh$X18<?k_e;FnpSelj#RZcy6i91#MH&zQ%1bpidT$H'
    's}TvDE>TKJE5_O1)R&i9hOT$c^YJ#3r)KO<ueW4m+sC$IYmMrwyoXySNT@nAPu)VWah%Z6%%C@Ll6h5Y#A;$c<iw9uI?lq8d;csqCxc}L9UAMAlby91'
    'RJ2c5hZ6&5LMI2`;fv95qWY?87p^!L)b*XSY;{_ixr@|%p;(DozR}uIrN$oAaXwLY%zjJLNW=|9aHDg$N00O^b@!WVD>$0N;~Cw*<w{vIF{qUn({yIQ'
    '`jn|WPP&&-pJl8xvhF9gRRRtzj#c_#_+~YoVeg5g%JytAe}ZXc9tz&}ettMw5>zr1VJe&gCmZ`<pr%h&If%hC)$(8!lags3X(c;>i8$<Mt`*x_1=-PX'
    'Dh{Z|d=~91w&uU=sr!ao&dV_)7ahWfel^R6*7PpuOtE!(zn<<UtaesXhGQslVLz4Yl_(UUVLLTr<EQj#)k-*-DFa8<n%rHc*U45L>QO3Q9t9S{ATzwy'
    'n?$Z&63z>Hedf(`VT|SqY>MJQNX=JsiA}b)xOJ`Jj&~>IL=R|}eS|1iT$PZKK;@vcD&k<7;R@Y(!Ng%37-xrM{fT+H8DB;7wBC=;3h2IpU3$r-dRyZf'
    '6G+Ba^nQ9gv;J%(RTJ95J9N)%yW7^#)M8g8b1~_<e7p7_Sy+UM4v}v<lbyN<v8o*NK@y7A<z$D;@n||#KKKU}|E!u=2zR5O?M)Zb5^anQ%)F0Z@wI#B'
    'R+omH9<Rs<Jd^b!y5FVPouihW$~GH|kn^0J$j(JB+u}=WE79B2SGCBLu5cGi3{Ls@Ju=BM-T9<C%x5cpOg5s1mMKqqT6d1F%hA%((6i1tx{XC|)pE=%'
    'Ko{s(h~eYOypeC==UH_CYq$L5veWVlLFww#(0+~weY~3{SDId?qJwc#Ttkh3Ft2b>TvbD<e6bW?A2Pjt$DdxZrS;maCVhR;(QfU?dLPvK=JHk;-D9nw'
    '(TKJJM09}_)m3U4k0YaUfm`+XLdx{Xg~}?mZ9|DYzw^)R-abT1t+c<`&umJgNUPO5ZYCC-^>cE47P|*K$Yfj!^quuNyT$v1*`BPVZoT$)C2n^aA#%*D'
    'DIaDBOVDt5i>@Q<VX?j%#HL25B&_1LaIz|udc2m{pUqaTSM%)(3Pg*AY`aqOhty8C?o0Ynb&gJGF>TnHlC|@X{MTKq7KO99T+-yC)#-U3iT5I#q@&Yj'
    '_14>q2gra!<Jsug>4`1IpEU(;4i!2f-0Vj}Cf^Nh>ixoSCFaxn7H*{*t(hEZbuMLy%~zMh^oYEqnT-L<=@Xy5@Q2QMvxoImV|nw%;{0l-RZDrlmWf)u'
    'cKzaC3R#y9HCH17xF>evGL=>aO&@%F)xh~~b#!L;GTt1mtq7en^<s=_EKtfHD43!8WIUOy(N1DI2ubLvvQJIU!{J4&?u#7(^09}})g0HQ&<rB{@-Y%|'
    'trlNj9~V(f@z*YgnK&eu*kTQh+QrR|S!h~7Xw0I~W>DOc=hMDD?iONIvYfi~xyB7~6HP6<xp8NO&TXc0;?g|&f|HRy+GHWM)X646OpeU8LU-AS5bJ7v'
    'jm+F+vf#3{a<Ouh;yJ4n8kO=^Dg<H2jB<0+Z0|UW!vSJ?w^^}mWw7Nb!iMm@*d8r{k!2y#$3|{5Fj$r@&CQw4G06xQ+QXcb?(Z8gEoZFrK4~Opu$(0I'
    'wao2Wl3JMFPr5n`BuO+yo@6~zh{f@CMUE_^Nue`ygM%4HmfK5o%Cu^vMF-^Jj5$jn!>i6oKenm#c1!ISWeKH6h0DAdbuYP4^NM1rCNqqWi9v<V6f5Ow'
    '5`vu2ym3I9(wV5%NF$9M4)K7SY?0J*7enQE+KINMObF`5d8xL<s++nEUz1ena8H>f4Qgg)b|Kb?NPSJHwhEZESqqcu_&Asmb9b<qa@PYBDocwZP5|@C'
    's=MCOr#Gm&L`ILgjZ(WsYtX(F?oO3S!n5-&)HG0)jOB^Tr8=MjCj_UAi{0|AR_Agicpu~z<hU7w@0px6>_kVYdzTHy0{vTjc%jPJh-xK3K3y8=gg>wo'
    'Yf!6QnkJaZnoCbWa+TQJrhHdg<LD}1-sJDp_Cmy4rc|qJGEj1v6|xY$G#i^uqj`{_U38R)1fquBbLCX}2w=o2iee`}F-Mjn0U64Yrx~mV7XvqlyhbNx'
    'B(DTdJ-#1Z++2#SPx33Rii8q#2`TOyj+Djwa&n$+N6$Rb%t4ikn_O)}%bqZY7dD*Abqa@hu3Vc`pn4i=T^n8ZtnM0za_nUCW&BEtyF*vZo^$X~oD?#R'
    'D;Y!?E4xG^rE)?x^dpns+nCwzN3hMMv&~A)o_M$f#;eLSmP9ArZfqnIsobQT3g!nFx==W;ORG$Fd$8)gK>grGHc=@s?G8Fa%HFJZ@u@|n8t3AEpDzau'
    '*uT%q+GQ*Z)vkE2)mN-ohP#p}`kN@BD1B(Y@SpqBbS}2N?KJv2s1Gf)GVUzlY>=F(@I)QABGbk?U&{Cw8>qg@`({SP4TZmb;}piS-WrTzLzJW0k}6b9'
    'Q#ax71{nYLjTgi>O+P~LcT=z(Q+wm>51z3FZ{mmd0z7-)&$=djI5r1=NR}?@rpnk5{8+uSAou6$ouz9ZyrUXm@ZHi)o0dWFCC8>Cyh|D{Xf}zx3!3Ga'
    '0?k;QB#|{nv4nR<QS>9I@3rSMjD-J>7JiWDrHx?O5c_kBEn{sJiT$v)43<cO7av=}ea(7u1m1wq{QcY8mi%Kf{O#>1+1ocA6t9r+9yZx9{Ki;sBAEOJ'
    'PW8sVN!pua3nrv6D$6t9#Sc?pc!-F^Pz;YmF!-GfhvDzr56AGt1h)V7_Q}}a7}*_7KnrZ7vpbI;gB&pS7VwraJ`BbL()dtiUNWKQ!%I$rD_J(J%a12O'
    '-c}9L`0)4!0#X6F5AP2W{s1q+a0YySJoce&!L=cv4sSOgWKQQLZ9|3~TZ|H~iKbvE3<qdD8B}zJr!CzvIl+QF4kgt9N_(?UhG2n%F-P+3iH{dJT{TR>'
    'vLFC3g(UvL5(#gR7ob72(;w5L{)8%5=@OCnc<d}d;fH?1Fbs|${nR5M3H!+3){Gj(?<ONz!qZB_15*N;u)&b_FJ3qZif*VtI&V)tUxeON$pSXWZNq>('
    '`#*CzO8mzxvVcKv`X6jnU52xP#5h^t-v+?21T_9OO8hr$8RSEbA6=by6k!a$Lt&aT8BVqsQJ@VpP8;4?X>dxQAHfwk$UA|g2%eliUaZ6K;pa4O0hsz>'
    'oWk#6Z*~}dH$G^ZmpGfI!G_lVyrGq#Et}a0{GS`LU$%SB0CRua%KEz1%jP8w%vBWnfYT~t$@G&3FlaFS8C*VR@B;8T5CAyoWorJn=JVB?t|=!^a27Dv'
    'e;JY%{S+b)v||Zo)G~n0zJU)J@MUG*WI-_A+RSF72x053JgF4lY~AFxU^5#i>EqkOmtXVW4}$4=^<Plf5;mSCF}9?GO#q{c6*W$erQn$8N8$qo=Y+x_'
    ';Hh9B_JMxHUn3@?Szg!^Obfh8d|>DY7KVR#al<xG-$mWgcv9s*z-AVp4!t%9l6n(dPB4IF@E?QcU{AMe-uy=(el6{R_zLC%D||e^`N%gw$|L^<l&=B%'
    '`;^vCL7u*c-#^b7wi$CH*kSLm@8JiD@cZM>yQnzU_A{2ey_uxtu!gAvkGDQdM}z*dPS%H~V#j7!1(1q*XE!7Re7=+izD4)mN0gT=Z&DPzj=qi>1$yFS'
    'uv_@@;^sz@7x3~H{_)N+hV8|A-LVbFe#SR1OufE=!~))sNfdqX22O8&JT1#Jqu<=G^7GiACiNyVz;OAuF!I_L31{8}&ExN79KgarDAxr015AR<56cF&'
    'X}$s505SZpXpBJ7?>`<p`#&Qe$AIHlV2cBzNM3~ZOoww!FFwD$#P+Z*O*nosVgZADY3WlR|6mEfDdL;|ZHj+?<8(lnv%w*rQ6%d5>6^-ElIR5!$$GQ5'
    '!dt)BS-d?mB$)U$B1#$%>Ia}j2?q)A13o<o32z{Ffpqu%jpYRbiwj~rk>&+VWQjN%$50``2uz$2Q!zHiVoaP9a5TvzQaFys(pVbD2}0mSRK&S&<vvAK'
    '%j0T=UN#3_NF>8CTohAmS|o&If+b=&!?PHk=3;3y!Sf=PPK#U;6$wCti>C<=6%!~LOA09#3@|Z{4}T%rr<(O<?@2d~V=+DrvMiBG@M0>?CV8BTrwAbp'
    'xaF~!kQP{u!>~jGJPH__OeNw;24i`Q=lFCS(0v8?PO((&(3NJf-s_LX&GM7{|5g;?Z)2(mcrEuQy)l_grDESI)#>XoMI^pe=jG0%0uJ?LgA7l4?OuP@'
    '3x6wiI{n$t7j(BcATQtE!aRs#kKi2s4y;=+%8&2wZ($RJLNG=(M<eg?F$V|{43(*3x$S*@|MnWUO+mn(@4UzH>k%y5FoqQ;aPX#H9E`kgw{KI7?iE3`'
    'Nd6Umou=5UG%Lk%b@VeE7i(2|KsBde@Po4s!0o{~tDXL!TkVbiqXnKrz5WR|s+~%I@-x?ibz>Tkpr^$SJ)9J&VsG5+{kdVcIS1kz6e(}Ve>dyvE`v#_'
    '<0%@z@FKPN@K7iq^Yj8RIC|l~@Q-Is@*o|3z-R^pe(4}Q($RnY*MCg~QxZ>}Cm%e=|0kEs0KfP0i3cFxwt~Wg@byL&XKxmx8VX3gfDA~Ex^6$AUb$BP'
    'EBtU%+8Yf2fHQB;)Mk-?{dndn*9$S;+YfW2=qwbDggu75Xy^j8O^aT71~3(51@{5{KpbOC+wu@0^i=y(2Pj659z4?{Ly>HH1UR=(bom)%KrHgk{E}h#'
    'H-aFtzQ}!?LEwLE|H<bEV|bAlfc$T78(rtgzl>*k_1J#R@pyfrZj$_4$oTO9R|-Jf-f}n>`z@|_byEOQj?n-N5J0Z<K-K~K5*dF*Lr;va<PYYCpP<d7'
    '>9PC3^?-R#S|3ZvsCiQJUjBvtcwqeV_g9KJf7pN=m<ognBCMv{Z(jEOjjPcY-2-lZ@&JzXR;J32)C2Nb_>nxkEiLk*eLp{o1r2^?2v0!ghkzBys|mc#'
    'z1_4g^8QyCz5yD#DQzT;v~2Sy&)?nvJP}@o@RIW%k^}V|5YTVZ@)!iz6m0z(d*y=68?Qe$2if1UCEKx|%yYmB9+vn^r(aF+?=JMT4N+G>DkVS5n3rTh'
    'j)I`)tAA<m7u)?Qj?Vz}r1EAstRivbqiXbO5$osje{^a1A>U`613yIX?fNA00pJf5e#(&s2iJj{o<2RC{$&2EdHza1Pi7vn__X#X8}L*5p6z-sL1FN_'
    '@fK}>H@E+-)p`nfMHJ=q?qLtl&Y$=sD!2w93#LD}%e=r}81W3@=dO>Vez6d5(6bOE{Px!;g&$#0nI00e9F;ae>>*#%{`C(^{Pxpm|Ka8L-^e^g&Yo-!'
    'w%=ZM@^gkFX%B?-p!U~4Mckid#fRkuC2zqK|G#~@!*9euY%@L2{Lbl`EoqJiih+EMhg4~=oPSpZl@-iKElmEg*4qIJgM{!Ndzi;->O&l#V*A&Z^`1dl'
    'y5gmP7eilGylmU|Dac=QJuP~if1Ui(*q_Gqo1;EU8*i1PJwnQZ?XOniE&UGk{*||%q2;eHqd&;6n|{^Jw@-O`Ua0+4%0KzuSLT|UR{<)5E&Q_qdr`t8'
    '@xRgZi?M&FU*=sS|51P5y2r|&hXIVds`Mux+7IB9%x?#pVlTZsIz>MVa!<ONS0g*B%9y9GG3Rxpe~LIio??A1dd8S1E})+!wW%K@59Tup{peVP0Y9l^'
    'I^}WtAFcX==e@OQJmE>~j{|*$BAgfYEnq^7_9)|j*5Z$2yg%c_6X(#LeE_iZrF-_sL}2Kn0xEgRL((QB=^rDY6>Zb(|7rs~+eT4K+IamD_(>3y`Oir8'
    'Kr<G^>5B4%7M7>z{}%UnEg71Z%wNFgZ`tM-3jQ=3!}BYzL-vF#&}YK=*AxA|eXZ!<K)D9?0Al>Ni$3x5+gGss_KA;wt$X<JPli8p!Y3X0C!qN`<!_nc'
    '(JfL=k6`~)38TEtysGuFb+kNqt327Yp1=lZaqmc`!2g~Ee(vzscl?K!2)qIE*C_hqH@|<({}=Z~0*`S0C}O|r>MN`LPxgMzc;%zdrpy;3yKvus?(r)g'
    '{?jJ6pC<FPdCx37Wxtz(0`zqdJSmH&uD*G%6p8&T!3O|X4RlL#JxlubwkLlZfIgT1>}7`k&;J2n9{{OeA@FmK=Y@Mu77Pnm1(@d<yTczJVUOFrr%?M5'
    'RZQXI5hfrry!xvcT{6-0kMH5%c>T>OC(l6sbE8)m&^yZam+F45{p))y{s43a`LOn#4m9o^CGuk6w>0!R!o%`ENzk_bklpW!8$D`{$DVJ3#W4m5+dpmX'
    '#n8k1eyPuRD&^IIw*=*xS3iS2WJrpqp~qG~yTOwF{-!yqaq?iaX1ugEXm&m)G*xCJe0Bc^fy~!{ME<Sj`u1$`l0SpgD-8d~g{db|5Iz4!;Qz5)^~iXY'
    '+~4T@e<)o&TB_s<e<${zl(es8^8kJi*}(7KS8&GX7mv&5x5pkZ2s)6LYu-W9c#kL1)O!qzp?@>gFaG_R+#i{K>#2_69{I$BCtos(*VF=$3Bbc=Qv3e+'
    '^B(pZ)!)I#d)VV8{N4L}4?nmGe}DYBJwv-JoE|;bu;yJADbg7j*)w?39-4Y|*1Z0rXE>kW{5||EqG^c_KN_pDS5yJa^BwR4nSTMAKOV%*XWlqkufhUM'
    'dG{W@;TPv0k}arT04wv(zIwpV13i!O>g+zA9|!t6@H~|_;!XWB;raM_{o@nK%GO`Zj}Z7|fBbQq@iW(d`XuYccK{6D4w4T1Tj0Hj0Zs?-d>CO65Xunt'
    'r+fTm{yOtx<k2Pl>l06GC4d`#_{cxHPd}cn@pUnH{!f?w8zc1W?MoBv{b&E{_pmh4KJogS2llwnn;TH=aEjxJ!c*Q0$Goa|Mms@&a`uV*{@y<OLn4UB'
    '+`nbvDRa+AUQ+nkK7G2t?;N~(I`qcW9fOuMQ82xZswb!L<7zVe$QFM+_5ZrRF?B_FnfE_T^zKETH>MycczxQ(oI!CkqHBGsTO89~gE#z-hx)^nzqBRn'
    ')0?O;o1Z|?TlhNbSLpu0Y2=@{?E?7vm8!!({$cMMSnOCJl6lStXFUG`z%2uy>&ClbBhifDU0?h{bDj|1P3w0zds)!KX+Gr+fC;nFbjy}F+PnG*zk5gh'
    '@p%RN^lt0nSB}J!o>}~g3oma2FT80UgTLPrIG}{!9^JdoHC~rq;=P(nw*>7U?iD@8t!F34qxZq^JOt=J2A&cn|89bR{b{W~GOwP<L&AUkA>b$92KgHr'
    'dM<beQ=f70XSVm=dXQQDw+kLd_2>LAHML}X>Arkct6#>x|5JVX2?t)M_M+0K93<5P*Q8|pR&r*ZqpI;oA^P^^Hubp@gpQxb{Hzzh6mwtZJWlpG<?prh'
    '=yx7pt(OO3kDj;sS@-dh2z=N2dWc7h@PSo69zOs8bx+A-8I(OQE+36TNqufBsTRmvq@;d$e>33k;n5(5{wHEzXnFiRsd?9pr1$vYeam>?9^@ZGj}6{7'
    'CUYV+<Abpn6MTgpuRxs)wgo5efaJq__q*N{2iP3(Dd{JpKJt_1Ll9t-U-I;x{z;l$@bX_z;cWz66>N}{ybjeTVt9ztTl^VizKW9tQ$QgNd2Hv%7)HQ='
    'bNg}W{9rHqhb27cf7trVPM-SyS_?mC(Eo!8e9fqjeP5hBxqaQk|AYEpuJK~Si}(LFX$Y!epZ?`&&v*?6G-30^`Hww7<S{@!;tfO}m)9?2d?u8ydwOB>'
    'vwXaS4MsA(DE1w^et(6J@b~B6*L}v*N5bEsuxN$fy)r^F;P){2`keJ|gT?21kHN3Y)jya2W9ay8_fOe>|C`Ic|K^_G|8&<+DnO_1i53}E0@2-@_1~eu'
    'U#ngs4S#=J4nIjdnzVNWTJhqw75<LBhqWi;o}q`oKTZEV?8R`f6ZpKr&pY#HnnK9fOZA^4K~VmaB>MeztKI;yBKq)a>5p4jPg$0C#r|68m6-2e;1BW>'
    '{``K$4)(e{?fRDt{=ls-9QI)P2T8t?<t_C8w9fZ)-Jd97;E{NH3i?eE81MSo`)P#!8~R~H8&vqu@Zx#X<A*c=m?{E*{T8<sR3`lWx4i9{I(>+FK>g3c'
    '@ux|2`sFrAl-**lS@Q~}QKR_frY`LL2JCf@FVGx_YY(}HzdwHk7G^Zyp^s(`{n?e&faw2WCWug5+oC@;_c{J$tJhTkC^<iQ``gXbrw`H!!wSk*T=Kc('
    'zqwNWy4M5Q^1|nDslwm?z|@|B9xcy6j}|CNUcU?Tkix%y$<llIvmSh2VLtT%|GDCEyZ7oY(y!}}=YpRF+gF_aRf*y6zs%M<z;phKZ~BGFzk5IPfy$5j'
    'CG!_9|If@Aeg>1@Ea!hu-uEYYPp?m^f3v~=AN2f7!;U6v`cb2SjAJN~1U9QE^k>5SWXZc=d|X1AyhZOF#$>z_+cVQgXnRlu*_QEu!e>N$IzBI1UVF&?'
    '41)iI!x)13=+Wt#MT0a0GUM;Q;Dr-g2S^mS@&N~u1rZjcO~w1QQ1tVcYrhngc~d3M)LE|+hP}TDAct~Zw2dX-LA4<n76QnnQN-8+s$nnGzQSmY>MyIk'
    'w<-SirTg+$?l=0>`1@~R^ZltuudDd0fc(uwGjE^5etZ3dg7hRE{yogT8d^^e`TeJ6fBP|-*VKC5@lUK5W&~N=`)$qFi2oayc$f-+#Hyz{&dPY562Y*o'
    '2THRzMiJh;H&Kw*7!N&ve0!_tg7s=YKHT+JJqB>uqZQ46&Uop{V?i?CEKUOF2AbvGzJ|Sz2ghC|4=~j0EWoJVA1Uw5J$vLoTXbJKbnlO!w0ZT^KR4?B'
    'Z*||g+%}S=`(ICio{kLw5=ct2>~WKgj*w;At!-J-k>u`~!&RUu5wvJgB$ps{8MPwLL!2kvC)xZmvkC<eq-9U<#5uFuwh7b?Rasei{mL6$#5^f1w@a&f'
    ')YEZHEw^r}3b1Iw@7Gof(-)ye)lNTMxB_iImu64izoD&6zb7J3uM($ib7cX!UyzQS*1$PFan$c5?KHQEd!VZPE_%PvV}pe{ocA7=a8+vF3TF0MqGiJP'
    'g4BBz-Njr^x@=|f&0rK@-5OWCe2u%XHM$~+{XW==rmzPESL<0w5AO;O@9+>YhT}=h^=qT6u+i9V4b%S2Jh{UYxQOR7%+5d3={O7cosh~4qd1rmwjQd='
    'EtKLAxaX9VzB@F?CjGZDx{5HoCjQtM-JQEjCQA&ul;5;WZ2hKIgJ5(Q(dR`{<|Jhm9$V|p{=xoXd-sH=;^CY6;ni5*e);mty`d)(UBZx;mwM!RSl%ln'
    '%=?(P9Cw)TP7G*9lpw=8*fhKk+tFsU71sO1F~aBTy}@k~(3tA@uA8(witp>A^p1{?Q8u0io%+t}mr;G^<tzGQ^VNA+hc*(th_<MaRyOOEi*IlL5YZny'
    '_~T`<I7CF@n8AGwU4;#5AX*spLkF=i=XT^e)Ww1_DV2rkRAAmAyCxA!Ossn-cSWGgns9FRhfi?sC5T|04Q3BPI-(<$=q*iqY~4*nvtvJ_rYeYD`8gxi'
    '3USR>B(O^0HsV*Qb$L_{Hjn&^;XLJoOA5R>_}C`7>#!iLmgF)n=nNzBWqf{9$LV9Bq|9Nsritl%G)kvd$UuY?yGIhX!XydFGM?X^=Pt}7BL&oN?tXla'
    '^K<uX|L8C|J#O#iEvA=)Nm`Q2FG<E9B^Q@)3DO@VRovVqcU>a6$xD*C=yXcA5EDoY{60x1<I5|O!PrCVJUF*~NKGvvKk`DP!_Ee55taA$Cv-!|4heHi'
    ')4ua0?|@Fy`OSRDU#tbKq+tvD^iflY3CR>%q7*AWbB3Hk9l(L#eJ}%eA5n9nFYfHTehN+azwv6T=QT01l4L^ky~R&3CTZHpJJIW+`QVKqDo2L$Zsys{'
    'yGT7QpT}U)>W?6S%&E2%+)^`i^=y0>Z@q$HX6x0zEe@)z{^bL)?MNrohCB5#IO0LDj;c)LjzU#`%o`1WW)34*AR8Lm=B9m9BA4&tm-baoDO6=4$YGDe'
    '097*d%u2XEDKo4R4NzlgZ6_Z<VcEZn>ziAPUESP_N44QN%e?VqHn^cLoK|&de5KgmY(g7aru5QHmH}&2200qB1U9li>;3tNZNMaT)0=nv9nuPEO!HAX'
    'o6vOi`xh^KN1SuR8l>ke(L@EblHE*leojB?AE}*Vl8*6TAk?h^QlO;)i<eZSM!iJlwaiq;nFkT6fcWkabripdIA<));;pC)^5J1{xv+Bh5W+_3{VX9J'
    '2lgHv*US3Fd@$_guZc$!Z<-KK4N48s_eyd_a<R$tz=ErrbYW)Z)RcTTo}Q|3e(KCtWDD$#NBzMkM8)wx;?j#?gs3q{1^7-<vSwFV+}V!!|IVYUL5~h1'
    'rha<-Vmo@Z6K%cPjEIq+yP?O$|LGZ*IJIM{0ogq1&Sp5-XhE+?rlv->sHd0FYqr}<mfmkum<p!>k&UGANFWM_@+8pRk|dyGsUJ48Im(4~`<UbbdO;`g'
    'G>yMBM}f>pXxgU2v~PRdbabx)Y{T)V0a1r-m9(gP(w*tppVN%ohjl3+)>&Eccl7}@`UyKV19{K#BFQNtir2QA=Mh`N{fhxYzPeLCB0=I4>^)ODD>Kdi'
    '%gNi!2~@sf{Kb3%ih#A2$;k6J<R76)dOX%Gy(=ed+C#+6X*$a_L1=)mM%Og0%No9QAtU26mmDSf9=Ogen{tot`r4ZJ-q_G-+YrUFBtx1ahXHxH4b&s$'
    '$7>X0mska9h^R?FCM^=8pkzDk@etrVrLmEwJDszsnna9Z!`+#-!V3l^@^I{BG!a)_m^=`=KIJ~CMLG?{L!4q=Wh4k(OS?#{VaJ%OgcgV}GO;3enFwiS'
    'T676PMoc{e2tle&o>0rW3IW$FRWfzh-H1PgtkHf9Lj(Syq;6}Pf5>A#dg4tpU^-~+rZFLKq+aF&3r)#S=?p;w=IaKodTsRS`{*IMjq}Q#!Fe9uSQwS*'
    'jB7A8U&i*!q}p`Ui;2Va#y9m7{wu&2mNHiRe|fYs@YH<FKaqa0Qf_<-=TCph<r+5=wbR@pmbLYYl_wZ+lB<}yu1nWJ_+$NLiyz-EJifM1rZ^LW9W-&l'
    'rdQY}`0ac&0J90|KSWz~QcDomIVDP~dZ`I+Jcg#sWk(eYdKPq}ptco-64lXVIg1O0=T61jB^9?<RJ<<MVH*}zy5?%NttAx~6|$hHkcDLpYe6LoiYnPc'
    '1=vS|`*o_f!8%!;7}w1t*p#>5dwV*WUYE@QvaxI8u+#IJhnYoQZYU&Xh3G^&Zl)LGdvDV1^~63U)#KZ`1dS)%jMZ-(KVXm37(ds!P*~6AH$iW36YCU3'
    'NV~vqHorBv(CRYsEeQ-C`M08nn)DW~8&cUY4|3Ad@-X7$QzAoiG+8v3>$1GjSeqo#E;R?G-K%wZY{a^P+BKP`L`c-PNd+R7IZbg65U-M`0Pnbq=uGP;'
    'ji{>`tgoX%b`v?fBC^iHni&2hMqM(4iu^>5sndrMokPT$_c2Uxokd<-#VeadPHF2^GBTlNq5tPlGn*Gm#aGiZFPPL&d(n&NH7Q9U`?`y7w)Bk<WYsHe'
    'u+xmTo3HTSxfUa-fzyF$I8psI_!<m4kS?g|PP8N4QN0(@&Uw5gUAWD#L$I2HP%|<_R;DDzOVA}d#$ncU4zswph-;q2%Q!CIzTY3*b1v-T;K_etjx%6a'
    '2eZ~!JK#Cpc$v%+92Za9cxOAO2Q`V`cBg~xh=n$U?6CZhs5&DwBau3Me&k)hU;^pt$;p3eD2_8e9GV<*C)6bTv`YtB>wY_W*p6<u`4`?ncQ7<wC)K(;'
    'B&K@5Ey39C(+8@!`1F?A3L}4a?nZX!TTQcmeYdCiWS9mjV}^08geNWYyNCSUR{7m+`JL191GBi0qX`2lvBmXn%THhNW5@u(!+Nk)yQR&+@dXRhG06JZ'
    ')-+lwF&t`yI@i2is0qO?`N~tM#g)GLATlSBsl8$QE}co{bNHw*1HQ;;Hej|kbMhnU<S!%t*pXm|w1UJw^)%w;vtBdqytspe*gBi%^#9Kom2HZ4Zx*`^'
    'cQ)<TCt-n=Fl49@7Y;&>p7%_8x76h-=bc+5{9KG-3|d;Ru)+R_xpd$*Rx1pvFi}6jYQaM{yO!h5gd`yceiotRGY92~F=0Fcj|?8D`>2~GTvpEljCMAb'
    'G(#{?T_x;V(|-6<yc78+>R)d_aLR*zK!NFfI=v(f3wulJ7rrp^-`GR0Vb1PcNzc4HT48Q!d_lEuxgkWMeiKhK`haj6YKA_s1wfJ|bYjPhZRPTld2}aJ'
    'N&n!COcEGX%8P(TLcHd!6BfZI{#`dS0)kA35guB=-!OuN&4~bD#%zw9ZA5hfJ%PCX-2gcnIpZ=OU5#J)VUu2dnx<(oH6aiA0_1LD&yCAruO^>QNj~kS'
    'U8^n*E|Y?AlgCI%LDzIVACWd*c&n*tNoJ-wx$I^s^|JNDwmTxXiwcYNld<eyCt3G44G?D(aa>%pECq3Po&e*}A7N`2b8edhc!Qrs%o|=`ZWwJwGkTO`'
    '+n^QOJ5s#l(L%?%d@5I%awG*zsMe)ez5L6`+>g{&#t|?E!9gs?oM+0#hY8vS`b6+~e9;?BBgccGVkP5IOy@9xdEz4O6S-a>lW8_41;hp-k&a%YRy@s?'
    'P<z-_YN04-N#OMtsgg=LX;*@^Ve+m_?o+>zDgGjX@BP=E<Y4#YeLMO0qc_Q;4EHI?9$DGsOz=FpXnN^JkE+2(WD13*h#!46RVE}N&FYgG91i^ZA~nCw'
    '?oO_VQ;nw&n(t3vn#dM1srxRC3QcDvi~!E7u(2Of?`f?%4V(s_r+_~*1N;C>Y8LpxE`&+`7MnDOm5aRPCVGdtQGg5l7#s8sBp+3c+&C7v_;bdg044aF'
    '=7^?}hhrLbvl}%jy|P;~Qu~xs71gs9eiy4{jRR}l671`u_xQUo+7mNQq&-XR1_~8$ktAy)x)J~ZX}81Uvyp@*f5YYp7beJw9JVR<G@7eb6Q_ysS0;7e'
    '<b&q<vD|d+v6`|3iAQ#sZ11o!0;{S5<|2iJmLyG-St&f1guUhic3BprcMdLARzh7=aS_Y2z=M>-KBt~=pJRlxj^~*N&ce>0E8?KUb}q+PDRUJGS)%32'
    '9kMK-qJnYE!q90WUAptx6$v^2kmTXKl4kcZ4{doGVTe3;+WhI1y;B+W_8F$t!OYkgKNseA%>yAf7PvzT`~@$H+@^Jrx{nRhu^f!cm06g5TYTqwi}H_R'
    'V781|3}8Q)o?!+q4-iRU?hedmCENn+xD{LNi+hZzOI8I~4|mfsjl$V&exVP?xsExoSO~nb@JZt-b#&8NR&>F|XorAO?z(H(wVYQnC%RLYj_D#Tk160V'
    '?bKZowo21b@pNIRZVTt((y^l0v&U-AciG4j-8rlGm-ZwLBJsD&@DMo86;j4I(isW@{$EHjBn^zG*Ptb8nf3&ZMzI(5J?*bd?qLye03c+~;~)l)46|d{'
    '>sQDz%7FZn_TJI2?UVM~<lV{9hvfL^<ZSoNK|47(+S@%y-t6xE^6{9pJYI|3Xy^ZSbn?so;rnFo=l0$&$4C2zXH=}u&dP6<QFKbPn2+Bj3Fw++&OL>B'
    '3b(1dfJ-wU&IU-Yy&O*lIC2m|4q!r8EV~g17L)O7+X`K|i61rDo~y8AH{AzX1rmlXXP$v9BpNVf!Bxk^UDS8Pk=tEO$!4grXMN|_&Sj%SUycwnX$nS~'
    '4*#+{^5!Gpn&DK<86vH)I>sQW;=o0wI0FoMimg;UNIXRoVIY9jpHrrOaZ(b)zs$4y%Y4`X#1weDs4&<rs?4lr)t<Id;@7d^*WgzLh8+vTo|vqU%Zaw4'
    '+b%ddW!%5$S9nrgELICOGbz*J?#;#E(;O$aQjM?_CB~ek`MPEkgBv)W#uJzZa+Dq`?R5Ti!n)@0fS2)@fDI&PDXm9#HJ=TJ974t`q$MKddLyxsJdd+k'
    '9@GBY*~u<PfFGZ<f89U&cnXsD9WM0lq<#9cu#VX5JGLVHX$r)y!~9EZLxBrPmvEE_SAl>%_WvQJiZ>CFI1e<e>ClQ_tDgpWWS%Ed2@*@z0?&}hXgMMg'
    'Z!OVAX!|QwA_0gslQ{&usbal2iLc+L3yEf8hyN!lD5l*NU!%}9r$6s5AlH;MmAvNxOi=$aDiKx4iY2s{tJRyY)M$JXCidU$)0W*mJKH7U;H_*~an>n}'
    'S9K#%M%*KfeU-BKa1cDy|1qnGpl5YBcE;;V?lodjbabGODdVp&=v%*T($D{u>%?bMKR^0%q!=E(k>mt<_J)(b#vEIEcyzeeJ~@-saM3gjcHqk3*TOz&'
    'KFYUq>}x;Fr;CU1BnCx)B52A<e?A;0R;A3dTt1~E#H=9uW_IZ}8Lyn!679fAEDP;u;rhU}7cSxIs2}-vzMzlO(^orU+_S8{I9w+De+an;{YbtnG6Gz}'
    'A@Q~kWFDOzk$fx!kj?x3!F?S2SY<}`^W36u+wTr&X5V6)YYz}8D;ZVvISX#Ezc7AlLXgj1`l<5vYtOJ_)dUK-hoWDK_=!Pk+ceomiwj*qa;&^9LW{jo'
    'SfKwHIp{P_YqIaHU%nWANu-ZJX9;k;ocFrTP2cKZ-C-V}CCEw*34g6^d+(1wdJLW^LQ{*r+QTd~h(ir78nle?TX3pRR^hQP!Qd>$4ci?iHf0Qr(T8qn'
    '+E}A1*%p9YZE&BM*!VN^&-P6$eU!b7i(u_0$ex&+!J^;)uuEL)bpL<15v?1Yw%c#xo$YV|m#fGM1=_)dVEkfUVz_9};6!H;gC%v8j=Rd_QAWkc3olC~'
    'rRo)_q>GuZI#Fg8MG!FIe8(q8zwV#zA06%<6jEQ=$HUMQl`tzDq9qxd=hZomIi#roTznTh8E(sx%I&Hujtgf)Vg$Wxxr<CzU*LC^`Td31Xa?uVK%_ze'
    't$NELLFZKh_ZSlelbYg=t2!i)mIOT4XXC)xc%dmTA#XvULu{m3GD*AFbTr-2DaL1)U_+$$0+kzgdRgc5dG6(uK!e5jb`Hk6&ousn8oIm=I#6qJ5INK4'
    'dDSQ%7OHMNx9aUe)$Qk29ZzOjmt)N9yS?tsZ-LAh2_IC7L<E|VXgK8v_Q;J~N7y<Clp)V>s_jr`_4m_m@01?~%=i$_4aU;50PoEHwZfceiUbW)RFbw#'
    '+}yZ3+j=GPtH%1uir;4jEkPbjc*J+{LspSLmv2C>ESYp%EK~cMpg<tlUPC}HIs87s6B8r+kVUtdV{a(MN}y=Cq}$PWRFh)dho8?{o&S=`b-36##@Ts4'
    'AMrnufEEakKTSC-ds4?g9EjtEB@<vmF6A7YP9!L|1V)EYj*1WZWP(4?2~H`gQT)b31Zne!p`nWmj0?pO%P-$)a}1zCXZ1^jN1{Z+VG>oewpHJBCuo*V'
    '_~Tn<M&uj`TlA3QvADSadwDjdR!6BNGiS4B52MSg>3D>sItg1mRoExm^J%Te|0GoaG0GCogOCOiZ-LR1@|kSzBS3iRydOKsp9Np9d=WRxH&M_RQSq=v'
    'e;`2iI@rv)lAM-#QOC<m#fwf}XX{qDdLu!xvY$>87fA&1qSwDH6n~n`{iZWfbmApWS2SA5(KgVaCKp{(*s{y%U@}{oJ+@XzRC_iU!xuBP7BY~QWofMm'
    'r~||a-t~Ty4H);zE5gVK8}3_|ujn`MQ9<)%jtf>M{U!oo3g_6pDFBfjI8U^rwF%z-O-Vt-zw5me2ACkMUry#I<;yw(`-G^##5)jCCpHTx!Ip57^he`F'
    'g5eWEiPB2AxPn&yW>Qp0iZBhi6h~^N(IZua7a9ufy%GNL326ZP?|(k?gqW0uq+Ewd$z<5c+(*k3>~ucgzwtGT*tAkT%jXTb+vr8K9hPnjQMp{|uNf1j'
    'vm!_H*-`%kx5lPSh{kD2O6%tzm(XTUj*dQ<%pWr=p7R~?Bfjh}&Xd2KKearu=K75OZu2*7)>fphOXJ<uY<utU=q%5+>rKZK#92r(TwRU!z~_ANlJPnF'
    'h}9ahDAdcGke{LO6j#DwgrJ0P3$Y=^)A<N=nMV_60(eUFo~e4$23&Y92o3-XKz|Ahpxk6*E*S|dSOC04@DJhcnOwJF-WH0IlIuybR9XD^)WTF)bhgh*'
    'Q7pHR7Q^~<01HR^@csUwBC->0#Gn5SA=@_359h~lm<~ZVNB~U|(TI{H@a5*5P-a9q8%wofI!{Np@i?o~UjsTGG)uTeKRc}XB#_PFGJ4zob$_pYO7b=1'
    '-%sWol77>R%U9Fjb&&<;z^Ngwr^&Y*ekniJFF$9`N$K(%>LtQ3=|5^6$bW8$<Nf1yB$vjxk5A8l3tE{4w)AeBVLt{L5GTmj%JKwy<F|%=S8JZsWjKrX'
    'co|@1=bjuk?9*x+pM&N>wHA2GyWRbR_S>dqFFx%@KcX3rOn;r)_AA_qQ(^?>Dgtx(EvkzvxZNUbKf4@-C+Q(-AQ6d!FMf!4GfE`h(PfPdcdFo}N552k'
    'tL2XQ3jT9^av9rq*4ki!3;VYnKQesG><V#!a01~hnRy&Zig_TS7K=+FACbxQL_U%Id<ZC=g=u*beH_J#V8Lg*Cubjz#qFBB5z(OO@FI0efLZt(w0{_;'
    'SVpl?Me2Twq`K5=T(eWKp$_cEf<4t*4`1BBu(s6(Z-#vE>%0dHBNb*>Kmh0w1tDQ@A@8<ek@+g?3j|Mg(VF)yML_h<cGv_o1d|+3HFCjew8-iJ{jHSu'
    '8xLIgxD?7fVAWCMM2JgUC{oiuXYk?K`mn!ua+JK;J=^;^u8<Sj@7}fd&h~$8yG0g+yRh2Awc3~@_HPEkBt8JEb=iXMYnkU<efTGvrj~#yS^FoOn@a|&'
    ')z+*M=DS*YI8(FhInag$^AqED`3rWluQY`faU9JZIqF+4#M^z7g{xJ@4_;Mg6}s!6i6@K>9pGRGk0Z{(KsX{i&xLH6If=gpWn|5>xaH3a0V>AoEQEyR'
    '4(~PNnRn@WdA-unInXc^C20K&(1`hW*-(;){4Eei-8<yltZ<r;Cf@zj1%#G`KZ2RGA-UoWOlI9*9DyF+p*Xkpjy@b8w9nc}d-vqv?{I(|(@A2Ve%Sq6'
    '(mp=g`}x!aR0`WX{bm2y+dVjG@4o%JXG*{I3qC$yeSm(n+8G!wTzIH~ev4L8Nha15T<6JXQeS+y!knt>KT2n-Y^T;yv9`oEh<gglmcdHYlY*tN0`&rs'
    'lV5E(6v&RG2P4m>#8TF_`0qqhs#`!;!%KK3S+Q6Zo#xW}<#lW#@s|`Ay1Kl6CRsZ{nT(@0>f+Wzt1vNf+faGxQo<}~e#=&&%WI8;p=ou^JZu50Dr`*!'
    'IPBDHR#i@32`IBWwV-#r=VyE^3T9fY7I==I=CMw^sggh9zIidsJN|{oyf%2E?qgrglKzzF2e}M<OUJdErC`~^3m!Z-WQv`u)x@n0vXSbnUfbe$eqQh)'
    'y4Jz!F1U{EL;1j)HmRbyz|Kx(-TlI%NQe@)9LWSiy3Z$<Mf{?iRbg@~um|f(dA<Xth;<;;;+%h9Y6-U|RJ3E#ojB~)fB^PlfB$eF@saAFsmR`RzkdOo'
    '$A+iyb>8?R2!Ar7bnS;th59J<`R3!>_w6(0-pNV(^y3EvcKWRgI*P72FOfEqb^7QBl(I)Xgpp6<oRgAR-zAL&q`>@ggG(A@G=U|_MTF2MvvJCy16s61'
    'R&6vE#oX2i*1RmfLHf{TIv8>XJg~5$xa1M8Hhj0ZEc-6>0!PmS<WXD&ek3uIN2An(5D7RJSwA%a5wjk4;rE4fGaU%>C7P(sCkUW~gAmYN9&>dLmRxcm'
    'tOr5`8(U!m4KTD09{k9^IM^WAtiN0jwl-uk##zAsm<-lU)f&o%4FMlqAS8Jk%2jmYw!j4mAVW+UA6s%~Le}KJXpqGH;^EYk$67>#qWMWEcr(h^rNXMI'
    '8OJ2O+{av~GiVMr7R&K`!%x^TG+FR2QuXuJebJrwpT>#!pD1y!jy!u%w;syHw^kOvEf?QjRh)-=$I%FJfZ1s@TtW}m-CwP{6!%NhmJ_0mq~b=QhO|gZ'
    '&iF1F#mG^%6Wh67uK*4Uu8CwSUYv_(>N@8#(p`j6gr=Xb_cJc_0}@9G?nBPWG@+u;@&o2RGVA=SaR04i8fg7#cj)A}4#C2nw%0!{bt8m%RB<}l*otG!'
    'iGE#`W3lxGFWrLFr#?FGPjS;x8>GBsQKFL$oSy`<P5(vCsTCQrj7Xnnt=r=9z}&*P!m+f~@qsevpZ^4TWMeC>y|&j7NVz79G{wTMJfjT9h?9K5Bg~8Z'
    'TLIf?2=Su|NlmSwv2ls(=%E7!>EMC8hA;&a^JxyJ93;n}h<>v;9>-OJBBUbaCGg{1s$LF@mGw8hb%e2I{_jWyG5HqUZ)6~uA>Wu^>?-mgpc!(b&*iBm'
    'LHRM4ew*fBtK;p;v&14K4{GT!&DWQPYh=bI{-L3XeiIa0@VPoda1$h0c$qgX5^&Q}vD<<XGgVUjR1gx)doYu-|I{h<Br_)Y#npO6ba@??sgnl?Xq}r0'
    '@-HweP4Vx@T@LC1761%$|4)QSv7WtLCWj5=v9RNM*;s67=U1=nQkXf{3cG6cbpahE=PAFt#16AE;xrT(W<*Dh(mRNYvF|Gkv+)1e+#>lKjNelPL=Mtn'
    'PlDN5cmq|3tvhjRTnUi0{S77)pht-hDT$yb9}f@r58t!XPfQGszr9#E`nC;|<#XoF^U{sY|1R3Oz!_wsSEIBS+6SyqVyB!WKZfed-O&u!NJW;#!Y)bC'
    '<M4nb%5`x7OA_ZQ8ZeYAJ58M&%Pduq{0JSh4x4(4Rvb&F%*2g*dVJS$dHZ>^6Egqx-eQj3QkI`{83&Hwa#&f`GHkTSTO)E7wPY!RIl~Cjg0pHe;S!S3'
    ';qy%SQt6Qui{qG~PnmuG6qjI5^@?ltNtygOC++QPEm<sdRQ6QfK2>%&tH1GKyR?R5qtlF}LdoLY^IA#MvE+uWC8i!rOsz^g3|FnT#Rz&82e?Oir@~iD'
    'Gq#~3>-$m+{hz`GWyrX&do?rEf_m@ss5|`gSl(Lsz(A9@Q4;(&`X-*|Hd&m;?uc493-+Z8ELO8ax0C;A&iJj2(YnU}|5~$eT^BoqpEkc{d7S8|r6&ka'
    'pc51I??-RE)3e>PkEg{$q&%@m$@w~7>IC{&1D1pEW9F_90z^D!MF0@o)-#FU5J$YZ>ll7|ut&ZQJ?{A5d)7maOQ3IG)L(f2^?&}acQ~G<7vu4D?M1;?'
    'Xv2M^RSq5S{mApz*H4kI(_CMN0ckj9SM@{jG8aTYnRW4wkz`ZbeBr(OcyLh98#;CGyjtS1G%m@e_q)GQ;f&nyfA@k{fBoy0j5qTB`ub&YC|Kx0#wpF7'
    'OM0y5Tk;EaTzq>r>c9<k`qGoz=DBx?j-(E?sEZTF1s0%)NTcid=Xni$^dDYZ!lC}~P61Cse+dA<AKn|jKF;}g2o~nK7+?R8hW@D4YWS~N_($^(-^5(8'
    '{e&r?18UDiCSYYU{w%JJ>g~!ALm(KVAku*Sjn5s!FX(H$KQw-go-)@s4@(a<u*N)=tUdp*vFo!Wfl<I{4}-nf2>bpcKW)}u^`HDgbIxFe=S194$b~Rm'
    '9<8?#dmWzN!?DLHHG(wUK#q;T2@6P=B-oHNArna}%wolSqSGa}<sd*~ipz;_VBh%!z!1L4lV<gI8#>bXoy-2t0VTg%`pI{M#=T|p3nIVST#%2uid1Iq'
    'O!7313QHiWpWQca8hhI6*%i$ZaW`)|$gXRQfHod-@BeEHz>tF*4DklJ9rKZmDzWMD&MiNmphTts-lr{42fYOOz1a=__~+fzb}4%1tiAX1;XbexPTCMA'
    '>o+~WkY|NdDv8<QwFo)4sv>+$QSC*b<;$x$f+Gvet=~Y3F?nC~z9#Q;<RPLR0C0MI_tY$FdE?nt%5Z1QiD=elyv6U+9&6M9jM=*y3<a=+5d$VB(@OH4'
    'RON11wVD&N3}{9+oA(~%iqA~PO_ZaMG37hQfjA(ytec#7>XZ(MEY<`~6sQZNz!y?K*E!b{20s&j5I;&ti%A%7JPPE`liz&wHDr0hmJUD#&Jwws(3K-5'
    ';ZvSjpd))))8N%avq~p<5&jZIKv?%R^B_k>G2oFhj5=`&k<0^0e2jcSayZXF!59zeOK5B10~YR>uXnQqcQM`v;WvJm`#DdXphRj58>DYL+MHO%ZzYXJ'
    'hbGwa>=5iv(==9SfZhQ2KxVZ3ym(<9FV}b{3g)8Dqk$5ozJ3%9$)WB9-V7>bV)-OFmKv&Yg`XBHz!oAGZ8E8vifi88)kEy#5KVTGB)hwJcAD%T%6)&#'
    'TA4?P*Tp|PVp`fg+536_*Y;`h_x71@%-cF{De5Cz6yaP)C#Ez<b|nY<ANJ3{Vu_lk1EP;SJKgQ-@w#~{R{N+ptD30JIM@a44eQS$tj&~n?tRSn#_dkR'
    '*TvzK(#bI!AmO_pRota%iF^_MG*Y99U(7iR7w$046z|-5X08LTkEF!rxMxlCr6l|EWpD_V<e9?Ns=Vj*x3odLGMxgGQ~3)dMy3opQN-9s6+7p}+3j5C'
    '9uar`i!)l_4SvX0W|@-h&Fxo-GYj@8u#&O@DH>gd<nM)iXUTUhaxKfTz`k~gCXYUz9e+GKW!_c%hE+B958t-`#`lmX$Dr!n15I?-JbG-r2KsMt(l!4K'
    'iYeCLd%Hu(c6xOmFOy_rT2#Y}v=5;}rU;1rF8Ymw%*1DeqBeE2+5q;|e<FjM`ZM|WC4x{?H~t(gMf~uFBm&46!~x4o$~VPp!4)en6VAbk)y>M@bMwh3'
    'k(PW`Liw}D@SG%bhit4=@1&Y5cUYcr{=L^`h&><zvBZuZAzC_^^uq^FkwtkKW<=jNP-`Fn)SsEO&farcxd+MA!UdY*Pv*reAaY#ArF#^@)<|4U(v#SA'
    '^2S`74!XHumoW@F!|u&RuiNYkuEpzhi2>gZqYK}l5P4VTRK1{XHe#sA7sg_D<=(~cU5r6@eD@0Uk$I>G%;JUD@AeN#>}&rGVqda%ba)1N@WcM;2L$i('
    'wI}v7#AH{2c(Qzr5r%2?yF#~m1-OQmx)G(l*vaZ|1qF6c2?EfQ)PI}vUzOHSh@yALt6wgr0DIAG;oJfLM&raRK^j&x>Dh~|l`qNhA!ZDFEcoT4KvJ{0'
    'kP$ATf^eWR5z7vy3v(!%cLH-z7Md1MNaq?==c}83B%dO>uOId7Q5)ft%XdyQt_QVdLecj>u*v!Gtm*Z}cOy`PTzHzo+pJi0%xEDA>8g3quP(dOjKrvM'
    '%VEhG+Y+!RAlQqe%m!pu-ODOpZ$N-WRYLG#1q$yMzzIkIenzCYi42#UghQ>ex>SRXS%3$1`)M+v6H!C$@iUirZg=XF2fqk%vyg+-?Iq??`w|^%@{`<x'
    'JnBxetMSZ|@Z^EcemeA#=A;;zx&Ks>tJMFQB>Fx{#JHWPVhrEyRC6S?2-~L|M=TOq*d9${&5F>tL0HE8Uz)EF=Kk|4Nd*VRX6%HVwM@uaJlou`u|*1D'
    'j2lAc%=y+|ZGL46$KhCVE#4o0tQ&N?({wf?#*%qIdAoTM2mOLM;cUmd-v-w`+u>+t-7Hoz@>KOMTx0y={zc@mtmMfpk~J>+>rRW?)I^WxCwNrihUneJ'
    ')-pD5az3J@lnc^IM;We?SjZU&%%x1@S?Hz(x@g_;va3%@q+4A~iMP7xaua;NwHSL6L5#o5sh@KwG_6+se<fI>g;WB2;UiJ%h1{`b8*uDRKHLr}<<HpZ'
    '=Scme*^0U=FRr?zshS&G%tNE|O-Q|@pST2UKH+)b0{J)~Wsu?^67HqS@VIGPW!_A}{meR<8=>U}AI1&@V1lrCxm#k}TtLllL~t;I^RPP~!S-f)xAyrJ'
    '7rx`2bI<m(Jk`Eru@1)5<P*E-(#wI|F^>M_dPH{6Je3z5DX@c4e_Xzmz7kr>n7KZDJUH9`aP$^eujktiF#3!OwVklJ9(ndTo{O1;-0$!u5UDTepxT~I'
    'v~zM}#V<q8kYRNiIhU%JwQeM5xeB*Ig?n^CEu+FAry>DF4-VDypPtNEo^gNdmvHz^Q<hY)C9`yf3uzu5nt9rIv{dqBap-Fo4>d;CBeHaFN_Xf>WC5|x'
    'wInJOa{X00pR&E5P0S=a?C-xV82<V8co@aSOUkE0X(G<~HZcj~zkG;SrX?>wtH4`@RUs)F-9zj&dC`uKNKJAK^Yx50HYg<VGGCTnSXRh_u;|)u=1;#m'
    'H&zP%87Ve@LJl2+mx|3Jd3G|@(tIo%Kw11sH|JxaH`TLBx44S`j|Ev=*rAe$ck!UmGY0C3lb0H1)3&JN{ybylb8ugXzqVIMlC~m_5B43}5q}wqS@5fu'
    'c#9WMXrBvfYCWR6L#)h$XPuwiA8{{hV<{y~W!HlV_G3XvZz(!^>ISG3=TqUZ`+V!1abWN`9i^a^2ew-i4}jt%Wlm3GcR0m(7Tt=*#pO~DrW_6(*?>O&'
    '>9_FbYPsIQO<PhB_H*ghfM%}*ZOGBG+x!>6YYf6g@!&iD_~u}L&ppcMuzG*eK3!B;Z7OySiy?XW$>CALEE(|F2fH5+fr81>NH=3Ofzc|;zIhP}XtMxP'
    '#|YK-)O+UfC~<+yoI(zaH~ZdaB#m2$3CN)|%`?e|xVwVwN9Wj+5K5j_Oj+?INS7nCwM6)%!|lbEpQ{i*+N>1~S0CwzCl~V`R2f%`;N@}1DK?}w8BYWw'
    'h><REDRD3;Ydy{&_zWMIGo0^)%H0-b*w$@vu1Hs(u*c&-;T{$ML^;u2<BGaI0E~!L;YgC3to|v`4DWkf_NrJRaMRt5#5-r6wL~5r-{JPx5xZ5{CGTXc'
    'b1dsOp>IeZ70?@aqRCu_r{a-WQh97M{N+wAI1CF|96VzccpW<7OL02nP}Rli6u8WJjERf5uSnZ*0~qr?SYWC-ki_vmz2oZj$)Ja*M&h*Ekf4JWS)MAl'
    'c2Ag)JeDBShlq!R;iC~g@L4#rq^6>!qPF$LY(hor#`wnpdnK{M!*-2_uB4Fyt0#+nHRTdLa!xusj)SmMd)1U|8X9a$t2(=Sh&y=6105=i44P$I^yneD'
    '1NIINR|}a38U^YhD8P;9@%%GV22-SA;KzwrGZ;gti3fZ;0sc`HI2j6!=yd7`g`ZgV#8L&#DTxz#nNDp>?W<;qaT`JV#E;f7{l$Wv+V;y55=tv?ze5$y'
    'OHJznU;&BMFv6Rg6>09(wqZp&hIhpAPnpbk6h64A&t%|$yp)4MZAW>L?_y2p^9p>~9<XS<R3!{6W+Q2zoE)7bXLKm(ki0y&03T1^N)tnLvh}(GD+X6*'
    '4)Yy{dBRx>UY-rpbi&sNf5vIJObV}51~_qkr9{3=44)MDLqQsz6lC2N56ev0<H{&<bDU>IDLoG_*BAQJ8srL*d)Y!@O}I3KFIyQgWSd_a#<=?Cl_v}H'
    '^9^%AdOh3cPms3%#3n9;@wgayFm3<9zD8mLr-=<6W)4DIWtvAURmK=6ewbrX@DSC*<*UQeNmX?N!;R(PZTocZWdE2^PuS0iAs@Yax4(zFaOVrkHjR~A'
    'ht)?VV&e`#0K+#7(;W>v3=-+)hsc)`lecLiJlSt{0_OSGVRRe1riH?uK6e`}nMsTb*%)^c#|@XvM+3kHZ7y*!f1OOL(T&!O`X8eh|6686^=N%Q?4JDH'
    '<5Doc+9)}mDRo^qVQ!HPZi}3VwrF+iTnSA_hIV#zwtHZ)C7NdJorP6CkNn0DEsx0613?{}ofbX=q|@RMN>4GA{L21x?(~b%-}9!lvMm#hWn4DRFQYD1'
    'gfg`}2G299eyOkJ!K-V_^#b<p+;Ap1vDWS+?oK~Vy3;JJ?@m9>u_nj(g-G}A1Q(npU40%D(b~F~a@(Tbn?^(OE2|h0^SR}X9C~>*hK)XU0q?5Hi<HWa'
    'cbfXjBDvJc#W)X<3dUW&h$c%>m`(SZvk(4Ox1ZW9Lz;K%ImlT$<ihfiO+~>n1)?s)kwh?XsibT;w(DP8ThLfF%2|Z1x|l$zOaSN;=yYoX+!_uEGD&D#'
    'CJ7caN&KcP?Am_=`>d_L'
)

def extract_exact_runner():
    import base64, zlib
    raw=zlib.decompress(base64.b85decode(EXACT_RUNNER_B85.encode('ascii')))
    assert hashlib.sha256(raw).hexdigest()==EXACT_RUNNER_SHA, 'DO NOT RUN: embedded V4 runner SHA mismatch'
    p=Path('/kaggle/working')/EXACT_RUNNER_FILENAME
    p.parent.mkdir(parents=True,exist_ok=True)
    if p.is_file():
        assert hashlib.sha256(p.read_bytes()).hexdigest()==EXACT_RUNNER_SHA, 'Unexpected runner at target path: do not overwrite'
    else:p.write_bytes(raw)
    return p

def load_runner(p):
    spec=importlib.util.spec_from_file_location('cracksafe_v4_exact_three_seed_worker',p)
    m=importlib.util.module_from_spec(spec)
    sys.modules[spec.name]=m
    spec.loader.exec_module(m)
    assert tuple(m.SEEDS)==(42,123,2026)
    assert ENGINE in m.JOB_ENGINES
    assert m.CFG['micro_batch']==2 and m.CFG['effective_batch']==16
    assert m.CFG['image_size']==448 and m.CFG['max_epochs']==100
    assert m.CFG['input_root']=='/kaggle/input'
    # The worker will always reload the unchanged exact runner from disk.
    # No mutations to model, training, source SHA, input roles, or checkpoint identity.
    return m

def verified_job_archive(m,engine,seed):
    arc=Path('/kaggle/working')/f'CrackSAFE_P03_DEV_V4_LARGE_JOB_{engine}_s{seed}.zip'
    if not arc.is_file():return {'archive':'NOT_YET_AVAILABLE'}
    with zipfile.ZipFile(arc) as z:
        if z.testzip() is not None:raise RuntimeError('Job archive CRC failed: '+arc.name)
        manifest=json.loads(z.read('files_manifest.json'))
        if manifest['engine']!=engine or int(manifest['seed'])!=seed:
            raise RuntimeError('Job archive identity incorrect: '+arc.name)
        for name,digest in manifest['file_sha256'].items():
            if hashlib.sha256(z.read(name)).hexdigest()!=digest:
                raise RuntimeError('Job archive SHA mismatch: '+arc.name+'/'+name)
    return {'archive':arc.name,'sha256':m.sha(arc),'bytes':arc.stat().st_size,'integrity':'PASS'}

def run_one_engine():
    runner_path=extract_exact_runner()
    m=load_runner(runner_path)
    import torch
    if not torch.cuda.is_available():raise RuntimeError('CUDA unavailable: Kaggle Accelerator -> GPU T4 x2')
    root=Path(m.CFG['output_root']);root.mkdir(parents=True,exist_ok=True)
    logs=root/'logs';logs.mkdir(parents=True,exist_ok=True)
    pre=root/'preflight';pre.mkdir(parents=True,exist_ok=True)
    (root/'source').mkdir(exist_ok=True)
    # Exact unmodified worker source is exported with the result.
    (root/'source'/EXACT_RUNNER_FILENAME).write_bytes(runner_path.read_bytes())
    n_gpu=min(2,torch.cuda.device_count())
    print('=== CrackSAFE Phase03 V4 — ONE MODEL / THREE SEEDS ===',flush=True)
    print('Model:',ENGINE,'Seeds: 42, 123, 2026','GPUs:',n_gpu,flush=True)
    print('Immutable worker SHA:',EXACT_RUNNER_SHA,flush=True)
    print('Original data:',m.TRAIN_COUNT,'train /',m.DEV_COUNT,'dev; held-out raw data never accessed',flush=True)
    t0=time.time();deadline=t0+m.CFG['max_session_minutes']*60
    m.require_protocol()
    m.install_dependencies()
    pilot,_=m.load_sources()
    m.ensure_selected_manifest(pilot,root)
    aug=m.augmentation_preflight()
    m.jwrite(root/'augmentation_preflight.json',aug)
    m.jwrite(root/'data_access_contract.json',{
        'selected_engine':ENGINE,'seeds':[42,123,2026],
        'only_roles':['train','development_diagnostic_only'],
        'excluded_roles':['calibration','internal_test','external_positive_test','hard_negative_test','reserve_pool','corruption_test'],
        'image_mask_sha256_reverification':'PASS_BY_EXISTING_V4_EXACT_SOURCE',
        'source_safe_provisional_train':m.TRAIN_COUNT,
        'source_safe_provisional_dev':m.DEV_COUNT,
        'independence_certified':False,'official_full_authorized':False})
    m.jwrite(root/'run_info_public.json',{
        'scope':'PHASE03_SINGLE_ENGINE_THREE_SEED_DEVELOPMENT_ONLY',
        'engine':ENGINE,'seeds':[42,123,2026],
        'immutable_original_v4_runner_sha256':EXACT_RUNNER_SHA,
        'blueprint_sha256':m.BLUEPRINT_SHA,
        'corrected_manifest_sha256':m.LOCKED_MANIFEST_SHA,
        'training_n':m.TRAIN_COUNT,'development_n':m.DEV_COUNT,
        'micro_batch':m.CFG['micro_batch'],'effective_batch':m.CFG['effective_batch'],
        'augmentation_spec':m.AUGMENTATION_SPEC,'official_full_authorized':False})
    # Run a real GPU forward/backward/reload check ONLY for this model.
    env=os.environ.copy();env['CUDA_VISIBLE_DEVICES']='0'
    preflight_cmd=[sys.executable,str(runner_path),'--preflight','--engine',ENGINE]
    pf=subprocess.run(preflight_cmd,env=env,capture_output=True,text=True,timeout=600)
    (logs/f'preflight_{ENGINE}.log').write_text(pf.stdout+pf.stderr)
    if pf.returncode:
        raise RuntimeError('STARTUP_PREFLIGHT_FAILED for '+ENGINE+': '+(pf.stdout+pf.stderr)[-2000:])
    preflight_report=json.loads((pre/f'{ENGINE}.json').read_text())
    if preflight_report.get('status')!='PASS':raise RuntimeError('Preflight report did not pass')
    m.jwrite(pre/'preflight_summary.json',{'selected_engine':ENGINE,'status':'PASS','report':preflight_report})
    print('PREFLIGHT PASS',ENGINE,'peakGB',round(preflight_report['peak_gpu_memory_bytes']/1024**3,2),flush=True)
    # Existing V4 per-job checkpoint archives may be attached as Kaggle input.
    # Worker restore_previous_job verifies hashes and exact checkpoint identity.
    pending=[(ENGINE,seed) for seed in m.SEEDS]
    inflight={};errors=[];last_seen={}
    while pending or inflight:
        while pending and len(inflight)<n_gpu and time.time()<deadline-m.CFG['min_launch_budget_minutes']*60:
            eng,seed=pending.pop(0)
            gpu=next(i for i in range(n_gpu) if i not in [v[0] for v in inflight.values()])
            logfile=logs/f'{eng}_s{seed}.log'
            stream=logfile.open('a',buffering=1)
            worker_env=os.environ.copy();worker_env['CUDA_VISIBLE_DEVICES']=str(gpu)
            cmd=[sys.executable,str(runner_path),'--worker','--engine',eng,'--seed',str(seed),'--deadline',str(deadline)]
            proc=subprocess.Popen(cmd,env=worker_env,stdout=stream,stderr=subprocess.STDOUT)
            inflight[(eng,seed)]=(gpu,proc,stream,logfile)
            print('LAUNCHED',eng,'seed',seed,'GPU',gpu,'PID',proc.pid,flush=True)
        if not inflight:break
        for key,(gpu,proc,stream,logfile) in list(inflight.items()):
            try:tail=logfile.read_text(errors='replace').splitlines()[-10:]
            except OSError:tail=[]
            notable=[t for t in tail if any(w in t for w in ('EPOCH ','JOB STATUS','Traceback','Halt:','RuntimeError:','PORTABLE_CHECKPOINT','SKIP ALREADY COMPLETED'))]
            if notable and last_seen.get(key)!=notable[-1]:
                last_seen[key]=notable[-1]
                print('WORKER',*key,notable[-1][-240:],flush=True)
            if proc.poll() is not None:
                stream.close()
                print('FINISHED',*key,'exit',proc.returncode,flush=True)
                if proc.returncode:
                    errors.append({'engine':key[0],'seed':key[1],'exit_code':proc.returncode,'log_tail':tail[-6:]})
                    print('ERROR TAIL',*key,'\n'.join(tail[-8:])[-1500:],flush=True)
                del inflight[key]
        time.sleep(4)
    return errors

def report_and_export(root,m,errors=None,controller_error=None):
    jobs=[];complete=0;verified=[]
    for seed in m.SEEDS:
        st=root/f'job_{ENGINE}_s{seed}'/'job_status.json'
        j=json.loads(st.read_text()) if st.is_file() else {}
        row={'engine':ENGINE,'seed':seed,'status':j.get('status','NOT_STARTED'),
             'epochs':j.get('epochs',0),'best_epoch':j.get('best_epoch'),
             'best_development_dice':j.get('best_development_dice'),
             'checkpoint_best_sha256':j.get('checkpoint_best_sha256')}
        if row['status'] in ('COMPLETE_EARLY_STOP','COMPLETE_MAX_EPOCHS'):complete+=1
        jobs.append(row)
        verified.append({'seed':seed,**verified_job_archive(m,ENGINE,seed)})
    result={'status':'ALL_THREE_SEEDS_COMPLETE' if complete==3 else 'PARTIAL_RESUMABLE',
            'scope':'DESCRIPTIVE_ONLY_NOT_OFFICIAL_FULL',
            'engine':ENGINE,'seeds':list(m.SEEDS),'jobs':jobs,
            'completed_count':complete,'total_jobs':3,
            'controller_error':controller_error,'worker_errors':errors or [],
            'portable_job_archives':verified,'exact_runner_sha256':EXACT_RUNNER_SHA,
            'calibration_or_test_raw_access':False,
            'phase02_final':False,'official_phase03_full_authorized':False,
            'created_utc':datetime.now(timezone.utc).isoformat()}
    m.jwrite(root/'PUBLIC_SINGLE_ENGINE_PROGRESS.json',result)
    # Single-engine summary for future Phase03 integration.
    m.jwrite(root/'PUBLIC_MULTIMODEL_PROGRESS.json',result)
    lines=['# CrackSAFE-CP v3.3 — Phase03 one-engine development report',
           '', '**ENGINE:** `'+ENGINE+'`', '**SCOPE:** three real development seeds; NOT certified FULL.',
           '', 'Original worker SHA-256: `'+EXACT_RUNNER_SHA+'`',
           'Dataset: 6,771 provisional train, 785 provisional development; UNKNOWN excluded.',
           '', '| Seed | Status | Epochs | Best development Dice |', '|---:|---|---:|---:|']
    for row in jobs:
        score=row['best_development_dice']
        lines.append('| '+str(row['seed'])+' | '+row['status']+' | '+str(row['epochs'])+' | '+
            (f'{score:.6f}' if isinstance(score,(int,float)) else 'N/A')+' |')
    lines +=['','Checkpoint ZIPs are PRIVATE. Split parent IDs are proxies, not independently verified acquisition units.',
             'Original SCSegamba / DMSRCrack slots have NOT been replaced in the official blueprint.']
    (root/'PHASE03_SINGLE_ENGINE_DEVELOPMENT_REPORT.md').write_text('\n'.join(lines)+'\n')
    print('\n===== SINGLE ENGINE DEVELOPMENT SUMMARY =====')
    print(ENGINE,'completed',complete,'/3', 'official FULL: BLOCKED',flush=True)
    for j in jobs:print(' seed',j['seed'],j['status'],'epochs',j['epochs'],'best Dice',j['best_development_dice'],flush=True)
    print('Summary:',root/'PUBLIC_SINGLE_ENGINE_PROGRESS.json',flush=True)
    m.export_all_outputs(root)
    return result

def main():
    m=None;root=None;errors=[];fault=None
    try:
        exact=extract_exact_runner();m=load_runner(exact)
        root=Path(m.CFG['output_root'])
        errors=run_one_engine()
    except Exception as e:
        fault=repr(e);print('CONTROLLER ERROR:',fault,flush=True)
        traceback.print_exc()
    finally:
        if m is not None and root is not None and root.exists():
            try:report_and_export(root,m,errors,fault)
            except Exception as export_error:
                print('OUTPUT EXPORT ERROR:',repr(export_error),flush=True)
                traceback.print_exc()
                if fault is None:fault=repr(export_error)
    if fault:raise RuntimeError('Execution failed; see logs and generated progress artifacts: '+fault)

if __name__=='__main__':main()
